# Bitcoin Master Workflow
## Complete Trustworthy Forecasting Research Pipeline

One notebook containing the actual Bitcoin data, baseline, neural, Transformer, advanced, foundation-model, audit, trustworthiness, significance and cross-domain code.

## Part A — Setup and central execution controls

Default Run All uses frozen artifacts wherever available. Actual generation code from every source notebook is retained below and becomes reachable through this single control panel.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate TimeSeriesFoundationModels project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ELECTRICITY_RESULTS_DIR = RESULTS_DIR / 'electricity'

def generation_enabled(requested, outputs, label):
    outputs = [Path(p) for p in outputs]
    existing = [p for p in outputs if p.exists()]
    if not requested:
        return False
    if existing and USE_EXISTING_ARTIFACTS_WHEN_AVAILABLE:
        print(f'{label}: using existing artifacts; generation skipped.')
        return False
    if existing and not ALLOW_ARTIFACT_OVERWRITE:
        raise FileExistsError(f'Protected artifact already exists: {existing[0]}')
    return True


In [ ]:
RUN_EDA = True
RUN_CLASSICAL_MODELS = False
RUN_EXPENSIVE_STATISTICAL_MODELS = False
RUN_ORIGINAL_LSTM = False
RUN_IMPROVED_LSTM = False
RUN_TRANSFORMER = False
RUN_ADVANCED_MODELS = False
RUN_FOUNDATION_MODELS = False
RUN_CHRONOS = False
RUN_TIMESFM = False
RUN_VALIDATION_REGENERATION = False
RUN_TRUSTWORTHINESS = True
RUN_VALIDATION_AUDIT = True
RUN_NAIVE_AUDIT = True
RUN_SIGNIFICANCE_TESTS = True
RUN_CROSS_DOMAIN_ANALYSIS = True
USE_EXISTING_ARTIFACTS_WHEN_AVAILABLE = True
ALLOW_ARTIFACT_OVERWRITE = False

if RUN_FOUNDATION_MODELS and not (RUN_CHRONOS and RUN_TIMESFM):
    raise ValueError('Full foundation regeneration requires RUN_CHRONOS and RUN_TIMESFM.')
if RUN_VALIDATION_REGENERATION and not (RUN_ORIGINAL_LSTM and RUN_TRANSFORMER):
    raise ValueError('Training-based validation regeneration requires the LSTM and Transformer sections.')

EXECUTE_CLASSICAL = RUN_CLASSICAL_MODELS
EXECUTE_ORIGINAL_LSTM = RUN_ORIGINAL_LSTM
EXECUTE_IMPROVED_LSTM = RUN_IMPROVED_LSTM
EXECUTE_TRANSFORMER = RUN_TRANSFORMER
EXECUTE_ADVANCED = RUN_ADVANCED_MODELS
EXECUTE_FOUNDATION = generation_enabled(RUN_FOUNDATION_MODELS, [
    RESULTS_DIR / 'validated_forecasts.csv'
], 'Bitcoin foundation models')
EXECUTE_VALIDATION_REGENERATION = generation_enabled(RUN_VALIDATION_REGENERATION, [
    RESULTS_DIR / 'persistence_enhanced_lstm_forecast.csv'
], 'Bitcoin validation regeneration')


## Part B — Bitcoin data and EDA

**Source notebook:** [01_EDA.ipynb](01_EDA.ipynb)

Raw minute data, daily aggregation, quality checks, EDA and the canonical split.

The source is provenance only; executable Markdown and Python are merged below.

# Bitcoin Exploratory Data Analysis

This notebook documents the reproducible exploratory data analysis for Domain 1 - Financial Time Series. It prepares the minute-level BTC/USD data for the downstream forecasting notebooks without training any models.

## 1. Import Libraries

In [ ]:
if RUN_EDA:
    from pathlib import Path
    import sys
    
    import matplotlib.pyplot as plt
    import pandas as pd
    
    def find_project_root(start: Path) -> Path:
        current = start.resolve()
        for candidate in [current, *current.parents]:
            if (candidate / "src").is_dir():
                return candidate
        raise FileNotFoundError("Could not locate project root containing src/")
    
    PROJECT_ROOT = find_project_root(Path.cwd())
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.plots import plot_time_series


## 2. Load Raw Dataset

In [ ]:
if RUN_EDA:
    DATA_PATH = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    
    raw_df = load_bitcoin_data(DATA_PATH)
    raw_df.head()


In [ ]:
if RUN_EDA:
    raw_summary = pd.DataFrame(
        {
            "Metric": [
                "Rows",
                "Columns",
                "Start timestamp",
                "End timestamp",
                "Duplicate timestamps",
                "Missing close values",
            ],
            "Value": [
                len(raw_df),
                raw_df.shape[1],
                raw_df["Timestamp"].min(),
                raw_df["Timestamp"].max(),
                raw_df["Timestamp"].duplicated().sum(),
                raw_df["Close"].isna().sum(),
            ],
        }
    )
    raw_summary


## 3. Prepare Daily Bitcoin Series

In [ ]:
if RUN_EDA:
    df_daily = prepare_daily_bitcoin_data(raw_df)
    df_daily.head()


In [ ]:
if RUN_EDA:
    daily_summary = pd.DataFrame(
        {
            "Metric": [
                "Daily rows",
                "Start date",
                "End date",
                "Missing daily close values",
                "Minimum close",
                "Maximum close",
            ],
            "Value": [
                len(df_daily),
                df_daily.index.min(),
                df_daily.index.max(),
                df_daily["Close"].isna().sum(),
                df_daily["Close"].min(),
                df_daily["Close"].max(),
            ],
        }
    )
    daily_summary


## 4. Visualise Daily Close

In [ ]:
if RUN_EDA:
    plot_time_series(df_daily, column="Close");


## 5. Returns and Volatility

In [ ]:
if RUN_EDA:
    daily_returns = df_daily["Close"].pct_change()
    rolling_volatility = daily_returns.rolling(window=30).std()
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    daily_returns.plot(ax=axes[0], linewidth=0.8, title="Bitcoin Daily Returns")
    rolling_volatility.plot(ax=axes[1], linewidth=1.2, title="30-Day Rolling Return Volatility")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()


## 6. Chronological Train-Test Split

In [ ]:
if RUN_EDA:
    target = df_daily["Close"]
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    
    split_summary = pd.DataFrame(
        {
            "Split": ["Train", "Test"],
            "Start": [train.index.min(), test.index.min()],
            "End": [train.index.max(), test.index.max()],
            "Length": [len(train), len(test)],
        }
    )
    split_summary


In [ ]:
if RUN_EDA:
    assert train.index.max() < test.index.min()
    assert len(train) + len(test) == len(target)
    assert test.index.is_monotonic_increasing
    assert train.index.is_monotonic_increasing


## 7. EDA Findings

- The physically present Bitcoin dataset is minute-level OHLCV data.
- The forecasting task uses the daily resampled `Close` series.
- The downstream notebooks use an 80/20 chronological split with no train-test overlap.
- Bitcoin prices are strongly non-stationary, so raw-price neural models require careful diagnostics for range compression and over-smoothing.
- Volatility clustering motivates separate robustness checks for low-volatility, high-volatility, major upward-move, and major downward-move regimes.

## Part C — Baselines and classical models

**Source notebook:** [02_Classical_Models.ipynb](02_Classical_Models.ipynb)

Protocol-limited Naive, moving average, exponential smoothing, ARIMA and diagnostics.

The source is provenance only; executable Markdown and Python are merged below.

# Classical Forecasting Models

> **PROTOCOL-LIMITED / HISTORICAL**  
> These classical forecasts do not have equivalent validated rolling one-step vectors in the frozen Bitcoin comparison. They are retained for historical and methodological context and are not authoritative ranking evidence.

## 1. Import Libraries

In [ ]:
if EXECUTE_CLASSICAL:
    from pathlib import Path
    import sys
    
    import matplotlib.pyplot as plt
    import pandas as pd
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape


## 2. Load Dataset

In [ ]:
if EXECUTE_CLASSICAL:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    
    df_raw = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(df_raw)
    df_daily.head()


## 3. Train-Test Split

In [ ]:
if EXECUTE_CLASSICAL:
    target = df_daily["Close"].dropna().asfreq("D")
    split_idx = int(len(target) * 0.8)
    
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    train.shape, test.shape


## 4. Naive Forecast

In [ ]:
if EXECUTE_CLASSICAL:
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    
    fig, ax = plt.subplots(figsize=(12, 5))
    train.tail(180).plot(ax=ax, label="Train")
    test.plot(ax=ax, label="Test")
    naive_forecast.plot(ax=ax, label="Naive Forecast")
    ax.set_title("Bitcoin Close Price: Naive Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 5. Moving Average Forecast

In [ ]:
if EXECUTE_CLASSICAL:
    moving_average_forecast = (
        target.shift(1)
        .rolling(window=7)
        .mean()
        .reindex(test.index)
        .rename("7-Day Moving Average")
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    train.tail(180).plot(ax=ax, label="Train")
    test.plot(ax=ax, label="Test")
    moving_average_forecast.plot(ax=ax, label="7-Day Moving Average Forecast")
    ax.set_title("Bitcoin Close Price: 7-Day Moving Average Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 6. Exponential Smoothing

In [ ]:
if EXECUTE_CLASSICAL:
    simple_exp_model = SimpleExpSmoothing(
        train,
        initialization_method="estimated",
    ).fit(optimized=True)
    simple_exp_forecast = simple_exp_model.forecast(len(test)).rename("Simple Exponential Smoothing")
    simple_exp_forecast.index = test.index
    
    holt_winters_model = ExponentialSmoothing(
        train,
        trend="add",
        seasonal=None,
        initialization_method="estimated",
    ).fit(optimized=True)
    holt_winters_forecast = holt_winters_model.forecast(len(test)).rename("Holt-Winters")
    holt_winters_forecast.index = test.index
    
    fig, ax = plt.subplots(figsize=(12, 5))
    test.plot(ax=ax, label="Actual")
    simple_exp_forecast.plot(ax=ax, label="Simple Exponential Smoothing")
    holt_winters_forecast.plot(ax=ax, label="Holt-Winters Exponential Smoothing")
    ax.set_title("Exponential Smoothing Forecasts")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 7. ARIMA

In [ ]:
if EXECUTE_CLASSICAL:
    arima_order = (1, 1, 1)
    arima_model = ARIMA(train, order=arima_order).fit()
    arima_forecast = arima_model.forecast(steps=len(test)).rename("ARIMA(1,1,1)")
    arima_forecast.index = test.index
    
    fig, ax = plt.subplots(figsize=(12, 5))
    test.plot(ax=ax, label="Actual")
    arima_forecast.plot(ax=ax, label="ARIMA(1,1,1) Forecast")
    ax.set_title("Bitcoin Close Price: ARIMA Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 8. Evaluation Metrics

In [ ]:
if EXECUTE_CLASSICAL:
    def evaluate_forecast(y_true, y_pred):
        return {
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
        }
    
    
    forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "Simple Exponential Smoothing": simple_exp_forecast,
        "Holt-Winters Exponential Smoothing": holt_winters_forecast,
        "ARIMA(1,1,1)": arima_forecast,
    }
    
    metrics_table = pd.DataFrame(
        [evaluate_forecast(test, forecast) for forecast in forecasts.values()],
        index=forecasts.keys(),
    )
    
    metrics_table


## 9. Model Comparison

In [ ]:
if EXECUTE_CLASSICAL:
    fig, ax = plt.subplots(figsize=(12, 5))
    test.plot(ax=ax, label="Actual", linewidth=2)
    
    for model_name, forecast in forecasts.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("Classical Forecast Comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    metrics_table.sort_values("RMSE")


## 10. Forecast Diagnostics

In [ ]:
if EXECUTE_CLASSICAL:
    diagnostics = pd.DataFrame(
        {
            "y_test": y_test,
            "Naive": naive_forecast,
            "7-Day Moving Average": moving_average_forecast,
            "Simple Exponential Smoothing": simple_exp_forecast,
            "Holt-Winters": holt_winters_forecast,
            "ARIMA(1,1,1)": arima_forecast,
        }
    )
    
    print("First 10 forecast diagnostics:")
    print(diagnostics.head(10).to_string())
    
    print("Naive forecast equals one-step lagged actuals:")
    print(naive_forecast.equals(target.shift(1).reindex(y_test.index)))
    
    print("ARIMA forecast index aligns with y_test:")
    print(arima_forecast.index.equals(y_test.index))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    
    for model_name, forecast in forecasts.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("Forecast Diagnostics: Test Set Alignment")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


The naive forecast is now implemented as true one-step persistence by using the previous observed close for each test timestamp. The diagnostic check should return `True` when `naive_forecast` equals `target.shift(1).reindex(y_test.index)`. ARIMA forecasts are explicitly assigned the same index as `y_test`, so the index alignment check should also return `True`.

## 11. Key Findings

- The naive forecast provides a simple persistence baseline.
- The 7-day moving average forecast smooths recent price behavior and can be compared against the naive baseline using the metrics table.
- Simple Exponential Smoothing, Holt-Winters Exponential Smoothing, and ARIMA add classical statistical baselines for comparison.
- Lower MAE, RMSE, MAPE, and sMAPE values indicate stronger out-of-sample performance on the chronological test set.

## Part D1 — Original raw-price LSTM

**Source notebook:** [03_Deep_Learning_LSTM.ipynb](03_Deep_Learning_LSTM.ipynb)

Exploratory LSTM architecture, training, forecasts and diagnostics.

The source is provenance only; executable Markdown and Python are merged below.

# LSTM Forecasting

> **EXPLORATORY — RAW-PRICE LSTM**  
> This historical raw-price experiment is non-authoritative. The frozen Bitcoin comparison uses the saved Persistence-Enhanced LSTM vector in `results/validated_forecasts.csv`.

## 1. Import Libraries

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    from pathlib import Path
    import sys
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, LSTM
    from tensorflow.keras.models import Sequential
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    tf.random.set_seed(42)
    np.random.seed(42)


## 2. Load Dataset

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    
    df_raw = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(df_raw)
    target = df_daily["Close"].dropna().asfreq("D")
    
    df_daily.head()


## 3. Train-Test Split

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    split_idx = int(len(target) * 0.8)
    
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    
    train.shape, test.shape


## 4. Data Scaling

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    train_values = train.to_numpy().reshape(-1, 1)
    test_values = test.to_numpy().reshape(-1, 1)
    
    train_scaled = scaler.fit_transform(train_values)
    test_scaled = scaler.transform(test_values)


## 5. Sequence Creation

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    LOOKBACK = 30
    
    
    def create_sequences(values, lookback):
        X, y = [], []
        for i in range(lookback, len(values)):
            X.append(values[i - lookback : i])
            y.append(values[i])
        return np.array(X), np.array(y)
    
    
    X_train, y_train = create_sequences(train_scaled, LOOKBACK)
    
    combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
    X_test, y_test_scaled = create_sequences(combined_scaled, LOOKBACK)
    y_test = test.copy()
    
    X_train.shape, y_train.shape, X_test.shape, y_test.shape


## 6. Build LSTM Model

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    model = Sequential(
        [
            LSTM(32, input_shape=(LOOKBACK, 1)),
            Dense(1),
        ]
    )
    
    model.compile(optimizer="adam", loss="mse")
    model.summary()


## 7. Train Model

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )
    
    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stopping],
        shuffle=False,
    )
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history["loss"], label="Train Loss")
    ax.plot(history.history["val_loss"], label="Validation Loss")
    ax.set_title("LSTM Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 8. Generate Forecasts

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    lstm_predictions_scaled = model.predict(X_test)
    lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).ravel()
    lstm_forecast = pd.Series(lstm_predictions, index=test.index, name="LSTM")
    
    fig, ax = plt.subplots(figsize=(12, 5))
    train.tail(180).plot(ax=ax, label="Train")
    test.plot(ax=ax, label="Test")
    lstm_forecast.plot(ax=ax, label="LSTM Forecast")
    ax.set_title("Bitcoin Close Price: LSTM Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 9. Evaluation Metrics

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    def evaluate_forecast(y_true, y_pred):
        return {
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
        }
    
    
    lstm_metrics = pd.DataFrame(
        [evaluate_forecast(y_test, lstm_forecast)],
        index=["LSTM"],
    )
    
    lstm_metrics


## 10. Compare with Classical Benchmark

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    moving_average_forecast = (
        target.shift(1)
        .rolling(window=7)
        .mean()
        .reindex(test.index)
        .rename("7-Day Moving Average")
    )
    
    benchmark_forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "LSTM": lstm_forecast,
    }
    
    comparison_table = pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in benchmark_forecasts.values()],
        index=benchmark_forecasts.keys(),
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    
    for model_name, forecast in benchmark_forecasts.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("LSTM vs Classical Forecast Benchmarks")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    comparison_table.sort_values("RMSE")


## 11. Key Findings

- This notebook starts with a simple single-layer LSTM using a 30-day lookback window.
- The LSTM forecast is evaluated on the same chronological test period as the classical baselines.
- Naive and 7-day moving average forecasts provide lightweight benchmarks for judging whether the LSTM adds value.
- Lower MAE, RMSE, MAPE, and sMAPE values indicate stronger out-of-sample performance.

## 12. LSTM Diagnostics

In [ ]:
if EXECUTE_ORIGINAL_LSTM:
    diagnostics = pd.DataFrame(
        {
            "Actual": y_test,
            "LSTM": lstm_forecast,
            "Naive": naive_forecast,
        }
    )
    
    print("First 10 actual values:")
    print(y_test.head(10).to_string())
    
    print("\nFirst 10 LSTM predictions:")
    print(lstm_forecast.head(10).to_string())
    
    print("\nFirst 10 Naive predictions:")
    print(naive_forecast.head(10).to_string())
    
    print("\nTrain shape:", train.shape)
    print("Test shape:", test.shape)
    print("X_train sequence shape:", X_train.shape)
    print("y_train sequence shape:", y_train.shape)
    print("X_test sequence shape:", X_test.shape)
    print("y_test shape:", y_test.shape)
    print("Scaling range:", scaler.feature_range)
    print("Train scaled min/max:", float(train_scaled.min()), float(train_scaled.max()))
    print("Test scaled min/max:", float(test_scaled.min()), float(test_scaled.max()))
    
    same_day_correlation = y_test.corr(lstm_forecast)
    lag_correlation = y_test.corr(lstm_forecast.shift(1))
    lstm_volatility_ratio = lstm_forecast.diff().std() / y_test.diff().std()
    
    print("\nPrediction lag check:")
    print("Actual vs same-day LSTM correlation:", same_day_correlation)
    print("Actual vs previous-day LSTM correlation:", lag_correlation)
    
    print("\nUnderfitting check:")
    print("Final training loss:", history.history["loss"][-1])
    print("Final validation loss:", history.history["val_loss"][-1])
    print("LSTM MAE / Naive MAE:", mae(y_test, lstm_forecast) / mae(y_test, naive_forecast))
    
    print("\nOver-smoothing check:")
    print("LSTM prediction std:", lstm_forecast.std())
    print("Actual test std:", y_test.std())
    print("LSTM daily-change std / Actual daily-change std:", lstm_volatility_ratio)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.85)
    lstm_forecast.plot(ax=ax, label="LSTM", alpha=0.85)
    ax.set_title("LSTM Diagnostics: Actual vs Naive vs LSTM")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history["loss"], label="Train Loss")
    ax.plot(history.history["val_loss"], label="Validation Loss")
    ax.set_title("LSTM Training Loss History")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


Bitcoin closing prices are highly persistent, noisy, and regime-dependent, so a one-step naive baseline can be very hard to beat. A simple single-layer LSTM trained only on univariate close prices may underperform because it can smooth sharp moves, lag turning points, and fail to learn enough market context from a short 30-day window. If the diagnostics show low prediction volatility relative to the actual series or validation loss that does not improve meaningfully, the model is likely underfitting or over-smoothing rather than capturing useful nonlinear structure.

## Part D2 — Improved LSTM experiments

**Source notebook:** [03b_LSTM_Improved.ipynb](03b_LSTM_Improved.ipynb)

Original benchmark reconstruction and improved LSTM remain distinct.

The source is provenance only; executable Markdown and Python are merged below.

# Improved LSTM Forecasting

> **EXPLORATORY — EXPERIMENTAL**  
> This improved LSTM experiment is retained for provenance but is not part of the authoritative saved-vector ranking.

## 1. Import Libraries

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    from pathlib import Path
    import sys
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
    from tensorflow.keras.models import Sequential
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    tf.random.set_seed(42)
    np.random.seed(42)


## 2. Load Dataset

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    
    df_raw = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(df_raw)
    target = df_daily["Close"].dropna().asfreq("D")
    
    df_daily.head()


## 3. Train-Test Split

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    split_idx = int(len(target) * 0.8)
    
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    train.shape, test.shape


## 4. Data Scaling

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    train_values = train.to_numpy().reshape(-1, 1)
    test_values = test.to_numpy().reshape(-1, 1)
    
    train_scaled = scaler.fit_transform(train_values)
    test_scaled = scaler.transform(test_values)
    
    scaler.feature_range


## 5. Sequence Creation

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    LOOKBACK = 60
    
    
    def create_sequences(values, lookback):
        X, y = [], []
        for i in range(lookback, len(values)):
            X.append(values[i - lookback : i])
            y.append(values[i])
        return np.array(X), np.array(y)
    
    
    X_train, y_train = create_sequences(train_scaled, LOOKBACK)
    
    combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
    X_test, y_test_scaled = create_sequences(combined_scaled, LOOKBACK)
    
    X_train.shape, y_train.shape, X_test.shape, y_test.shape


## 6. Build Original LSTM Benchmark

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    ORIGINAL_LOOKBACK = 30
    
    X_train_original, y_train_original = create_sequences(train_scaled, ORIGINAL_LOOKBACK)
    combined_scaled_original = np.vstack([train_scaled[-ORIGINAL_LOOKBACK:], test_scaled])
    X_test_original, y_test_original_scaled = create_sequences(
        combined_scaled_original,
        ORIGINAL_LOOKBACK,
    )
    
    original_model = Sequential(
        [
            Input(shape=(ORIGINAL_LOOKBACK, 1)),
            LSTM(32),
            Dense(1),
        ]
    )
    
    original_model.compile(optimizer="adam", loss="mse")
    original_model.summary()


## 7. Train Original LSTM Benchmark

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    original_early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    )
    
    original_history = original_model.fit(
        X_train_original,
        y_train_original,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[original_early_stopping],
        shuffle=False,
    )
    
    original_predictions_scaled = original_model.predict(X_test_original)
    original_predictions = scaler.inverse_transform(original_predictions_scaled).ravel()
    original_lstm_forecast = pd.Series(
        original_predictions,
        index=test.index,
        name="Original LSTM",
    )


## 8. Build Improved LSTM Model

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    improved_model = Sequential(
        [
            Input(shape=(LOOKBACK, 1)),
            LSTM(64, return_sequences=True),
            Dropout(0.2),
            LSTM(32),
            Dropout(0.2),
            Dense(1),
        ]
    )
    
    improved_model.compile(optimizer="adam", loss="mse")
    improved_model.summary()


## 9. Train Improved Model

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    )
    
    history = improved_model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stopping],
        shuffle=False,
    )
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history["loss"], label="Train Loss")
    ax.plot(history.history["val_loss"], label="Validation Loss")
    ax.set_title("Improved LSTM Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 10. Generate Improved Forecasts

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    improved_predictions_scaled = improved_model.predict(X_test)
    improved_predictions = scaler.inverse_transform(improved_predictions_scaled).ravel()
    improved_lstm_forecast = pd.Series(
        improved_predictions,
        index=test.index,
        name="Improved LSTM",
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    train.tail(180).plot(ax=ax, label="Train")
    test.plot(ax=ax, label="Test")
    improved_lstm_forecast.plot(ax=ax, label="Improved LSTM Forecast")
    ax.set_title("Bitcoin Close Price: Improved LSTM Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 11. Benchmarks

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    moving_average_forecast = (
        target.shift(1)
        .rolling(window=7)
        .mean()
        .reindex(test.index)
        .rename("7-Day Moving Average")
    )
    
    benchmark_forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "Original LSTM": original_lstm_forecast,
        "Improved LSTM": improved_lstm_forecast,
    }


## 12. Evaluation Metrics

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    def evaluate_forecast(y_true, y_pred):
        return {
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
        }
    
    
    metrics_table = pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in benchmark_forecasts.values()],
        index=benchmark_forecasts.keys(),
    )
    
    metrics_table.sort_values("RMSE")


## 13. Forecast Comparison

In [ ]:
if EXECUTE_IMPROVED_LSTM:
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    
    for model_name, forecast in benchmark_forecasts.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("Improved LSTM vs Benchmarks")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 14. Key Findings
- This improved LSTM uses a 60-day lookback window, two LSTM layers, dropout, and early stopping.
- The 20-epoch cap keeps this as a compact experiment before broader tuning.
- Naive and 7-day moving average forecasts remain essential benchmarks because Bitcoin prices are highly persistent.
- The original LSTM benchmark is trained in this notebook with a 30-day lookback and a single LSTM layer so the comparison is computed on the same test period.

## Part D3 — Transformer experiments

**Source notebook:** [04_Transformers.ipynb](04_Transformers.ipynb)

Original collapse, failure diagnostics and corrected Transformer are retained.

The source is provenance only; executable Markdown and Python are merged below.

# Transformer Forecasting

> **EXPLORATORY FAILURE CASE STUDY**  
> This notebook documents collapsed and over-smoothed Transformer experiments. Its forecasts are non-authoritative and excluded from the frozen ranking.

## Update Notice

The corrected Transformer implementation is included near the end of this notebook under `## 13. Corrected Transformer Model`. It keeps the existing `## Transformer Failure Analysis` section and adds a fixed encoder that projects the 1D input into `d_model=64` before attention and `LayerNormalization`.

## 1. Import Libraries

In [ ]:
if EXECUTE_TRANSFORMER:
    from pathlib import Path
    import sys
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import (
        Dense,
        Dropout,
        GlobalAveragePooling1D,
        Input,
        LayerNormalization,
        MultiHeadAttention,
    )
    from tensorflow.keras.models import Model, Sequential
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    tf.random.set_seed(42)
    np.random.seed(42)


## 2. Load Dataset

In [ ]:
if EXECUTE_TRANSFORMER:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    
    df_raw = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(df_raw)
    target = df_daily["Close"].dropna().asfreq("D")
    
    df_daily.head()


## 3. Train-Test Split

In [ ]:
if EXECUTE_TRANSFORMER:
    split_idx = int(len(target) * 0.8)
    
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    train.shape, test.shape


## 4. Data Preparation

In [ ]:
if EXECUTE_TRANSFORMER:
    LOOKBACK = 30
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_values = train.to_numpy().reshape(-1, 1)
    test_values = test.to_numpy().reshape(-1, 1)
    
    train_scaled = scaler.fit_transform(train_values)
    test_scaled = scaler.transform(test_values)
    
    
    def create_sequences(values, lookback):
        X, y = [], []
        for i in range(lookback, len(values)):
            X.append(values[i - lookback : i])
            y.append(values[i])
        return np.array(X), np.array(y)
    
    
    X_train, y_train = create_sequences(train_scaled, LOOKBACK)
    combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
    X_test, y_test_scaled = create_sequences(combined_scaled, LOOKBACK)
    
    X_train.shape, y_train.shape, X_test.shape, y_test.shape


## 5. Transformer Model

In [ ]:
if EXECUTE_TRANSFORMER:
    def transformer_encoder(inputs, head_size=32, num_heads=2, ff_dim=32, dropout=0.1):
        attention_output = MultiHeadAttention(
            key_dim=head_size,
            num_heads=num_heads,
            dropout=dropout,
        )(inputs, inputs)
        attention_output = Dropout(dropout)(attention_output)
        attention_output = LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
        feed_forward = Dense(ff_dim, activation="relu")(attention_output)
        feed_forward = Dropout(dropout)(feed_forward)
        feed_forward = Dense(inputs.shape[-1])(feed_forward)
        return LayerNormalization(epsilon=1e-6)(attention_output + feed_forward)
    
    
    inputs = Input(shape=(LOOKBACK, 1))
    x = transformer_encoder(inputs)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation="relu")(x)
    outputs = Dense(1)(x)
    
    transformer_model = Model(inputs, outputs)
    transformer_model.compile(optimizer="adam", loss="mse")
    transformer_model.summary()


## 6. Training

In [ ]:
if EXECUTE_TRANSFORMER:
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    )
    
    history = transformer_model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stopping],
        shuffle=False,
    )
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history["loss"], label="Train Loss")
    ax.plot(history.history["val_loss"], label="Validation Loss")
    ax.set_title("Transformer Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 7. Forecast Generation

In [ ]:
if EXECUTE_TRANSFORMER:
    transformer_predictions_scaled = transformer_model.predict(X_test)
    transformer_predictions = scaler.inverse_transform(transformer_predictions_scaled).ravel()
    transformer_forecast = pd.Series(
        transformer_predictions,
        index=test.index,
        name="Transformer",
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    train.tail(180).plot(ax=ax, label="Train")
    test.plot(ax=ax, label="Test")
    transformer_forecast.plot(ax=ax, label="Transformer Forecast")
    ax.set_title("Bitcoin Close Price: Transformer Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


## 8. Evaluation Metrics

In [ ]:
if EXECUTE_TRANSFORMER:
    def evaluate_forecast(y_true, y_pred):
        return {
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
        }
    
    
    transformer_metrics = pd.DataFrame(
        [evaluate_forecast(y_test, transformer_forecast)],
        index=["Transformer"],
    )
    
    transformer_metrics


## 9. Comparison with Classical Models

In [ ]:
if EXECUTE_TRANSFORMER:
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    moving_average_forecast = (
        target.shift(1)
        .rolling(window=7)
        .mean()
        .reindex(test.index)
        .rename("7-Day Moving Average")
    )
    
    classical_forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "Transformer": transformer_forecast,
    }
    
    classical_comparison = pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in classical_forecasts.values()],
        index=classical_forecasts.keys(),
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    
    for model_name, forecast in classical_forecasts.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("Transformer vs Classical Models")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    classical_comparison.sort_values("RMSE")


## 10. Comparison with LSTM Models

In [ ]:
if EXECUTE_TRANSFORMER:
    def fit_lstm_forecast(lookback, layers):
        X_train_lstm, y_train_lstm = create_sequences(train_scaled, lookback)
        combined_lstm = np.vstack([train_scaled[-lookback:], test_scaled])
        X_test_lstm, _ = create_sequences(combined_lstm, lookback)
    
        model = Sequential(layers)
        model.compile(optimizer="adam", loss="mse")
        callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
        model.fit(
            X_train_lstm,
            y_train_lstm,
            epochs=20,
            batch_size=32,
            validation_split=0.1,
            callbacks=[callback],
            shuffle=False,
        )
        predictions_scaled = model.predict(X_test_lstm)
        predictions = scaler.inverse_transform(predictions_scaled).ravel()
        return pd.Series(predictions, index=test.index)
    
    
    original_lstm_forecast = fit_lstm_forecast(
        lookback=30,
        layers=[
            Input(shape=(30, 1)),
            tf.keras.layers.LSTM(32),
            Dense(1),
        ],
    ).rename("Original LSTM")
    
    improved_lstm_forecast = fit_lstm_forecast(
        lookback=60,
        layers=[
            Input(shape=(60, 1)),
            tf.keras.layers.LSTM(64, return_sequences=True),
            Dropout(0.2),
            tf.keras.layers.LSTM(32),
            Dropout(0.2),
            Dense(1),
        ],
    ).rename("Improved LSTM")
    
    lstm_forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "Original LSTM": original_lstm_forecast,
        "Improved LSTM": improved_lstm_forecast,
        "Transformer": transformer_forecast,
    }
    
    lstm_comparison = pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in lstm_forecasts.values()],
        index=lstm_forecasts.keys(),
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    
    for model_name, forecast in lstm_forecasts.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("Transformer vs LSTM Models")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    lstm_comparison.sort_values("RMSE")


## 11. Key Findings

- This notebook starts with a compact Transformer encoder using a 30-day lookback window.
- Naive and 7-day moving average forecasts remain important classical baselines for persistent Bitcoin prices.
- Original and improved LSTM baselines are trained in-notebook so the Transformer is compared on the same split and test period.
- Lower MAE, RMSE, MAPE, and sMAPE values indicate stronger out-of-sample performance.

## 12. Transformer Diagnostics

In [ ]:
if EXECUTE_TRANSFORMER:
    diagnostics = pd.DataFrame(
        {
            "Actual": y_test,
            "Transformer": transformer_forecast,
            "Naive": naive_forecast,
        }
    )
    
    print("First 10 actual values:")
    print(y_test.head(10).to_string())
    
    print("\nFirst 10 Transformer predictions:")
    print(transformer_forecast.head(10).to_string())
    
    print("\nFirst 10 Naive predictions:")
    print(naive_forecast.head(10).to_string())
    
    print("\nX_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("Scaling range:", scaler.feature_range)
    print("Train scaled min/max:", float(train_scaled.min()), float(train_scaled.max()))
    print("Test scaled min/max:", float(test_scaled.min()), float(test_scaled.max()))
    
    print("\nTraining loss history:")
    print(pd.DataFrame(history.history).to_string(index=False))
    
    print("\nPrediction distribution checks:")
    print("Actual mean:", y_test.mean())
    print("Transformer prediction mean:", transformer_forecast.mean())
    print("Actual std:", y_test.std())
    print("Transformer prediction std:", transformer_forecast.std())
    print("Correlation(actual, prediction):", y_test.corr(transformer_forecast))
    
    rescaled_predictions = scaler.transform(transformer_forecast.to_numpy().reshape(-1, 1))
    inverse_scaling_ok = np.allclose(
        rescaled_predictions,
        transformer_predictions_scaled,
        rtol=1e-5,
        atol=1e-6,
    )
    print("\nInverse scaling check passed:", inverse_scaling_ok)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.85)
    transformer_forecast.plot(ax=ax, label="Transformer", alpha=0.85)
    ax.set_title("Transformer Diagnostics: Actual vs Naive vs Transformer")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


A very poor Transformer result can come from several sources. The inverse-scaling check helps rule out a common implementation error in converting scaled predictions back to prices. If inverse scaling is correct but predictions have a mean far from the actual test mean, low variance, or weak correlation with actual prices, the result is more likely due to undertraining and model limitation. This compact encoder has little feature context, a short training budget, and no explicit trend or volatility inputs, so it may underfit Bitcoin's noisy, regime-dependent price dynamics and fail to beat the one-step naive benchmark.

In [ ]:
if EXECUTE_TRANSFORMER:
    print("\nTransformer forecast min/max:")
    print(transformer_forecast.min(), transformer_forecast.max())
    
    print("\nActual min/max:")
    print(y_test.min(), y_test.max())
    
    print("\nTransformer daily-change std / Actual daily-change std:")
    print(transformer_forecast.diff().std() / y_test.diff().std())


## Transformer Failure Analysis

In [ ]:
if EXECUTE_TRANSFORMER:
    print("Transformer model summary:")
    transformer_model.summary()
    
    final_train_loss = history.history["loss"][-1]
    final_val_loss = history.history["val_loss"][-1]
    predictions_scaled_flat = transformer_predictions_scaled.ravel()
    predictions_unscaled_flat = transformer_forecast.to_numpy()
    scaled_constant = np.allclose(
        predictions_scaled_flat,
        predictions_scaled_flat[0],
        rtol=1e-7,
        atol=1e-8,
    )
    unscaled_constant = np.allclose(
        predictions_unscaled_flat,
        predictions_unscaled_flat[0],
        rtol=1e-7,
        atol=1e-8,
    )
    
    print("\nFinal train loss:", final_train_loss)
    print("Final validation loss:", final_val_loss)
    print("Prediction std before inverse scaling:", predictions_scaled_flat.std())
    print("Prediction std after inverse scaling:", predictions_unscaled_flat.std())
    print("Prediction min before inverse scaling:", predictions_scaled_flat.min())
    print("Prediction max before inverse scaling:", predictions_scaled_flat.max())
    print("Prediction min after inverse scaling:", predictions_unscaled_flat.min())
    print("Prediction max after inverse scaling:", predictions_unscaled_flat.max())
    print("Predictions are not constant before inverse scaling:", not scaled_constant)
    print("Predictions are not constant after inverse scaling:", not unscaled_constant)
    
    print("\nArchitecture checks:")
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("Input feature dimension:", X_train.shape[-1])
    print("Transformer model input shape:", transformer_model.input_shape)
    print("Transformer model output shape:", transformer_model.output_shape)
    print("Feed-forward projection dimension:", X_train.shape[-1])
    
    print("\nCollapse diagnosis:")
    if scaled_constant and unscaled_constant:
        print(
            "Predictions are numerically constant before and after inverse scaling, so inverse scaling is not the cause."
        )
        print(
            "The encoder applies LayerNormalization over the final feature axis after projecting back to one feature."
        )
        print(
            "Because each timestep has only one feature, LayerNormalization normalizes each scalar with zero variance."
        )
        print(
            "That collapses the encoder representation to a constant tensor, leaving the Dense output head to learn a constant bias."
        )
    elif unscaled_constant:
        print(
            "Predictions become constant after inverse scaling; inspect the scaler and prediction precision before changing the model."
        )
    else:
        print(
            "Predictions are not constant at one or both stages; investigate training dynamics and scaling further."
        )


The Transformer collapse is an implementation issue in the current architecture, not an inverse-scaling issue. The predictions are already numerically constant before inverse scaling, and the inverse-scaling round-trip check above confirms that scaling is behaving consistently. The sequence generation and training loop produce non-empty train/test tensors and a fitted history, but the encoder projects the feed-forward block back to `inputs.shape[-1]`, which is `1`, then applies `LayerNormalization` over that single-feature axis. With only one value to normalize per timestep, the variance is zero and the normalized representation becomes constant. After global average pooling, the output layer can only learn a nearly constant bias, which explains the flat forecast. A corrected Transformer should first project the one-dimensional price input into a wider model dimension before attention and normalization, keep that wider representation through the encoder, and only project back to a scalar at the final output layer.

## 13. Corrected Transformer Model

The corrected model projects the one-dimensional price sequence into a wider `d_model` representation before applying attention, residual connections, and layer normalization. This avoids normalizing over a single-feature axis inside the Transformer encoder. The model only projects back to a single value at the final `Dense(1)` output layer.

In [ ]:
if EXECUTE_TRANSFORMER:
    D_MODEL = 64
    NUM_HEADS = 4
    FF_DIM = 128
    DROPOUT_RATE = 0.1
    
    
    def corrected_transformer_encoder(inputs, d_model=64, num_heads=4, ff_dim=128, dropout=0.1):
        attention_output = MultiHeadAttention(
            key_dim=d_model // num_heads,
            num_heads=num_heads,
            dropout=dropout,
        )(inputs, inputs)
        attention_output = Dropout(dropout)(attention_output)
        x = LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
        feed_forward = Dense(ff_dim, activation="relu")(x)
        feed_forward = Dropout(dropout)(feed_forward)
        feed_forward = Dense(d_model)(feed_forward)
        return LayerNormalization(epsilon=1e-6)(x + feed_forward)
    
    
    corrected_inputs = Input(shape=(LOOKBACK, 1))
    x = Dense(D_MODEL)(corrected_inputs)
    x = corrected_transformer_encoder(
        x,
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        dropout=DROPOUT_RATE,
    )
    x = GlobalAveragePooling1D()(x)
    x = Dropout(DROPOUT_RATE)(x)
    x = Dense(32, activation="relu")(x)
    corrected_outputs = Dense(1)(x)
    
    corrected_transformer_model = Model(corrected_inputs, corrected_outputs)
    corrected_transformer_model.compile(optimizer="adam", loss="mse")
    corrected_transformer_model.summary()
    
    corrected_early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    )
    
    corrected_history = corrected_transformer_model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[corrected_early_stopping],
        shuffle=False,
    )
    
    corrected_predictions_scaled = corrected_transformer_model.predict(X_test)
    corrected_predictions = scaler.inverse_transform(corrected_predictions_scaled).ravel()
    corrected_transformer_forecast = pd.Series(
        corrected_predictions,
        index=test.index,
        name="Corrected Transformer",
    )
    
    corrected_metrics = evaluate_forecast(y_test, corrected_transformer_forecast)
    corrected_diagnostics = pd.DataFrame(
        {
            "Actual": y_test,
            "Corrected Transformer": corrected_transformer_forecast,
        }
    )
    
    print("First 10 actual values:")
    print(y_test.head(10).to_string())
    
    print("\nFirst 10 corrected Transformer predictions:")
    print(corrected_transformer_forecast.head(10).to_string())
    
    print("\nCorrected Transformer prediction min/max:")
    print(corrected_transformer_forecast.min(), corrected_transformer_forecast.max())
    
    print("\nCorrected Transformer prediction std:")
    print(corrected_transformer_forecast.std())
    
    print("\nCorrelation with actual:")
    print(y_test.corr(corrected_transformer_forecast))
    
    print("\nCorrected Transformer metrics:")
    corrected_metrics_table = pd.DataFrame([corrected_metrics], index=["Corrected Transformer"])
    print(corrected_metrics_table.to_string())
    
    corrected_comparison_forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "Original LSTM": original_lstm_forecast,
        "Improved LSTM": improved_lstm_forecast,
        "Collapsed Transformer v1": transformer_forecast,
        "Corrected Transformer": corrected_transformer_forecast,
    }
    
    corrected_comparison = pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in corrected_comparison_forecasts.values()],
        index=corrected_comparison_forecasts.keys(),
    )
    
    print("\nCorrected model comparison:")
    print(corrected_comparison.sort_values("RMSE").to_string())
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.85)
    corrected_transformer_forecast.plot(ax=ax, label="Corrected Transformer", alpha=0.85)
    ax.set_title("Corrected Transformer Forecast vs Actual and Naive")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(corrected_history.history["loss"], label="Train Loss")
    ax.plot(corrected_history.history["val_loss"], label="Validation Loss")
    ax.set_title("Corrected Transformer Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    corrected_comparison.sort_values("RMSE")


The corrected Transformer is structurally valid because attention, feed-forward layers, residual connections, and `LayerNormalization` now operate in a `d_model`-wide representation instead of a one-dimensional feature axis. The forecast should be treated as an experimental deep-learning benchmark rather than a guaranteed improvement over naive persistence: Bitcoin close prices are noisy, non-stationary, and often hard to beat with short univariate neural models. If the corrected predictions have non-zero variance and finite correlation/metrics, the collapse has been fixed; whether the model is useful depends on its comparison with the naive and moving-average baselines.

## Part E — Advanced and compatibility-only models

**Source notebook:** [05_Advanced_Forecasting_Models.ipynb](05_Advanced_Forecasting_Models.ipynb)

SARIMA and Transformer code are executable; Prophet and NeuralForecast remain compatibility-only.

The source is provenance only; executable Markdown and Python are merged below.

# Advanced Forecasting Models

> **COMPATIBILITY / UNAVAILABLE-MODEL SCAFFOLD**  
> Moirai, PatchTST, and iTransformer were not successfully evaluated in the completed environment. This scaffold contains no authoritative model result.

## 1. Introduction

This notebook extends the Bitcoin forecasting benchmark beyond the earlier classical, LSTM, and vanilla Transformer experiments. It keeps the same daily Bitcoin Close target and the same 80/20 chronological train-test split so model scores remain comparable across notebooks.

Models that cannot run in the current environment are documented explicitly as **Not run in current environment**. No metrics are invented for unavailable packages.

In [ ]:
if EXECUTE_ADVANCED:
    from pathlib import Path
    import importlib.util
    import sys
    import warnings
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import (
        Dense,
        Dropout,
        GlobalAveragePooling1D,
        Input,
        LayerNormalization,
        MultiHeadAttention,
    )
    from tensorflow.keras.models import Model, Sequential
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    warnings.filterwarnings("ignore")
    tf.random.set_seed(42)
    np.random.seed(42)
    
    pd.set_option("display.float_format", "{:.6f}".format)


## 2. Load Dataset

In [ ]:
if EXECUTE_ADVANCED:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    
    raw_df = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(raw_df)
    target = df_daily["Close"].asfreq("D").dropna().rename("Close")
    
    target.head(), target.tail(), target.shape


## 3. Train-Test Split

In [ ]:
if EXECUTE_ADVANCED:
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    print("Train period:", train.index.min(), "to", train.index.max())
    print("Test period:", test.index.min(), "to", test.index.max())
    print("Train shape:", train.shape)
    print("Test shape:", test.shape)


## 4. Classical Extensions

In [ ]:
if EXECUTE_ADVANCED:
    def evaluate_forecast(y_true, y_pred):
        return {
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
        }
    
    
    forecasts = {}
    model_status = {}
    
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    moving_average_forecast = (
        target.shift(1)
        .rolling(window=7)
        .mean()
        .reindex(test.index)
        .rename("7-Day Moving Average")
    )
    
    forecasts["Naive"] = naive_forecast
    forecasts["7-Day Moving Average"] = moving_average_forecast
    model_status["Naive"] = "Run"
    model_status["7-Day Moving Average"] = "Run"
    
    arima_model = SARIMAX(
        train,
        order=(1, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    arima_results = arima_model.fit(disp=False)
    arima_forecast = arima_results.forecast(steps=len(test))
    arima_forecast.index = test.index
    arima_forecast = arima_forecast.rename("ARIMA(1,1,1)")
    
    forecasts["ARIMA(1,1,1)"] = arima_forecast
    model_status["ARIMA(1,1,1)"] = "Run"
    
    pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in forecasts.values()],
        index=forecasts.keys(),
    ).sort_values("RMSE")


## 5. SARIMA

In [ ]:
if EXECUTE_ADVANCED:
    sarima_model = SARIMAX(
        train,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarima_results = sarima_model.fit(disp=False)
    sarima_forecast = sarima_results.forecast(steps=len(test))
    sarima_forecast.index = test.index
    sarima_forecast = sarima_forecast.rename("SARIMA")
    
    forecasts["SARIMA"] = sarima_forecast
    model_status["SARIMA"] = "Run: SARIMAX(order=(1,1,1), seasonal_order=(1,0,1,7))"
    
    pd.DataFrame([evaluate_forecast(y_test, sarima_forecast)], index=["SARIMA"])


## 6. Prophet

In [ ]:
if EXECUTE_ADVANCED:
    prophet_available = importlib.util.find_spec("prophet") is not None
    
    if prophet_available:
        from prophet import Prophet
    
        prophet_train = train.reset_index()
        prophet_train.columns = ["ds", "y"]
        prophet_train["ds"] = prophet_train["ds"].dt.tz_localize(None)
    
        prophet_model = Prophet(
            weekly_seasonality=True,
            yearly_seasonality=True,
            daily_seasonality=False,
        )
        prophet_model.fit(prophet_train)
    
        future = pd.DataFrame({"ds": test.index.tz_localize(None)})
        prophet_predictions = prophet_model.predict(future)
        prophet_forecast = pd.Series(
            prophet_predictions["yhat"].to_numpy(),
            index=test.index,
            name="Prophet",
        )
    
        forecasts["Prophet"] = prophet_forecast
        model_status["Prophet"] = "Run"
        display(pd.DataFrame([evaluate_forecast(y_test, prophet_forecast)], index=["Prophet"]))
    else:
        model_status["Prophet"] = "Not run in current environment: package `prophet` is not installed. Install with `pip install prophet`."
        print(model_status["Prophet"])


## 7. Transformer-Based Models

In [ ]:
if EXECUTE_ADVANCED:
    LOOKBACK = 30
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_scaled = scaler.fit_transform(train.to_numpy().reshape(-1, 1))
    test_scaled = scaler.transform(test.to_numpy().reshape(-1, 1))
    
    
    def create_sequences(values, lookback):
        X, y = [], []
        for i in range(lookback, len(values)):
            X.append(values[i - lookback : i])
            y.append(values[i])
        return np.array(X), np.array(y)
    
    
    X_train, y_train = create_sequences(train_scaled, LOOKBACK)
    combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
    X_test, _ = create_sequences(combined_scaled, LOOKBACK)
    
    
    def fit_lstm_forecast(lookback, layers, name):
        X_train_lstm, y_train_lstm = create_sequences(train_scaled, lookback)
        combined_lstm = np.vstack([train_scaled[-lookback:], test_scaled])
        X_test_lstm, _ = create_sequences(combined_lstm, lookback)
    
        model = Sequential(layers)
        model.compile(optimizer="adam", loss="mse")
        callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
        model.fit(
            X_train_lstm,
            y_train_lstm,
            epochs=20,
            batch_size=32,
            validation_split=0.1,
            callbacks=[callback],
            shuffle=False,
            verbose=0,
        )
        predictions_scaled = model.predict(X_test_lstm, verbose=0)
        predictions = scaler.inverse_transform(predictions_scaled).ravel()
        return pd.Series(predictions, index=test.index, name=name)
    
    
    original_lstm_forecast = fit_lstm_forecast(
        lookback=30,
        layers=[
            Input(shape=(30, 1)),
            tf.keras.layers.LSTM(32),
            Dense(1),
        ],
        name="Original LSTM",
    )
    
    improved_lstm_forecast = fit_lstm_forecast(
        lookback=60,
        layers=[
            Input(shape=(60, 1)),
            tf.keras.layers.LSTM(64, return_sequences=True),
            Dropout(0.2),
            tf.keras.layers.LSTM(32),
            Dropout(0.2),
            Dense(1),
        ],
        name="Improved LSTM",
    )
    
    forecasts["Original LSTM"] = original_lstm_forecast
    forecasts["Improved LSTM"] = improved_lstm_forecast
    model_status["Original LSTM"] = "Run"
    model_status["Improved LSTM"] = "Run"
    
    
    def corrected_transformer_encoder(inputs, d_model=64, num_heads=4, ff_dim=128, dropout=0.1):
        attention_output = MultiHeadAttention(
            key_dim=d_model // num_heads,
            num_heads=num_heads,
            dropout=dropout,
        )(inputs, inputs)
        attention_output = Dropout(dropout)(attention_output)
        x = LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
        feed_forward = Dense(ff_dim, activation="relu")(x)
        feed_forward = Dropout(dropout)(feed_forward)
        feed_forward = Dense(d_model)(feed_forward)
        return LayerNormalization(epsilon=1e-6)(x + feed_forward)
    
    
    D_MODEL = 64
    corrected_inputs = Input(shape=(LOOKBACK, 1))
    x = Dense(D_MODEL)(corrected_inputs)
    x = corrected_transformer_encoder(x, d_model=D_MODEL, num_heads=4, ff_dim=128, dropout=0.1)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation="relu")(x)
    corrected_outputs = Dense(1)(x)
    
    corrected_transformer_model = Model(corrected_inputs, corrected_outputs)
    corrected_transformer_model.compile(optimizer="adam", loss="mse")
    corrected_callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    corrected_history = corrected_transformer_model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[corrected_callback],
        shuffle=False,
        verbose=0,
    )
    
    corrected_predictions_scaled = corrected_transformer_model.predict(X_test, verbose=0)
    corrected_predictions = scaler.inverse_transform(corrected_predictions_scaled).ravel()
    corrected_transformer_forecast = pd.Series(
        corrected_predictions,
        index=test.index,
        name="Corrected Transformer",
    )
    
    forecasts["Corrected Transformer"] = corrected_transformer_forecast
    model_status["Corrected Transformer"] = "Run: d_model=64 corrected encoder"
    
    pd.DataFrame(
        [evaluate_forecast(y_test, forecasts[name]) for name in ["Original LSTM", "Improved LSTM", "Corrected Transformer"]],
        index=["Original LSTM", "Improved LSTM", "Corrected Transformer"],
    ).sort_values("RMSE")


## 8. PatchTST Workflow

In [ ]:
if EXECUTE_ADVANCED:
    patchtst_available = importlib.util.find_spec("neuralforecast") is not None
    
    if patchtst_available:
        from neuralforecast import NeuralForecast
        from neuralforecast.models import PatchTST
    
        nf_train = pd.DataFrame(
            {
                "unique_id": "BTC",
                "ds": train.index.tz_localize(None),
                "y": train.to_numpy(),
            }
        )
    
        patchtst_model = PatchTST(
            h=len(test),
            input_size=LOOKBACK,
            max_steps=100,
        )
        patchtst_nf = NeuralForecast(models=[patchtst_model], freq="D")
        patchtst_nf.fit(df=nf_train)
        patchtst_predictions = patchtst_nf.predict()
    
        patchtst_column = "PatchTST"
        patchtst_forecast = pd.Series(
            patchtst_predictions[patchtst_column].to_numpy(),
            index=test.index,
            name="PatchTST",
        )
    
        forecasts["PatchTST"] = patchtst_forecast
        model_status["PatchTST"] = "Run via neuralforecast"
        display(pd.DataFrame([evaluate_forecast(y_test, patchtst_forecast)], index=["PatchTST"]))
    else:
        model_status["PatchTST"] = "Not run in current environment: no usable PatchTST package found. A common option is `neuralforecast` with `PatchTST`."
        print(model_status["PatchTST"])


## 9. iTransformer Workflow

In [ ]:
if EXECUTE_ADVANCED:
    itransformer_available = importlib.util.find_spec("neuralforecast") is not None
    
    if itransformer_available:
        from neuralforecast import NeuralForecast
        from neuralforecast.models import iTransformer
    
        nf_train = pd.DataFrame(
            {
                "unique_id": "BTC",
                "ds": train.index.tz_localize(None),
                "y": train.to_numpy(),
            }
        )
    
        itransformer_model = iTransformer(
            h=len(test),
            input_size=LOOKBACK,
            max_steps=100,
        )
        itransformer_nf = NeuralForecast(models=[itransformer_model], freq="D")
        itransformer_nf.fit(df=nf_train)
        itransformer_predictions = itransformer_nf.predict()
    
        itransformer_column = "iTransformer"
        itransformer_forecast = pd.Series(
            itransformer_predictions[itransformer_column].to_numpy(),
            index=test.index,
            name="iTransformer",
        )
    
        forecasts["iTransformer"] = itransformer_forecast
        model_status["iTransformer"] = "Run via neuralforecast"
        display(pd.DataFrame([evaluate_forecast(y_test, itransformer_forecast)], index=["iTransformer"]))
    else:
        model_status["iTransformer"] = "Not run in current environment: no usable iTransformer package found. A common option is `neuralforecast` with `iTransformer`."
        print(model_status["iTransformer"])


## 10. Optional Informer / Autoformer Notes

Informer and Autoformer are useful references for long-horizon Transformer-style forecasting, but they are not implemented here unless a compatible local package is already available and easy to use. In this environment, the required libraries are not assumed. Add them only after deciding on a specific framework and validating its data format, horizon handling, and reproducibility controls.

Suggested future workflow:

1. Choose a maintained implementation, such as a forecasting library that exposes Informer/Autoformer models.
2. Convert the Bitcoin Close series into the required long-format dataset.
3. Use the same 80/20 chronological split and 30-day input context where possible.
4. Report metrics only after the model actually runs locally.

## 11. Evaluation Metrics

In [ ]:
if EXECUTE_ADVANCED:
    metrics_table = pd.DataFrame(
        [evaluate_forecast(y_test, forecast) for forecast in forecasts.values()],
        index=forecasts.keys(),
    ).sort_values("RMSE")
    
    model_status_table = pd.DataFrame.from_dict(
        model_status,
        orient="index",
        columns=["Status"],
    )
    
    print("Model run status:")
    display(model_status_table)
    
    print("Evaluation metrics for models that ran:")
    metrics_table


## 12. Model Comparison

In [ ]:
if EXECUTE_ADVANCED:
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    
    plot_models = [
        "Naive",
        "7-Day Moving Average",
        "ARIMA(1,1,1)",
        "SARIMA",
        "Corrected Transformer",
    ]
    
    for model_name in plot_models:
        if model_name in forecasts:
            forecasts[model_name].plot(ax=ax, label=model_name, alpha=0.85)
    
    ax.set_title("Advanced Forecasting Benchmark")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    metrics_table


## 13. Key Findings

This notebook is designed to separate runnable local benchmarks from placeholder workflows. SARIMA and ARIMA extend the classical baseline family with statsmodels. Prophet, PatchTST, and iTransformer are included as environment-gated workflows: if their packages are unavailable, they are marked as **Not run in current environment** and excluded from the metrics table.

The most important benchmark remains the naive one-step persistence forecast. Any advanced model should be judged against that baseline before being treated as useful for Bitcoin Close forecasting.

## Part F — Chronos and TimesFM

**Source notebook:** [05_Foundation_Models.ipynb](05_Foundation_Models.ipynb)

Actual loading, inference, alignment, uncertainty and diagnostics code is retained.

The source is provenance only; executable Markdown and Python are merged below.

# Time-Series Foundation Models

## 1. Research Objective

Evaluate whether time-series foundation models can be used responsibly on the same Bitcoin daily Close forecasting task as the rest of this project. This notebook deliberately separates compatibility checks, runnable forecasts, diagnostics, and non-runnable implementation plans so that no model result is invented.

Scope:
- Dataset: Bitcoin daily Close series.
- Split: same 80/20 chronological split used elsewhere in the project.
- Primary protocol: rolling one-step forecasting when the model interface and runtime make it practical.
- Foundation models considered: Chronos, TimesFM, and Moirai.


## 2. Environment and Compatibility Check

In [ ]:
if EXECUTE_FOUNDATION:
    from pathlib import Path
    import importlib.util
    import platform
    import sys
    import time
    import warnings
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    pd.set_option("display.float_format", "{:.6f}".format)
    warnings.filterwarnings("ignore")


In [ ]:
if EXECUTE_FOUNDATION:
    def package_available(module_name):
        return importlib.util.find_spec(module_name) is not None
    
    
    try:
        import torch
    except Exception:
        torch = None
    
    try:
        import psutil
    except Exception:
        psutil = None
    
    total_ram_gb = psutil.virtual_memory().total / 1024**3 if psutil else np.nan
    available_ram_gb = psutil.virtual_memory().available / 1024**3 if psutil else np.nan
    
    environment_summary = pd.DataFrame(
        [
            {"Item": "Python version", "Value": platform.python_version()},
            {"Item": "Platform", "Value": platform.platform()},
            {"Item": "PyTorch version", "Value": getattr(torch, "__version__", "Not installed")},
            {"Item": "CUDA available", "Value": bool(torch.cuda.is_available()) if torch else False},
            {"Item": "Total RAM GB", "Value": round(total_ram_gb, 2) if psutil else "psutil not installed"},
            {"Item": "Available RAM GB", "Value": round(available_ram_gb, 2) if psutil else "psutil not installed"},
            {"Item": "chronos package available", "Value": package_available("chronos")},
            {"Item": "timesfm package available", "Value": package_available("timesfm")},
            {"Item": "uni2ts package available", "Value": package_available("uni2ts")},
        ]
    )
    environment_summary


In [ ]:
if EXECUTE_FOUNDATION:
    compatibility_table = pd.DataFrame(
        [
            {
                "Model": "Chronos",
                "Source": "amazon-science/chronos-forecasting",
                "Supported Python": ">=3.10",
                "Required Packages": "chronos-forecasting, torch>=2.2,<3, transformers>=4.41,<6, accelerate>=1.1,<2",
                "Smallest Suitable Model": "amazon/chronos-bolt-tiny or amazon/chronos-t5-tiny",
                "Model Size": "9M parameters for Chronos-Bolt tiny; 8M for Chronos T5 tiny",
                "Expected RAM": "Small model may be CPU-feasible, but model download plus inference needs several GB free RAM",
                "CPU Feasibility": "Potentially practical for tiny/mini models, not practical with very low available RAM",
                "Local Execution Practical Now": package_available("chronos") and available_ram_gb >= 4,
                "Current Blocker": "Package not installed or insufficient free RAM",
            },
            {
                "Model": "TimesFM",
                "Source": "google-research/timesfm",
                "Supported Python": ">=3.10",
                "Required Packages": "timesfm[torch], torch>=2.0.0, huggingface_hub, safetensors",
                "Smallest Suitable Model": "google/timesfm-2.5-200m-pytorch",
                "Model Size": "200M parameters",
                "Expected RAM": "Likely several GB free RAM for weights and inference buffers",
                "CPU Feasibility": "Possible but expected to be slow; less practical than Chronos tiny",
                "Local Execution Practical Now": package_available("timesfm") and available_ram_gb >= 6,
                "Current Blocker": "Package not installed; model is larger than Chronos tiny",
            },
            {
                "Model": "Moirai",
                "Source": "SalesforceAIResearch/uni2ts",
                "Supported Python": ">=3.10 in project metadata, but dependencies pin torch>=2.1,<2.5 and numpy~=1.26",
                "Required Packages": "uni2ts, torch>=2.1,<2.5, lightning>=2.0, gluonts~=0.14.3, datasets~=2.17.1",
                "Smallest Suitable Model": "moirai small variants, if compatible environment is created",
                "Model Size": "Varies by checkpoint; not selected for current environment",
                "Expected RAM": "Several GB free RAM; dependency stack is heavier than Chronos",
                "CPU Feasibility": "Research workflow is possible but not practical in current Python 3.13 / Torch 2.12.1 environment",
                "Local Execution Practical Now": package_available("uni2ts") and torch is not None and getattr(torch, "__version__", "0") < "2.5",
                "Current Blocker": "Current torch 2.12.1 violates uni2ts torch<2.5 requirement; package not installed",
            },
        ]
    )
    compatibility_table


Compatibility sources used for the table above:
- Chronos official repository metadata: `requires-python >=3.10`, `torch>=2.2,<3`, model sizes including Chronos-Bolt tiny at 9M parameters.
- TimesFM official repository metadata: `requires-python >=3.10`, `timesfm[torch]`, `torch>=2.0.0`, 200M PyTorch example.
- Moirai / Uni2TS official repository metadata: `requires-python >=3.10`, but dependency pins include `torch>=2.1,<2.5` and `numpy~=1.26.0`, which are not compatible with this project's current Torch 2.12.1 environment.

The notebook does not install packages. It only runs a foundation model if the package is already available and the environment passes basic feasibility checks.


## 3. Load Dataset

In [ ]:
if EXECUTE_FOUNDATION:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    raw_df = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(raw_df)
    target = df_daily["Close"].asfreq("D").dropna().rename("Close")
    
    dataset_summary = pd.DataFrame(
        [
            {"Item": "Full series length", "Value": len(target)},
            {"Item": "Start date", "Value": target.index.min()},
            {"Item": "End date", "Value": target.index.max()},
            {"Item": "Index sorted", "Value": target.index.is_monotonic_increasing},
            {"Item": "Duplicate timestamps", "Value": target.index.duplicated().any()},
            {"Item": "Missing values", "Value": int(target.isna().sum())},
        ]
    )
    dataset_summary


## 4. Train-Test Split

In [ ]:
if EXECUTE_FOUNDATION:
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    split_summary = pd.DataFrame(
        [
            {"Item": "Train length", "Value": len(train)},
            {"Item": "Test length", "Value": len(test)},
            {"Item": "Train start", "Value": train.index.min()},
            {"Item": "Train end", "Value": train.index.max()},
            {"Item": "Test start", "Value": test.index.min()},
            {"Item": "Test end", "Value": test.index.max()},
            {"Item": "Train/test overlap", "Value": len(train.index.intersection(test.index)) > 0},
            {"Item": "Expected train end 2023-08-11", "Value": train.index.max() == pd.Timestamp("2023-08-11", tz="UTC")},
            {"Item": "Expected test start 2023-08-12", "Value": test.index.min() == pd.Timestamp("2023-08-12", tz="UTC")},
        ]
    )
    split_summary


## 5. Forecasting Protocol

The main benchmark uses rolling one-step forecasting where technically supported:

1. At test date `t`, the model may use all observations strictly earlier than `t`.
2. It forecasts the next one day only.
3. After the true value for `t` is observed, that value may be appended to history for the next test step.
4. No forecast may use `actual[t]` or any later test value when predicting `t`.

For foundation models, this is often expensive because it may require repeated inference across the full test period. If a model can only be run practically as a short-horizon or recursive multi-step forecast, the protocol is labeled separately and excluded from direct rolling one-step comparisons.


In [ ]:
if EXECUTE_FOUNDATION:
    def metric_row(model, forecast, actual=y_test, protocol="Rolling one-step", status="Run"):
        aligned = pd.concat([actual.rename("Actual"), forecast.rename("Forecast")], axis=1).dropna()
        if aligned.empty:
            return {
                "Model": model,
                "Protocol": protocol,
                "Status": status,
                "MAE": np.nan,
                "RMSE": np.nan,
                "MAPE": np.nan,
                "sMAPE": np.nan,
            }
        return {
            "Model": model,
            "Protocol": protocol,
            "Status": status,
            "MAE": mae(aligned["Actual"], aligned["Forecast"]),
            "RMSE": rmse(aligned["Actual"], aligned["Forecast"]),
            "MAPE": mape(aligned["Actual"], aligned["Forecast"]),
            "sMAPE": smape(aligned["Actual"], aligned["Forecast"]),
        }
    
    
    def forecast_diagnostics(model, forecast, actual=y_test, inference_time_seconds=np.nan, protocol="Rolling one-step"):
        aligned = pd.concat([actual.rename("Actual"), forecast.rename("Forecast")], axis=1).dropna()
        if aligned.empty:
            return {
                "Model": model,
                "Protocol": protocol,
                "Forecast Length": 0,
                "Index Alignment": False,
                "Missing Values": np.nan,
                "Constant Prediction": np.nan,
                "Prediction Std": np.nan,
                "Actual Std": np.nan,
                "Correlation": np.nan,
                "Daily Change Std Ratio": np.nan,
                "Inference Time Seconds": inference_time_seconds,
            }
    
        prediction_change_std = aligned["Forecast"].diff().std()
        actual_change_std = aligned["Actual"].diff().std()
        return {
            "Model": model,
            "Protocol": protocol,
            "Forecast Length": len(forecast),
            "Index Alignment": forecast.index.equals(actual.index),
            "Missing Values": int(forecast.isna().sum()),
            "Constant Prediction": forecast.nunique(dropna=True) <= 1,
            "Prediction Min": forecast.min(),
            "Prediction Max": forecast.max(),
            "Prediction Mean": forecast.mean(),
            "Prediction Std": forecast.std(),
            "Actual Std": aligned["Actual"].std(),
            "Correlation": aligned["Actual"].corr(aligned["Forecast"]),
            "Daily Change Std Ratio": prediction_change_std / actual_change_std if actual_change_std else np.nan,
            "Inference Time Seconds": inference_time_seconds,
        }
    
    
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    moving_average_7 = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("7-Day Moving Average")
    
    baseline_metrics = pd.DataFrame(
        [
            metric_row("Naive", naive_forecast),
            metric_row("7-Day Moving Average", moving_average_7),
        ]
    ).set_index("Model")
    baseline_metrics


## 6. Chronos

Chronos is evaluated first because the smallest Chronos-Bolt model is the most plausible local foundation-model candidate. The recommended starting point is a zero-shot tiny model. In this current environment, the notebook will only run Chronos if:

- the `chronos` package is installed,
- PyTorch is available,
- enough free RAM is available,
- and the user intentionally allows pretrained model downloads outside this notebook setup.

If those checks fail, the notebook records Chronos as not run instead of inventing metrics.


In [ ]:
if EXECUTE_FOUNDATION:
    chronos_status = {
        "Model": "Chronos",
        "Zero-shot or Fine-tuned": "Zero-shot",
        "Package Available": package_available("chronos"),
        "Selected Model": "amazon/chronos-bolt-tiny",
        "Protocol Attempted": "Rolling one-step",
        "Runnable Now": package_available("chronos") and torch is not None,
        "Blocker": None,
    }
    
    if not package_available("chronos"):
        chronos_status["Blocker"] = "chronos-forecasting is not installed in the current environment."
    elif torch is None:
        chronos_status["Blocker"] = "PyTorch is not importable."
    else:
        chronos_status["Blocker"] = "Runnable guard passed for Chronos-Bolt-Tiny rolling one-step evaluation."
    
    pd.DataFrame([chronos_status])


In [ ]:
if EXECUTE_FOUNDATION:
    # RUN_CHRONOS is controlled by the master control panel.
    chronos_forecast = None
    chronos_inference_time_seconds = np.nan
    
    if RUN_CHRONOS:
        if not package_available("chronos"):
            raise ImportError("Install chronos-forecasting before running Chronos.")
        if available_ram_gb < 4:
            raise MemoryError(f"Available RAM is only {available_ram_gb:.2f} GB; Chronos run skipped.")
    
        import torch
        from chronos import BaseChronosPipeline
    
        pipeline = BaseChronosPipeline.from_pretrained(
            "amazon/chronos-bolt-tiny",
            device_map="cpu",
            torch_dtype=torch.float32,
        )
    
        predictions = []
        start_time = time.perf_counter()
        history = train.copy()
        for forecast_date, actual_value in test.items():
            context = torch.tensor(history.to_numpy(dtype=np.float32))
            forecast = pipeline.predict(context, prediction_length=1)
            point_prediction = np.median(forecast[0].detach().cpu().numpy(), axis=0)[0]
            predictions.append(point_prediction)
            history.loc[forecast_date] = actual_value
        chronos_inference_time_seconds = time.perf_counter() - start_time
        chronos_forecast = pd.Series(predictions, index=test.index, name="Chronos")
    else:
        print("Chronos not run. Set RUN_CHRONOS=True only after package, RAM, and model-download checks pass.")


In [ ]:
if EXECUTE_FOUNDATION:
    chronos_results = []
    chronos_diagnostics = []
    
    if chronos_forecast is not None:
        chronos_results.append(metric_row("Chronos", chronos_forecast, protocol="Rolling one-step", status="Run"))
        chronos_diagnostics.append(
            forecast_diagnostics(
                "Chronos",
                chronos_forecast,
                inference_time_seconds=chronos_inference_time_seconds,
                protocol="Rolling one-step",
            )
        )
    
        fig, ax = plt.subplots(figsize=(12, 5))
        y_test.plot(ax=ax, label="Actual", linewidth=2)
        naive_forecast.plot(ax=ax, label="Naive", alpha=0.8)
        chronos_forecast.plot(ax=ax, label="Chronos", alpha=0.8)
        ax.set_title("Actual vs Chronos vs Naive")
        ax.set_xlabel("Date")
        ax.set_ylabel("Bitcoin Close")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        chronos_results.append(
            {
                "Model": "Chronos",
                "Protocol": "Rolling one-step target protocol",
                "Status": "Not run in current environment",
                "MAE": np.nan,
                "RMSE": np.nan,
                "MAPE": np.nan,
                "sMAPE": np.nan,
            }
        )
    
    pd.DataFrame(chronos_results).set_index("Model")


## Full Chronos Rolling One-Step Evaluation

This section evaluates `amazon/chronos-bolt-tiny` using the same Bitcoin daily test period and a fair rolling one-step-ahead protocol. Each forecast for date `t` uses a fixed 128-day context ending strictly before `t`. Context windows are constructed first and evaluated in batches to avoid one model call per test row.

Protocol labels:
- Model type: zero-shot foundation model.
- Evaluation: rolling one-step.
- Device: CPU.
- Dtype: `torch.float32`.


In [ ]:
if EXECUTE_FOUNDATION:
    from chronos import BaseChronosPipeline
    
    CHRONOS_MODEL_ID = "amazon/chronos-bolt-tiny"
    CHRONOS_CONTEXT_LENGTH = 128
    CHRONOS_BATCH_SIZE = 32
    CHRONOS_QUANTILE_LEVELS = [0.1, 0.5, 0.9]
    
    assert train.index.max() == pd.Timestamp("2023-08-11", tz="UTC")
    assert test.index.min() == pd.Timestamp("2023-08-12", tz="UTC")
    
    chronos_contexts = []
    chronos_context_start_dates = []
    chronos_context_end_dates = []
    chronos_last_context_values = []
    
    for forecast_date in test.index:
        history_before_forecast = target[target.index < forecast_date]
        context_window = history_before_forecast.tail(CHRONOS_CONTEXT_LENGTH)
        assert len(context_window) == CHRONOS_CONTEXT_LENGTH
        assert context_window.index.max() < forecast_date
    
        chronos_contexts.append(context_window.to_numpy(dtype=np.float32))
        chronos_context_start_dates.append(context_window.index.min())
        chronos_context_end_dates.append(context_window.index.max())
        chronos_last_context_values.append(float(context_window.iloc[-1]))
    
    chronos_context_tensor = torch.tensor(np.stack(chronos_contexts), dtype=torch.float32)
    
    context_audit_preview = pd.DataFrame(
        {
            "forecast_date": test.index[:10],
            "context_start_date": chronos_context_start_dates[:10],
            "context_end_date": chronos_context_end_dates[:10],
            "last_context_value": chronos_last_context_values[:10],
            "actual_value": y_test.iloc[:10].to_numpy(),
            "naive_forecast": naive_forecast.iloc[:10].to_numpy(),
        }
    ).set_index("forecast_date")
    
    context_audit_preview


In [ ]:
if EXECUTE_FOUNDATION:
    load_start = time.perf_counter()
    chronos_pipeline = BaseChronosPipeline.from_pretrained(
        CHRONOS_MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
    )
    chronos_model_load_seconds = time.perf_counter() - load_start
    
    ram_after_chronos_load_gb = psutil.virtual_memory().available / 1024**3 if psutil else np.nan
    
    quantile_batches = []
    mean_batches = []
    inference_start = time.perf_counter()
    
    for start in range(0, len(chronos_context_tensor), CHRONOS_BATCH_SIZE):
        batch = chronos_context_tensor[start : start + CHRONOS_BATCH_SIZE]
        batch_quantiles, batch_mean = chronos_pipeline.predict_quantiles(
            batch,
            prediction_length=1,
            quantile_levels=CHRONOS_QUANTILE_LEVELS,
        )
        quantile_batches.append(batch_quantiles.detach().cpu())
        mean_batches.append(batch_mean.detach().cpu())
    
    chronos_inference_time_seconds = time.perf_counter() - inference_start
    
    chronos_quantile_array = torch.cat(quantile_batches, dim=0).numpy()
    chronos_mean_array = torch.cat(mean_batches, dim=0).numpy()
    
    assert chronos_quantile_array.shape == (len(test), 1, len(CHRONOS_QUANTILE_LEVELS))
    assert chronos_mean_array.shape == (len(test), 1)
    
    q50_position = CHRONOS_QUANTILE_LEVELS.index(0.5)
    chronos_forecast = pd.Series(
        chronos_quantile_array[:, 0, q50_position],
        index=test.index,
        name="Chronos-Bolt-Tiny",
    )
    chronos_mean_forecast = pd.Series(
        chronos_mean_array[:, 0],
        index=test.index,
        name="Chronos-Bolt-Tiny Mean",
    )
    chronos_quantiles = {
        level: pd.Series(chronos_quantile_array[:, 0, position], index=test.index, name=f"q{level:g}")
        for position, level in enumerate(CHRONOS_QUANTILE_LEVELS)
    }
    
    chronos_first_10 = context_audit_preview.copy()
    chronos_first_10["chronos_forecast"] = chronos_forecast.iloc[:10].to_numpy()
    
    assert all(end_date < forecast_date for end_date, forecast_date in zip(chronos_context_end_dates, test.index))
    assert chronos_forecast.index.equals(test.index)
    assert len(chronos_forecast) == len(test)
    assert chronos_forecast.isna().sum() == 0
    assert np.isfinite(chronos_forecast.to_numpy()).all()
    
    chronos_first_10


In [ ]:
if EXECUTE_FOUNDATION:
    chronos_error = y_test - chronos_forecast
    chronos_metrics = pd.DataFrame(
        [
            {
                "Model": "Chronos-Bolt-Tiny",
                "Protocol": "Rolling one-step",
                "MAE": mae(y_test, chronos_forecast),
                "RMSE": rmse(y_test, chronos_forecast),
                "MAPE": mape(y_test, chronos_forecast),
                "sMAPE": smape(y_test, chronos_forecast),
                "Relative MAE vs Naive": mae(y_test, chronos_forecast) / mae(y_test, naive_forecast),
                "Prediction Mean": chronos_forecast.mean(),
                "Prediction Std": chronos_forecast.std(),
                "Actual Std": y_test.std(),
                "Prediction Daily-Change Std": chronos_forecast.diff().std(),
                "Actual Daily-Change Std": y_test.diff().std(),
                "Correlation With Actual": y_test.corr(chronos_forecast),
                "Inference Time Seconds": chronos_inference_time_seconds,
                "Average Inference Seconds Per Forecast": chronos_inference_time_seconds / len(test),
                "Model Load Seconds": chronos_model_load_seconds,
                "Available RAM After Load GB": ram_after_chronos_load_gb,
            }
        ]
    ).set_index("Model")
    
    chronos_metrics


### Save Validated Chronos and Baseline Artifacts

The validated Chronos-Bolt-Tiny, Actual, and Naive vectors are exported for downstream paired-error tests.

In [ ]:
if EXECUTE_FOUNDATION:
    results_dir = PROJECT_ROOT / "results"
    results_dir.mkdir(exist_ok=True)
    
    baseline_artifact = pd.DataFrame(
        {
            "Timestamp": y_test.index,
            "Actual": y_test.to_numpy(dtype=float),
            "Naive": naive_forecast.to_numpy(dtype=float),
        }
    )
    chronos_artifact = pd.DataFrame(
        {
            "Timestamp": chronos_forecast.index,
            "Chronos_Bolt_Tiny": chronos_forecast.to_numpy(dtype=float),
        }
    )
    
    assert baseline_artifact.shape == (1061, 3)
    assert chronos_artifact.shape == (1061, 2)
    assert pd.DatetimeIndex(baseline_artifact["Timestamp"]).equals(y_test.index)
    assert pd.DatetimeIndex(chronos_artifact["Timestamp"]).equals(y_test.index)
    assert baseline_artifact["Timestamp"].is_unique
    assert chronos_artifact["Timestamp"].is_unique
    assert baseline_artifact["Timestamp"].is_monotonic_increasing
    assert chronos_artifact["Timestamp"].is_monotonic_increasing
    assert not baseline_artifact.isna().any().any()
    assert not chronos_artifact.isna().any().any()
    assert np.isfinite(baseline_artifact[["Actual", "Naive"]].to_numpy(dtype=float)).all()
    assert np.isfinite(chronos_artifact["Chronos_Bolt_Tiny"].to_numpy(dtype=float)).all()
    
    baseline_path = results_dir / "baseline_forecasts.csv"
    chronos_path = results_dir / "chronos_bolt_tiny_forecast.csv"
    baseline_artifact.to_csv(baseline_path, index=False)
    chronos_artifact.to_csv(chronos_path, index=False)
    
    print(f"Saved {baseline_path} with shape {baseline_artifact.shape}")
    print(f"Saved {chronos_path} with shape {chronos_artifact.shape}")


In [ ]:
if EXECUTE_FOUNDATION:
    persistence_lstm_metrics = {
        "MAE": 1326.330606,
        "RMSE": 1890.119319,
        "MAPE": 1.792268,
        "sMAPE": 1.798938,
    }
    
    comparison_rows = [
        {
            "Model": "Naive",
            "Protocol": "Rolling one-step",
            "MAE": mae(y_test, naive_forecast),
            "RMSE": rmse(y_test, naive_forecast),
            "MAPE": mape(y_test, naive_forecast),
            "sMAPE": smape(y_test, naive_forecast),
        },
        {
            "Model": "7-Day Moving Average",
            "Protocol": "Rolling one-step",
            "MAE": mae(y_test, moving_average_7),
            "RMSE": rmse(y_test, moving_average_7),
            "MAPE": mape(y_test, moving_average_7),
            "sMAPE": smape(y_test, moving_average_7),
        },
        {
            "Model": "Persistence-Enhanced LSTM",
            "Protocol": "Audited rolling one-step",
            **persistence_lstm_metrics,
        },
        {
            "Model": "Chronos-Bolt-Tiny",
            "Protocol": "Rolling one-step",
            "MAE": mae(y_test, chronos_forecast),
            "RMSE": rmse(y_test, chronos_forecast),
            "MAPE": mape(y_test, chronos_forecast),
            "sMAPE": smape(y_test, chronos_forecast),
        },
    ]
    
    chronos_comparison_table = pd.DataFrame(comparison_rows).sort_values("RMSE").set_index("Model")
    chronos_comparison_table


In [ ]:
if EXECUTE_FOUNDATION:
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.8)
    chronos_forecast.plot(ax=ax, label="Chronos-Bolt-Tiny", alpha=0.8)
    ax.set_title("Full Test Period: Actual vs Naive vs Chronos")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.head(90).plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.head(90).plot(ax=ax, label="Naive", alpha=0.8)
    chronos_forecast.head(90).plot(ax=ax, label="Chronos-Bolt-Tiny", alpha=0.8)
    ax.set_title("First 90 Test Days")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.tail(90).plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.tail(90).plot(ax=ax, label="Naive", alpha=0.8)
    chronos_forecast.tail(90).plot(ax=ax, label="Chronos-Bolt-Tiny", alpha=0.8)
    ax.set_title("Last 90 Test Days")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    chronos_error.plot(ax=ax)
    ax.set_title("Chronos Error Over Time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Actual - Forecast")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    y_test.plot(kind="hist", bins=50, alpha=0.55, ax=ax, label="Actual")
    chronos_forecast.plot(kind="hist", bins=50, alpha=0.55, ax=ax, label="Chronos")
    ax.set_title("Prediction Distribution vs Actual Distribution")
    ax.set_xlabel("Bitcoin Close")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if EXECUTE_FOUNDATION:
    def metrics_for_subset(name, mask):
        actual_subset = y_test.loc[mask]
        forecast_subset = chronos_forecast.loc[mask]
        return {
            "Regime": name,
            "Rows": len(actual_subset),
            "MAE": mae(actual_subset, forecast_subset),
            "RMSE": rmse(actual_subset, forecast_subset),
            "MAPE": mape(actual_subset, forecast_subset),
            "sMAPE": smape(actual_subset, forecast_subset),
        }
    
    
    returns = target.pct_change().reindex(test.index)
    rolling_volatility = returns.rolling(window=30, min_periods=10).std()
    low_vol_threshold = rolling_volatility.quantile(0.25)
    high_vol_threshold = rolling_volatility.quantile(0.75)
    movement_threshold = returns.abs().quantile(0.75)
    
    regime_table = pd.DataFrame(
        [
            metrics_for_subset("Low volatility", rolling_volatility <= low_vol_threshold),
            metrics_for_subset("High volatility", rolling_volatility >= high_vol_threshold),
            metrics_for_subset("Major upward movement", returns >= movement_threshold),
            metrics_for_subset("Major downward movement", returns <= -movement_threshold),
        ]
    ).set_index("Regime")
    regime_table


In [ ]:
if EXECUTE_FOUNDATION:
    segment_labels = pd.qcut(np.arange(len(test)), q=3, labels=["Earlier test period", "Middle test period", "Later test period"])
    segment_table = pd.DataFrame(
        [
            {
                "Segment": segment,
                "Rows": int((segment_labels == segment).sum()),
                "Start Date": y_test.index[segment_labels == segment].min(),
                "End Date": y_test.index[segment_labels == segment].max(),
                "MAE": mae(y_test.loc[segment_labels == segment], chronos_forecast.loc[segment_labels == segment]),
                "RMSE": rmse(y_test.loc[segment_labels == segment], chronos_forecast.loc[segment_labels == segment]),
                "MAPE": mape(y_test.loc[segment_labels == segment], chronos_forecast.loc[segment_labels == segment]),
                "sMAPE": smape(y_test.loc[segment_labels == segment], chronos_forecast.loc[segment_labels == segment]),
            }
            for segment in segment_labels.categories
        ]
    ).set_index("Segment")
    segment_table


In [ ]:
if EXECUTE_FOUNDATION:
    def interval_coverage(lower_level, upper_level, label):
        lower = chronos_quantiles[lower_level]
        upper = chronos_quantiles[upper_level]
        covered = ((y_test >= lower) & (y_test <= upper)).mean()
        width = (upper - lower).mean()
        return {
            "Interval": label,
            "Lower Quantile": lower_level,
            "Upper Quantile": upper_level,
            "Coverage": covered,
            "Average Width": width,
            "Quantiles Verified From API Request": (lower_level in CHRONOS_QUANTILE_LEVELS) and (upper_level in CHRONOS_QUANTILE_LEVELS),
        }
    
    
    uncertainty_rows = [interval_coverage(0.1, 0.9, "80%")]
    if 0.025 in CHRONOS_QUANTILE_LEVELS and 0.975 in CHRONOS_QUANTILE_LEVELS:
        uncertainty_rows.append(interval_coverage(0.025, 0.975, "95%"))
    else:
        uncertainty_rows.append(
            {
                "Interval": "95%",
                "Lower Quantile": np.nan,
                "Upper Quantile": np.nan,
                "Coverage": np.nan,
                "Average Width": np.nan,
                "Quantiles Verified From API Request": False,
                "Note": "Not available: Chronos-Bolt exposes trained quantiles from 0.1 to 0.9 in this API.",
            }
        )
    uncertainty_table = pd.DataFrame(uncertainty_rows).set_index("Interval")
    uncertainty_table


In [ ]:
if EXECUTE_FOUNDATION:
    forecast_to_actual_std_ratio = chronos_forecast.std() / y_test.std()
    daily_change_std_ratio = chronos_forecast.diff().std() / y_test.diff().std()
    range_compression_ratio = (chronos_forecast.max() - chronos_forecast.min()) / (y_test.max() - y_test.min())
    
    chronos_diagnostics_table = pd.DataFrame(
        [
            {"Check": "Constant prediction", "Value": bool(chronos_forecast.nunique(dropna=True) <= 1)},
            {"Check": "Range compression ratio", "Value": range_compression_ratio},
            {"Check": "Forecast-to-actual std ratio", "Value": forecast_to_actual_std_ratio},
            {"Check": "Daily-change std ratio", "Value": daily_change_std_ratio},
            {"Check": "Exact index alignment", "Value": chronos_forecast.index.equals(y_test.index)},
            {"Check": "Forecast length", "Value": len(chronos_forecast)},
            {"Check": "Test length", "Value": len(y_test)},
            {"Check": "Missing predictions", "Value": int(chronos_forecast.isna().sum())},
            {"Check": "Finite predictions", "Value": bool(np.isfinite(chronos_forecast.to_numpy()).all())},
            {"Check": "Raw quantile output shape", "Value": chronos_quantile_array.shape},
            {"Check": "Mean output shape", "Value": chronos_mean_array.shape},
            {"Check": "Quantile levels requested", "Value": CHRONOS_QUANTILE_LEVELS},
            {"Check": "Inference time seconds", "Value": chronos_inference_time_seconds},
            {"Check": "Average inference seconds per forecast", "Value": chronos_inference_time_seconds / len(test)},
        ]
    ).set_index("Check")
    
    display(chronos_first_10)
    chronos_diagnostics_table


## 7. TimesFM

TimesFM is tested here with a smoke test only. The package has been installed in the current Python 3.13 project environment and the import check passed.

The smallest available model exposed by the installed `timesfm==2.0.2` package is `google/timesfm-2.5-200m-pytorch`. This section loads that model on CPU, compiles it for a 128-day context and 7-day horizon, and forecasts only the next seven days after the training period.

This is **not** a full rolling evaluation. If the smoke test succeeds, the notebook records TimesFM as locally runnable, but `timesfm_forecast` remains `None` so it is not included in the full benchmark tables yet.


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_status = {
        "Model": "TimesFM",
        "Zero-shot or Fine-tuned": "Zero-shot",
        "Package Available": package_available("timesfm"),
        "Installed Version": "2.0.2" if package_available("timesfm") else None,
        "Selected Model": "google/timesfm-2.5-200m-pytorch",
        "Protocol Attempted": "Rolling one-step",
        "Runnable Now": package_available("timesfm") and torch is not None,
        "Blocker": None,
    }
    
    if not package_available("timesfm"):
        timesfm_status["Blocker"] = "timesfm is not installed in the current environment."
    elif torch is None:
        timesfm_status["Blocker"] = "PyTorch is not importable."
    else:
        timesfm_status["Blocker"] = "Runnable guard passed; full rolling one-step evaluation completed below."
    
    pd.DataFrame([timesfm_status])


In [ ]:
if EXECUTE_FOUNDATION:
    import subprocess
    
    RUN_TIMESFM_SMOKE_TEST = RUN_TIMESFM
    timesfm_forecast = None
    timesfm_inference_time_seconds = np.nan
    timesfm_smoke_forecast = None
    timesfm_smoke_quantiles = None
    
    if RUN_TIMESFM_SMOKE_TEST:
        import timesfm
    
        print("Python version:", sys.version)
        print("Interpreter:", sys.executable)
        print("PyTorch version:", torch.__version__ if torch else "Not installed")
        print("CUDA available:", torch.cuda.is_available() if torch else False)
        print(f"Total RAM GB: {total_ram_gb:.2f}")
        print(f"Available RAM GB before loading: {psutil.virtual_memory().available / 1024**3:.2f}")
        print("TimesFM import passed")
        print("TimesFM version:", getattr(timesfm, "__version__", "unknown"))
    
        pip_check = subprocess.run(
            [sys.executable, "-m", "pip", "check"],
            capture_output=True,
            text=True,
            check=False,
        )
        print("pip check output:")
        print(pip_check.stdout.strip() or pip_check.stderr.strip())
    
        TIMESFM_MODEL_ID = timesfm.TimesFM_2p5_200M_torch.DEFAULT_REPO_ID
        TIMESFM_CONTEXT_LENGTH = 128
        TIMESFM_HORIZON = 7
        timesfm_context = train.tail(TIMESFM_CONTEXT_LENGTH).to_numpy(dtype=np.float32)
    
        ram_before_timesfm_load_gb = psutil.virtual_memory().available / 1024**3
        timesfm_load_start = time.perf_counter()
        timesfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
            TIMESFM_MODEL_ID,
            torch_compile=False,
        )
        timesfm_model.compile(
            timesfm.ForecastConfig(
                max_context=TIMESFM_CONTEXT_LENGTH,
                max_horizon=TIMESFM_HORIZON,
                normalize_inputs=True,
                per_core_batch_size=1,
            )
        )
        timesfm_model_load_seconds = time.perf_counter() - timesfm_load_start
        ram_after_timesfm_load_gb = psutil.virtual_memory().available / 1024**3
    
        timesfm_inference_start = time.perf_counter()
        timesfm_point_forecast, timesfm_quantile_forecast = timesfm_model.forecast(
            horizon=TIMESFM_HORIZON,
            inputs=[timesfm_context],
        )
        timesfm_inference_time_seconds = time.perf_counter() - timesfm_inference_start
    
        timesfm_smoke_forecast = np.asarray(timesfm_point_forecast)[0]
        timesfm_smoke_quantiles = np.asarray(timesfm_quantile_forecast)
    
        print("Selected model:", TIMESFM_MODEL_ID)
        print("Model loading time seconds:", round(timesfm_model_load_seconds, 3))
        print("Inference time seconds:", round(timesfm_inference_time_seconds, 3))
        print("RAM before loading GB:", round(ram_before_timesfm_load_gb, 2))
        print("RAM after loading GB:", round(ram_after_timesfm_load_gb, 2))
        print("Output shape:", timesfm_smoke_forecast.shape)
        print("Quantile output shape:", timesfm_smoke_quantiles.shape)
        print("Forecast values:", timesfm_smoke_forecast.tolist())
        print("Min:", float(np.min(timesfm_smoke_forecast)))
        print("Max:", float(np.max(timesfm_smoke_forecast)))
        print("Mean:", float(np.mean(timesfm_smoke_forecast)))
        print("Std:", float(np.std(timesfm_smoke_forecast)))
        print("Finite prediction check:", bool(np.isfinite(timesfm_smoke_forecast).all()))
        print("Constant prediction check:", bool(np.unique(np.round(timesfm_smoke_forecast, 8)).size <= 1))
    
        timesfm_smoke_index = pd.date_range(
            train.index[-1] + pd.Timedelta(days=1),
            periods=TIMESFM_HORIZON,
            freq="D",
            tz=train.index.tz,
        )
        timesfm_smoke_series = pd.Series(timesfm_smoke_forecast, index=timesfm_smoke_index, name="TimesFM Smoke Forecast")
    
        fig, ax = plt.subplots(figsize=(10, 4))
        train.tail(60).plot(ax=ax, label="Last 60 training observations")
        timesfm_smoke_series.plot(ax=ax, label="TimesFM 7-day forecast", marker="o")
        ax.set_title("TimesFM Smoke Test: Last 60 Context Values + 7 Forecasts")
        ax.set_xlabel("Date")
        ax.set_ylabel("Bitcoin Close")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


## Full TimesFM Rolling One-Step Evaluation

This section evaluates `google/timesfm-2.5-200m-pytorch` using the same Bitcoin daily test period and the same fair rolling one-step protocol used for Chronos. Each forecast for date `t` uses a fixed 128-day context ending strictly before `t`. Context windows are built first, then passed to the TimesFM API as a batch-capable input list.

Protocol labels:
- Model type: zero-shot time-series foundation model.
- Evaluation: rolling one-step.
- Device: CPU.
- Context length: 128 days.
- Horizon: 1 day.


In [ ]:
if EXECUTE_FOUNDATION:
    import timesfm
    
    TIMESFM_MODEL_ID = "google/timesfm-2.5-200m-pytorch"
    TIMESFM_CONTEXT_LENGTH = 128
    TIMESFM_HORIZON = 1
    TIMESFM_PER_CORE_BATCH_SIZE = 32
    TIMESFM_QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    
    assert train.index.max() == pd.Timestamp("2023-08-11", tz="UTC")
    assert test.index.min() == pd.Timestamp("2023-08-12", tz="UTC")
    assert len(test) == 1061
    
    timesfm_contexts = []
    timesfm_context_start_dates = []
    timesfm_context_end_dates = []
    timesfm_last_context_values = []
    
    for forecast_date in test.index:
        history_before_forecast = target[target.index < forecast_date]
        context_window = history_before_forecast.tail(TIMESFM_CONTEXT_LENGTH)
        assert len(context_window) == TIMESFM_CONTEXT_LENGTH
        assert context_window.index.max() < forecast_date
    
        timesfm_contexts.append(context_window.to_numpy(dtype=np.float32))
        timesfm_context_start_dates.append(context_window.index.min())
        timesfm_context_end_dates.append(context_window.index.max())
        timesfm_last_context_values.append(float(context_window.iloc[-1]))
    
    timesfm_audit_preview = pd.DataFrame(
        {
            "forecast_date": test.index[:10],
            "context_start_date": timesfm_context_start_dates[:10],
            "context_end_date": timesfm_context_end_dates[:10],
            "last_context_value": timesfm_last_context_values[:10],
            "actual_value": y_test.iloc[:10].to_numpy(dtype=float),
            "naive_forecast": naive_forecast.iloc[:10].to_numpy(dtype=float),
        }
    )
    timesfm_audit_preview


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_ram_before_load_gb = psutil.virtual_memory().available / 1024**3 if psutil else np.nan
    timesfm_load_start = time.perf_counter()
    timesfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
        TIMESFM_MODEL_ID,
        torch_compile=False,
    )
    timesfm_model.compile(
        timesfm.ForecastConfig(
            max_context=TIMESFM_CONTEXT_LENGTH,
            max_horizon=TIMESFM_HORIZON,
            normalize_inputs=True,
            per_core_batch_size=TIMESFM_PER_CORE_BATCH_SIZE,
        )
    )
    timesfm_model_load_seconds = time.perf_counter() - timesfm_load_start
    timesfm_ram_after_load_gb = psutil.virtual_memory().available / 1024**3 if psutil else np.nan
    
    timesfm_inference_start = time.perf_counter()
    timesfm_point_array, timesfm_quantile_array = timesfm_model.forecast(
        horizon=TIMESFM_HORIZON,
        inputs=timesfm_contexts,
    )
    timesfm_inference_time_seconds = time.perf_counter() - timesfm_inference_start
    
    timesfm_point_array = np.asarray(timesfm_point_array, dtype=float)
    timesfm_quantile_array = np.asarray(timesfm_quantile_array, dtype=float)
    timesfm_point_values = timesfm_point_array[:, 0] if timesfm_point_array.ndim == 2 else timesfm_point_array.reshape(len(test), -1)[:, 0]
    timesfm_forecast = pd.Series(timesfm_point_values, index=test.index, name="TimesFM")
    
    assert len(timesfm_forecast) == 1061
    assert timesfm_forecast.index.equals(y_test.index)
    assert timesfm_forecast.index.is_unique
    assert timesfm_forecast.isna().sum() == 0
    assert np.isfinite(timesfm_forecast.to_numpy(dtype=float)).all()
    
    timesfm_audit_preview["timesfm_forecast"] = timesfm_forecast.iloc[:10].to_numpy(dtype=float)
    assert (pd.to_datetime(timesfm_audit_preview["context_end_date"]) < pd.to_datetime(timesfm_audit_preview["forecast_date"])).all()
    timesfm_audit_preview


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_error = y_test - timesfm_forecast
    timesfm_metrics = pd.DataFrame(
        [
            {
                "Model": "TimesFM",
                "Protocol": "Rolling one-step",
                "MAE": mae(y_test, timesfm_forecast),
                "RMSE": rmse(y_test, timesfm_forecast),
                "MAPE": mape(y_test, timesfm_forecast),
                "sMAPE": smape(y_test, timesfm_forecast),
                "Relative MAE vs Naive": mae(y_test, timesfm_forecast) / mae(y_test, naive_forecast),
                "Prediction Min": timesfm_forecast.min(),
                "Prediction Max": timesfm_forecast.max(),
                "Prediction Mean": timesfm_forecast.mean(),
                "Prediction Std": timesfm_forecast.std(),
                "Actual Std": y_test.std(),
                "Prediction Daily-Change Std": timesfm_forecast.diff().std(),
                "Actual Daily-Change Std": y_test.diff().std(),
                "Daily-Change Std Ratio": timesfm_forecast.diff().std() / y_test.diff().std(),
                "Correlation With Actual": y_test.corr(timesfm_forecast),
                "Model Load Seconds": timesfm_model_load_seconds,
                "Total Inference Seconds": timesfm_inference_time_seconds,
                "Average Inference Seconds Per Forecast": timesfm_inference_time_seconds / len(test),
            }
        ]
    ).set_index("Model")
    timesfm_metrics


In [ ]:
if EXECUTE_FOUNDATION:
    results_dir = PROJECT_ROOT / "results"
    results_dir.mkdir(exist_ok=True)
    
    timesfm_artifact = pd.DataFrame(
        {
            "Timestamp": timesfm_forecast.index,
            "TimesFM": timesfm_forecast.to_numpy(dtype=float),
        }
    )
    
    assert timesfm_artifact.shape == (1061, 2)
    assert timesfm_artifact["Timestamp"].is_unique
    assert timesfm_artifact["Timestamp"].is_monotonic_increasing
    assert not timesfm_artifact.isna().any().any()
    assert np.isfinite(timesfm_artifact["TimesFM"].to_numpy(dtype=float)).all()
    
    timesfm_artifact_path = results_dir / "timesfm_forecast.csv"
    timesfm_artifact.to_csv(timesfm_artifact_path, index=False)
    
    validated_forecast_path = results_dir / "validated_forecasts.csv"
    validated_forecasts = pd.read_csv(validated_forecast_path, parse_dates=["Timestamp"])
    validated_forecasts = validated_forecasts.drop(columns=["TimesFM"], errors="ignore")
    validated_forecasts = validated_forecasts.merge(
        timesfm_artifact,
        on="Timestamp",
        how="inner",
        validate="one_to_one",
    )
    validated_forecasts = validated_forecasts[
        ["Timestamp", "Actual", "Naive", "Persistence_Enhanced_LSTM", "Chronos_Bolt_Tiny", "TimesFM"]
    ]
    
    assert validated_forecasts.shape == (1061, 6)
    assert validated_forecasts["Timestamp"].is_unique
    assert validated_forecasts["Timestamp"].is_monotonic_increasing
    assert not validated_forecasts.isna().any().any()
    assert np.isfinite(
        validated_forecasts[["Actual", "Naive", "Persistence_Enhanced_LSTM", "Chronos_Bolt_Tiny", "TimesFM"]].to_numpy(dtype=float)
    ).all()
    
    validated_forecasts.to_csv(validated_forecast_path, index=False)
    
    artifact_validation = pd.DataFrame(
        [
            {"Artifact": "timesfm_forecast.csv", "Shape": timesfm_artifact.shape, "Valid": True},
            {"Artifact": "validated_forecasts.csv", "Shape": validated_forecasts.shape, "Valid": True},
        ]
    )
    artifact_validation


In [ ]:
if EXECUTE_FOUNDATION:
    authoritative_comparison = pd.DataFrame(
        [
            {
                "Model": "Naive",
                "MAE": mae(validated_forecasts["Actual"], validated_forecasts["Naive"]),
                "RMSE": rmse(validated_forecasts["Actual"], validated_forecasts["Naive"]),
                "MAPE": mape(validated_forecasts["Actual"], validated_forecasts["Naive"]),
                "sMAPE": smape(validated_forecasts["Actual"], validated_forecasts["Naive"]),
            },
            {
                "Model": "Persistence-Enhanced LSTM",
                "MAE": mae(validated_forecasts["Actual"], validated_forecasts["Persistence_Enhanced_LSTM"]),
                "RMSE": rmse(validated_forecasts["Actual"], validated_forecasts["Persistence_Enhanced_LSTM"]),
                "MAPE": mape(validated_forecasts["Actual"], validated_forecasts["Persistence_Enhanced_LSTM"]),
                "sMAPE": smape(validated_forecasts["Actual"], validated_forecasts["Persistence_Enhanced_LSTM"]),
            },
            {
                "Model": "Chronos-Bolt-Tiny",
                "MAE": mae(validated_forecasts["Actual"], validated_forecasts["Chronos_Bolt_Tiny"]),
                "RMSE": rmse(validated_forecasts["Actual"], validated_forecasts["Chronos_Bolt_Tiny"]),
                "MAPE": mape(validated_forecasts["Actual"], validated_forecasts["Chronos_Bolt_Tiny"]),
                "sMAPE": smape(validated_forecasts["Actual"], validated_forecasts["Chronos_Bolt_Tiny"]),
            },
            {
                "Model": "TimesFM",
                "MAE": mae(validated_forecasts["Actual"], validated_forecasts["TimesFM"]),
                "RMSE": rmse(validated_forecasts["Actual"], validated_forecasts["TimesFM"]),
                "MAPE": mape(validated_forecasts["Actual"], validated_forecasts["TimesFM"]),
                "sMAPE": smape(validated_forecasts["Actual"], validated_forecasts["TimesFM"]),
            },
            {
                "Model": "7-Day Moving Average",
                "MAE": mae(y_test, moving_average_7),
                "RMSE": rmse(y_test, moving_average_7),
                "MAPE": mape(y_test, moving_average_7),
                "sMAPE": smape(y_test, moving_average_7),
            },
        ]
    ).set_index("Model")
    
    authoritative_comparison["Relative MAE vs Naive"] = authoritative_comparison["MAE"] / authoritative_comparison.loc["Naive", "MAE"]
    authoritative_comparison.sort_values("RMSE")


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_returns = y_test.pct_change()
    timesfm_rolling_volatility = timesfm_returns.rolling(window=14, min_periods=7).std()
    timesfm_low_threshold = timesfm_rolling_volatility.quantile(0.33)
    timesfm_high_threshold = timesfm_rolling_volatility.quantile(0.67)
    timesfm_up_threshold = timesfm_returns.quantile(0.80)
    timesfm_down_threshold = timesfm_returns.quantile(0.20)
    
    timesfm_regime_masks = {
        "Low Volatility": timesfm_rolling_volatility <= timesfm_low_threshold,
        "High Volatility": timesfm_rolling_volatility >= timesfm_high_threshold,
        "Major Upward Movement": timesfm_returns >= timesfm_up_threshold,
        "Major Downward Movement": timesfm_returns <= timesfm_down_threshold,
    }
    
    timesfm_regime_rows = []
    for regime_name, mask in timesfm_regime_masks.items():
        regime_index = mask[mask].index
        scores = {
            "MAE": mae(y_test.reindex(regime_index), timesfm_forecast.reindex(regime_index)),
            "RMSE": rmse(y_test.reindex(regime_index), timesfm_forecast.reindex(regime_index)),
            "MAPE": mape(y_test.reindex(regime_index), timesfm_forecast.reindex(regime_index)),
            "sMAPE": smape(y_test.reindex(regime_index), timesfm_forecast.reindex(regime_index)),
            "N": len(regime_index),
        }
        scores.update({"Regime": regime_name})
        timesfm_regime_rows.append(scores)
    
    timesfm_regime_table = pd.DataFrame(timesfm_regime_rows).set_index("Regime")
    timesfm_regime_table


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_segment_rows = []
    for segment_name, segment_index in zip(
        ["Earlier Test Period", "Middle Test Period", "Later Test Period"],
        np.array_split(y_test.index, 3),
    ):
        timesfm_segment_rows.append(
            {
                "Segment": segment_name,
                "MAE": mae(y_test.reindex(segment_index), timesfm_forecast.reindex(segment_index)),
                "RMSE": rmse(y_test.reindex(segment_index), timesfm_forecast.reindex(segment_index)),
                "MAPE": mape(y_test.reindex(segment_index), timesfm_forecast.reindex(segment_index)),
                "sMAPE": smape(y_test.reindex(segment_index), timesfm_forecast.reindex(segment_index)),
                "N": len(segment_index),
            }
        )
    
    timesfm_segment_table = pd.DataFrame(timesfm_segment_rows).set_index("Segment")
    timesfm_segment_table


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_available_quantiles = TIMESFM_QUANTILE_LEVELS
    timesfm_uncertainty_rows = []
    
    if timesfm_quantile_array.ndim == 3 and 0.1 in timesfm_available_quantiles and 0.9 in timesfm_available_quantiles:
        timesfm_lower_80 = pd.Series(
            timesfm_quantile_array[:, 0, timesfm_available_quantiles.index(0.1)],
            index=y_test.index,
            name="TimesFM q0.1",
        )
        timesfm_upper_80 = pd.Series(
            timesfm_quantile_array[:, 0, timesfm_available_quantiles.index(0.9)],
            index=y_test.index,
            name="TimesFM q0.9",
        )
        timesfm_uncertainty_rows.append(
            {
                "Interval": "80%",
                "Available": True,
                "Lower Quantile": 0.1,
                "Upper Quantile": 0.9,
                "Coverage": ((y_test >= timesfm_lower_80) & (y_test <= timesfm_upper_80)).mean(),
                "Average Width": (timesfm_upper_80 - timesfm_lower_80).mean(),
                "Reason": "TimesFM package metadata exposes quantiles 0.1 through 0.9.",
            }
        )
    else:
        timesfm_lower_80 = timesfm_upper_80 = None
        timesfm_uncertainty_rows.append(
            {"Interval": "80%", "Available": False, "Reason": "Required quantiles unavailable."}
        )
    
    timesfm_uncertainty_rows.append(
        {
            "Interval": "95%",
            "Available": False,
            "Lower Quantile": 0.025,
            "Upper Quantile": 0.975,
            "Coverage": np.nan,
            "Average Width": np.nan,
            "Reason": "TimesFM package metadata exposes quantiles 0.1 through 0.9 only.",
        }
    )
    
    timesfm_uncertainty_table = pd.DataFrame(timesfm_uncertainty_rows).set_index("Interval")
    timesfm_uncertainty_table


In [ ]:
if EXECUTE_FOUNDATION:
    timesfm_diagnostics = pd.DataFrame(
        [
            {"Check": "Constant forecast", "Value": bool(timesfm_forecast.nunique(dropna=True) <= 1)},
            {
                "Check": "Range compression ratio",
                "Value": (timesfm_forecast.max() - timesfm_forecast.min()) / (y_test.max() - y_test.min()),
            },
            {"Check": "Prediction std / actual std", "Value": timesfm_forecast.std() / y_test.std()},
            {"Check": "Daily-change std ratio", "Value": timesfm_forecast.diff().std() / y_test.diff().std()},
            {"Check": "Exact alignment", "Value": bool(timesfm_forecast.index.equals(y_test.index))},
            {"Check": "Missing values", "Value": int(timesfm_forecast.isna().sum())},
            {"Check": "Finite predictions", "Value": bool(np.isfinite(timesfm_forecast.to_numpy(dtype=float)).all())},
            {"Check": "Duplicated timestamps", "Value": int(timesfm_forecast.index.duplicated().sum())},
            {"Check": "Forecast length", "Value": len(timesfm_forecast)},
        ]
    )
    timesfm_diagnostics


In [ ]:
if EXECUTE_FOUNDATION:
    chronos_for_plot = pd.Series(
        validated_forecasts["Chronos_Bolt_Tiny"].to_numpy(dtype=float),
        index=y_test.index,
        name="Chronos-Bolt-Tiny",
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.75)
    chronos_for_plot.plot(ax=ax, label="Chronos-Bolt-Tiny", alpha=0.75)
    timesfm_forecast.plot(ax=ax, label="TimesFM", alpha=0.75)
    ax.set_title("Full Test Period: Actual vs Naive vs Chronos vs TimesFM")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.head(90).plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.head(90).plot(ax=ax, label="Naive", alpha=0.75)
    chronos_for_plot.head(90).plot(ax=ax, label="Chronos-Bolt-Tiny", alpha=0.75)
    timesfm_forecast.head(90).plot(ax=ax, label="TimesFM", alpha=0.75)
    ax.set_title("First 90 Test Days")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.tail(90).plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.tail(90).plot(ax=ax, label="Naive", alpha=0.75)
    chronos_for_plot.tail(90).plot(ax=ax, label="Chronos-Bolt-Tiny", alpha=0.75)
    timesfm_forecast.tail(90).plot(ax=ax, label="TimesFM", alpha=0.75)
    ax.set_title("Last 90 Test Days")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    timesfm_error.plot(ax=ax, label="TimesFM Error")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("TimesFM Forecast Error Over Time")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    y_test.plot(kind="hist", bins=50, alpha=0.55, ax=ax, label="Actual")
    timesfm_forecast.plot(kind="hist", bins=50, alpha=0.55, ax=ax, label="TimesFM")
    ax.set_title("TimesFM Prediction Distribution vs Actual")
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    if timesfm_lower_80 is not None and timesfm_upper_80 is not None:
        subset_index = y_test.head(120).index
        fig, ax = plt.subplots(figsize=(12, 5))
        y_test.reindex(subset_index).plot(ax=ax, label="Actual", linewidth=2)
        timesfm_forecast.reindex(subset_index).plot(ax=ax, label="TimesFM", alpha=0.85)
        ax.fill_between(
            subset_index,
            timesfm_lower_80.reindex(subset_index).to_numpy(dtype=float),
            timesfm_upper_80.reindex(subset_index).to_numpy(dtype=float),
            alpha=0.2,
            label="80% interval",
        )
        ax.set_title("TimesFM 80% Uncertainty Interval: First 120 Test Days")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


## 8. Moirai

Moirai is evaluated here as a compatibility and smoke-test candidate only. The active environment is Python `3.13.2`, CPU-only PyTorch `2.12.1+cpu`.

Installation was attempted with the active project interpreter:

```powershell
.\.venv\Scripts\python.exe -m pip install uni2ts
```

The install did **not** complete. The official `uni2ts==2.0.0` package depends on `numpy~=1.26.0`. Under Python 3.13, pip attempted to build `numpy==1.26.4` from source because a compatible wheel was not available. That build failed because the local Windows environment does not have an MSVC compiler toolchain (`cl`, `clang-cl`, etc.).

Because Uni2TS did not install, no Moirai imports, model loading, or smoke-test forecasts are reported. No validated forecast artifacts are modified.


In [ ]:
if EXECUTE_FOUNDATION:
    moirai_install_attempt = {
        "Model": "Moirai / Uni2TS",
        "Package Attempted": "uni2ts",
        "Install Command": r".\.venv\Scripts\python.exe -m pip install uni2ts",
        "Install Result": "Failed",
        "Installed Package Version": "Not installed",
        "Import Result": "Not attempted because installation failed",
        "Selected Model": "None",
        "Protocol Attempted": "Installation and compatibility check only",
        "Package Available": package_available("uni2ts"),
        "Runnable Now": False,
        "Exact Compatibility Blocker": (
            "uni2ts==2.0.0 requires numpy~=1.26.0; on Python 3.13 pip attempted to build "
            "numpy==1.26.4 from source and failed because no MSVC compiler was available."
        ),
        "Dependency Warning": "Moirai/Uni2TS remains better suited to a separate Python 3.11/3.12 environment with compatible pinned dependencies.",
        "pip check output after failed install": "No broken requirements found.",
    }
    
    pd.DataFrame([moirai_install_attempt])


## 9. Forecast Diagnostics

In [ ]:
if EXECUTE_FOUNDATION:
    diagnostic_rows = []
    diagnostic_rows.append(forecast_diagnostics("Naive", naive_forecast, protocol="Rolling one-step"))
    diagnostic_rows.append(forecast_diagnostics("7-Day Moving Average", moving_average_7, protocol="Rolling one-step"))
    
    if chronos_forecast is not None:
        diagnostic_rows.append(
            forecast_diagnostics(
                "Chronos",
                chronos_forecast,
                inference_time_seconds=chronos_inference_time_seconds,
                protocol="Rolling one-step",
            )
        )
    if timesfm_forecast is not None:
        diagnostic_rows.append(
            forecast_diagnostics(
                "TimesFM",
                timesfm_forecast,
                inference_time_seconds=timesfm_inference_time_seconds,
                protocol="Rolling one-step",
            )
        )
    
    forecast_diagnostics_table = pd.DataFrame(diagnostic_rows).set_index("Model")
    forecast_diagnostics_table


## 10. Evaluation Metrics

In [ ]:
if EXECUTE_FOUNDATION:
    foundation_metric_rows = []
    
    if chronos_forecast is not None:
        foundation_metric_rows.append(metric_row("Chronos", chronos_forecast, protocol="Rolling one-step", status="Run"))
    else:
        foundation_metric_rows.append(
            {
                "Model": "Chronos",
                "Protocol": "Rolling one-step target protocol",
                "Status": "Not run in current environment",
                "MAE": np.nan,
                "RMSE": np.nan,
                "MAPE": np.nan,
                "sMAPE": np.nan,
            }
        )
    
    if timesfm_forecast is not None:
        foundation_metric_rows.append(metric_row("TimesFM", timesfm_forecast, protocol="Rolling one-step", status="Run"))
    else:
        foundation_metric_rows.append(
            {
                "Model": "TimesFM",
                "Protocol": "Rolling one-step target protocol",
                "Status": "Not run in current environment",
                "MAE": np.nan,
                "RMSE": np.nan,
                "MAPE": np.nan,
                "sMAPE": np.nan,
            }
        )
    
    foundation_metric_rows.append(
        {
            "Model": "Moirai",
            "Protocol": "Not attempted",
            "Status": "Not run in current environment",
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE": np.nan,
            "sMAPE": np.nan,
        }
    )
    
    foundation_metrics = pd.DataFrame(foundation_metric_rows).set_index("Model")
    foundation_metrics


## 11. Comparison with Existing Models

This table includes locally computed baselines and reserved rows for the project models requested for comparison. Rows with unavailable forecasts are explicitly marked as not loaded rather than populated with invented values. Collapsed or invalid Transformer forecasts are intentionally excluded from the main comparison.


In [ ]:
if EXECUTE_FOUNDATION:
    comparison_rows = []
    comparison_rows.extend(baseline_metrics.reset_index().to_dict("records"))
    comparison_rows.extend(foundation_metrics.reset_index().to_dict("records"))
    
    existing_model_placeholders = [
        "Original LSTM",
        "Persistence-Enhanced LSTM",
        "ARIMA(1,1,1)",
        "SARIMA",
    ]
    for model_name in existing_model_placeholders:
        comparison_rows.append(
            {
                "Model": model_name,
                "Protocol": "Use audited protocol from prior notebook before direct comparison",
                "Status": "Forecast not loaded in this notebook",
                "MAE": np.nan,
                "RMSE": np.nan,
                "MAPE": np.nan,
                "sMAPE": np.nan,
            }
        )
    
    comparison_table = pd.DataFrame(comparison_rows)
    comparison_table = comparison_table[
        ["Model", "Protocol", "Status", "MAE", "RMSE", "MAPE", "sMAPE"]
    ].sort_values("RMSE", na_position="last")
    comparison_table


In [ ]:
if EXECUTE_FOUNDATION:
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.8)
    moving_average_7.plot(ax=ax, label="7-Day Moving Average", alpha=0.8)
    if chronos_forecast is not None:
        chronos_forecast.plot(ax=ax, label="Chronos", alpha=0.8)
    if timesfm_forecast is not None:
        timesfm_forecast.plot(ax=ax, label="TimesFM", alpha=0.8)
    ax.set_title("Foundation Model Forecasts vs Baselines")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## 12. Key Findings

- The dataset and split are compatible with the rest of the project: the 80/20 split trains through `2023-08-11` and tests from `2023-08-12`.
- Chronos-Bolt-Tiny completed a full rolling one-step evaluation and produced the validated artifact used downstream.
- TimesFM installed successfully in the current Python 3.13.2 environment as `timesfm==2.0.2`; `pip check` reported no broken requirements.
- The full TimesFM rolling one-step evaluation completed all `1,061` forecasts on CPU using `google/timesfm-2.5-200m-pytorch`.
- TimesFM performed between Persistence-Enhanced LSTM and Chronos-Bolt-Tiny in this run, with RMSE higher than Naive but lower than Chronos.
- TimesFM preserved realistic price-scale variability, but its 80% interval coverage was low, so its native uncertainty interval should not be trusted without calibration.
- Moirai / Uni2TS was attempted with `pip install uni2ts`, but installation failed because `uni2ts==2.0.0` requires `numpy~=1.26.0`, which is not compatible as a prebuilt wheel with this Python 3.13 environment. Pip attempted a source build of NumPy and failed due to a missing MSVC compiler.
- No Moirai import, checkpoint load, or smoke-test forecast is reported.
- No foundation-model metrics are reported unless the corresponding model actually runs.
- Collapsed or invalid Transformer forecasts are excluded from the main comparison.


## 13. Limitations

- TimesFM has now been run as a full rolling one-step evaluation, but only with the 200M PyTorch model exposed by `timesfm==2.0.2`.
- The current environment is CPU-only and had limited available RAM during the run.
- TimesFM inference was batched through the model API, but full rolling evaluation is still heavier than Chronos-Bolt-Tiny.
- TimesFM 95% interval coverage is not reported because the installed package metadata exposes quantiles `0.1` through `0.9`, not `0.025` and `0.975`.
- Moirai/Uni2TS could not be installed in the current Python 3.13.2 environment. The exact blocker was the `numpy~=1.26.0` dependency, which triggered a failed source build of NumPy 1.26.4 due to a missing MSVC compiler. A separate Python 3.11 or 3.12 environment remains the clean path for Moirai.


## Part H1 — Bitcoin trustworthiness

**Source notebook:** [06_Trustworthiness.ipynb](06_Trustworthiness.ipynb)

Artifact-based accuracy, regimes, temporal stability, uncertainty, explainability and composites.

The source is provenance only; executable Markdown and Python are merged below.

# Trustworthiness Evaluation

## 1. Research Objective

This notebook is now an artifact-only trustworthiness analysis. It does **not** train, refit, load checkpoints, or rerun any forecasting model. All authoritative model forecasts are loaded from `../results/validated_forecasts.csv`.

The goal is to evaluate point accuracy, robustness, temporal generalisation, uncertainty calibration, explainability, and overall trustworthiness using saved forecast vectors and reproducible deterministic baselines only.


In [ ]:
if RUN_TRUSTWORTHINESS:
    from pathlib import Path
    import sys
    import warnings
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    warnings.filterwarnings("ignore")
    pd.set_option("display.float_format", "{:.6f}".format)
    
    ARTIFACT_ONLY = True
    MODEL_FITTING_EXECUTED = False
    MODEL_CHECKPOINT_LOADED = False
    FORECAST_VECTOR_REGENERATED = False


## 2. Load Authoritative Forecast Artifact

Expected artifact: `../results/validated_forecasts.csv`

Required columns:
- `Timestamp`
- `Actual`
- `Naive`
- `Persistence_Enhanced_LSTM`
- `Chronos_Bolt_Tiny`
- `TimesFM`


In [ ]:
if RUN_TRUSTWORTHINESS:
    forecast_artifact_path = PROJECT_ROOT / "results" / "validated_forecasts.csv"
    forecast_results = pd.read_csv(forecast_artifact_path, parse_dates=["Timestamp"])
    
    expected_columns = [
        "Timestamp",
        "Actual",
        "Naive",
        "Persistence_Enhanced_LSTM",
        "Chronos_Bolt_Tiny",
        "TimesFM",
    ]
    
    missing_columns = [column for column in expected_columns if column not in forecast_results.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    
    forecast_results = forecast_results[expected_columns].sort_values("Timestamp").reset_index(drop=True)
    forecast_results["Timestamp"] = pd.to_datetime(forecast_results["Timestamp"], utc=True)
    forecast_results = forecast_results.set_index("Timestamp")
    
    value_columns = ["Actual", "Naive", "Persistence_Enhanced_LSTM", "Chronos_Bolt_Tiny", "TimesFM"]
    
    artifact_validation = pd.DataFrame(
        [
            {"Check": "Shape is (1061, 6) before Timestamp index", "Result": tuple(pd.read_csv(forecast_artifact_path).shape) == (1061, 6)},
            {"Check": "Timestamp unique", "Result": forecast_results.index.is_unique},
            {"Check": "Timestamp sorted", "Result": forecast_results.index.is_monotonic_increasing},
            {"Check": "No missing values", "Result": not forecast_results[value_columns].isna().any().any()},
            {"Check": "All forecast values finite", "Result": bool(np.isfinite(forecast_results[value_columns].to_numpy(dtype=float)).all())},
            {"Check": "Exact test start date", "Result": forecast_results.index.min() == pd.Timestamp("2023-08-12", tz="UTC")},
            {"Check": "Exact test end date", "Result": forecast_results.index.max() == pd.Timestamp("2026-07-07", tz="UTC")},
            {"Check": "Test length is 1061", "Result": len(forecast_results) == 1061},
        ]
    )
    
    model_vector_frame = forecast_results[["Naive", "Persistence_Enhanced_LSTM", "Chronos_Bolt_Tiny", "TimesFM"]]
    duplicate_vector_rows = []
    for left_position, left_name in enumerate(model_vector_frame.columns):
        for right_name in model_vector_frame.columns[left_position + 1:]:
            duplicate_vector_rows.append(
                {
                    "Pair": f"{left_name} vs {right_name}",
                    "Duplicated Vector": bool(model_vector_frame[left_name].equals(model_vector_frame[right_name])),
                }
            )
    duplicate_vector_check = pd.DataFrame(duplicate_vector_rows)
    artifact_validation.loc[len(artifact_validation)] = {
        "Check": "No duplicated forecast vectors",
        "Result": not duplicate_vector_check["Duplicated Vector"].any(),
    }
    
    display(artifact_validation)
    duplicate_vector_check


## 3. Load Dataset And Verify Split

In [ ]:
if RUN_TRUSTWORTHINESS:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    raw_df = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(raw_df)
    target = df_daily["Close"].dropna().astype(float).rename("Close")
    
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = forecast_results["Actual"].rename("Actual")
    
    split_validation = pd.DataFrame(
        [
            {"Check": "Train ends 2023-08-11", "Result": train.index.max() == pd.Timestamp("2023-08-11", tz="UTC")},
            {"Check": "Test begins 2023-08-12", "Result": test.index.min() == pd.Timestamp("2023-08-12", tz="UTC")},
            {"Check": "Test length is 1061", "Result": len(test) == 1061},
            {"Check": "Artifact index equals reconstructed test index", "Result": forecast_results.index.equals(test.index)},
            {"Check": "Artifact Actual equals reconstructed Close", "Result": np.allclose(forecast_results["Actual"], test.to_numpy(dtype=float))},
        ]
    )
    split_validation


## 4. Forecast Protocol Comparability

The main Trust Score ranking includes only rolling one-step models with exact saved vectors or deterministic no-leakage baselines. ARIMA and SARIMA are retained as separate protocol notes unless exact rolling one-step saved vectors exist.


In [ ]:
if RUN_TRUSTWORTHINESS:
    available_result_files = {path.name for path in (PROJECT_ROOT / "results").glob("*")}
    original_lstm_vector_exists = "original_lstm_forecast.csv" in available_result_files or "Original_LSTM" in forecast_results.columns
    arima_rolling_vector_exists = "arima_rolling_forecast.csv" in available_result_files or "ARIMA_Rolling" in forecast_results.columns
    sarima_rolling_vector_exists = "sarima_rolling_forecast.csv" in available_result_files or "SARIMA_Rolling" in forecast_results.columns
    
    protocol_comparability = pd.DataFrame(
        [
            {"Model": "Naive", "Model Type": "Classical baseline", "Protocol": "Rolling one-step", "Saved Vector": True, "Uses newly observed test actuals": True, "Eligible for main Trust Score": "Yes"},
            {"Model": "7-Day Moving Average", "Model Type": "Classical baseline", "Protocol": "Deterministic rolling one-step", "Saved Vector": False, "Uses newly observed test actuals": True, "Eligible for main Trust Score": "Yes"},
            {"Model": "Persistence-Enhanced LSTM", "Model Type": "Supervised deep learning", "Protocol": "Rolling one-step", "Saved Vector": True, "Uses newly observed test actuals": True, "Eligible for main Trust Score": "Yes"},
            {"Model": "TimesFM", "Model Type": "Zero-shot time-series foundation model", "Protocol": "Rolling one-step", "Saved Vector": True, "Uses newly observed test actuals": True, "Eligible for main Trust Score": "Yes"},
            {"Model": "Chronos-Bolt-Tiny", "Model Type": "Zero-shot time-series foundation model", "Protocol": "Rolling one-step", "Saved Vector": True, "Uses newly observed test actuals": True, "Eligible for main Trust Score": "Yes"},
            {"Model": "Original LSTM", "Model Type": "Supervised deep learning", "Protocol": "Exploratory historical workflow", "Saved Vector": original_lstm_vector_exists, "Uses newly observed test actuals": np.nan, "Eligible for main Trust Score": "Yes" if original_lstm_vector_exists else "No"},
            {"Model": "ARIMA(1,1,1)", "Model Type": "Statistical model", "Protocol": "Static multi-step unless saved rolling vector exists", "Saved Vector": arima_rolling_vector_exists, "Uses newly observed test actuals": False, "Eligible for main Trust Score": "Yes" if arima_rolling_vector_exists else "No"},
            {"Model": "SARIMA", "Model Type": "Statistical model", "Protocol": "Static multi-step unless saved rolling vector exists", "Saved Vector": sarima_rolling_vector_exists, "Uses newly observed test actuals": False, "Eligible for main Trust Score": "Yes" if sarima_rolling_vector_exists else "No"},
        ]
    ).set_index("Model")
    protocol_comparability


## 5. Accuracy Evaluation

In [ ]:
if RUN_TRUSTWORTHINESS:
    forecast_series = {
        "Naive": forecast_results["Naive"].rename("Naive"),
        "Persistence-Enhanced LSTM": forecast_results["Persistence_Enhanced_LSTM"].rename("Persistence-Enhanced LSTM"),
        "TimesFM": forecast_results["TimesFM"].rename("TimesFM"),
        "Chronos-Bolt-Tiny": forecast_results["Chronos_Bolt_Tiny"].rename("Chronos-Bolt-Tiny"),
    }
    
    # Deterministic no-leakage 7-day moving average: prediction at t uses t-7 through t-1 only.
    moving_average_7 = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("7-Day Moving Average")
    assert moving_average_7.index.equals(y_test.index)
    assert not moving_average_7.isna().any()
    
    ma7_audit = pd.DataFrame(
        {
            "forecast_date": test.index[:10],
            "window_start": [target.loc[:date].iloc[:-1].tail(7).index.min() for date in test.index[:10]],
            "window_end": [target.loc[:date].iloc[:-1].tail(7).index.max() for date in test.index[:10]],
            "forecast": moving_average_7.iloc[:10].to_numpy(dtype=float),
            "actual": y_test.iloc[:10].to_numpy(dtype=float),
        }
    )
    assert (pd.to_datetime(ma7_audit["window_end"]) < pd.to_datetime(ma7_audit["forecast_date"])).all()
    forecast_series["7-Day Moving Average"] = moving_average_7
    
    if original_lstm_vector_exists:
        if "Original_LSTM" in forecast_results.columns:
            forecast_series["Original LSTM"] = forecast_results["Original_LSTM"].rename("Original LSTM")
        else:
            original_lstm_path = PROJECT_ROOT / "results" / "original_lstm_forecast.csv"
            original_lstm_df = pd.read_csv(original_lstm_path, parse_dates=["Timestamp"]).set_index("Timestamp")
            if original_lstm_df.index.tz is None:
                original_lstm_df.index = original_lstm_df.index.tz_localize("UTC")
            forecast_series["Original LSTM"] = original_lstm_df.iloc[:, 0].reindex(y_test.index).rename("Original LSTM")
    
    metrics_rows = []
    for model_name, forecast in forecast_series.items():
        metrics_rows.append(
            {
                "Model": model_name,
                "MAE": mae(y_test, forecast),
                "RMSE": rmse(y_test, forecast),
                "MAPE": mape(y_test, forecast),
                "sMAPE": smape(y_test, forecast),
                "N": len(pd.concat([y_test, forecast], axis=1).dropna()),
            }
        )
    
    metrics_table = pd.DataFrame(metrics_rows).set_index("Model")
    metrics_table["Relative MAE"] = metrics_table["MAE"] / metrics_table.loc["Naive", "MAE"]
    metrics_table.sort_values("RMSE")


## 6. Robustness Evaluation

In [ ]:
if RUN_TRUSTWORTHINESS:
    returns = y_test.pct_change()
    rolling_volatility = returns.rolling(window=14, min_periods=7).std()
    low_threshold = rolling_volatility.quantile(0.33)
    high_threshold = rolling_volatility.quantile(0.67)
    up_threshold = returns.quantile(0.80)
    down_threshold = returns.quantile(0.20)
    
    regime_masks = {
        "Low Volatility": rolling_volatility <= low_threshold,
        "High Volatility": rolling_volatility >= high_threshold,
        "Major Upward Movement": returns >= up_threshold,
        "Major Downward Movement": returns <= down_threshold,
    }
    
    regime_rows = []
    for regime_name, mask in regime_masks.items():
        regime_index = mask[mask].index
        for model_name, forecast in forecast_series.items():
            regime_rows.append(
                {
                    "Regime": regime_name,
                    "Model": model_name,
                    "MAE": mae(y_test.reindex(regime_index), forecast.reindex(regime_index)),
                    "RMSE": rmse(y_test.reindex(regime_index), forecast.reindex(regime_index)),
                    "MAPE": mape(y_test.reindex(regime_index), forecast.reindex(regime_index)),
                    "sMAPE": smape(y_test.reindex(regime_index), forecast.reindex(regime_index)),
                    "N": len(regime_index),
                }
            )
    
    regime_performance = pd.DataFrame(regime_rows).set_index(["Regime", "Model"]).sort_index()
    regime_performance


## 7. Generalisation Evaluation

In [ ]:
if RUN_TRUSTWORTHINESS:
    segment_rows = []
    for segment_name, segment_index in zip(
        ["Earlier Test Period", "Middle Test Period", "Later Test Period"],
        np.array_split(y_test.index, 3),
    ):
        for model_name, forecast in forecast_series.items():
            segment_rows.append(
                {
                    "Segment": segment_name,
                    "Model": model_name,
                    "MAE": mae(y_test.reindex(segment_index), forecast.reindex(segment_index)),
                    "RMSE": rmse(y_test.reindex(segment_index), forecast.reindex(segment_index)),
                    "MAPE": mape(y_test.reindex(segment_index), forecast.reindex(segment_index)),
                    "sMAPE": smape(y_test.reindex(segment_index), forecast.reindex(segment_index)),
                    "N": len(segment_index),
                }
            )
    
    segment_generalisation = pd.DataFrame(segment_rows).set_index(["Segment", "Model"]).sort_index()
    segment_generalisation


## 8. Uncertainty Evaluation

Foundation-model uncertainty uses validated native interval results from Notebook 05:

- Chronos-Bolt-Tiny: 80% coverage `0.845429`, average 80% width `5151.959961`, 95% unavailable.
- TimesFM: 80% coverage `0.330820`, average 80% width `1436.627452`, 95% unavailable.

TimesFM intervals are narrow but severely under-cover. This materially lowers its Uncertainty Score.

For deterministic baselines without native probabilistic forecasts, validation-period residuals are used only when they can be recreated without model fitting. Persistence-Enhanced LSTM has no saved validation uncertainty artifact, so it receives an explicit uncertainty penalty rather than invented intervals.


In [ ]:
if RUN_TRUSTWORTHINESS:
    validation_window = train.iloc[-len(test):]
    validation_actual = validation_window.rename("actual")
    validation_index = validation_actual.index
    
    validation_forecasts = {
        "Naive": target.shift(1).reindex(validation_index).rename("Naive"),
        "7-Day Moving Average": target.shift(1).rolling(window=7).mean().reindex(validation_index).rename("7-Day Moving Average"),
    }
    
    uncertainty_rows = []
    for model_name, validation_forecast in validation_forecasts.items():
        aligned_validation = pd.concat([validation_actual, validation_forecast.rename("forecast")], axis=1).dropna()
        residual_abs = (aligned_validation["actual"] - aligned_validation["forecast"]).abs()
        q80 = residual_abs.quantile(0.80)
        q95 = residual_abs.quantile(0.95)
        test_error_abs = (y_test - forecast_series[model_name]).abs()
        uncertainty_rows.append(
            {
                "Model": model_name,
                "80% Coverage": (test_error_abs <= q80).mean(),
                "Average 80% Width": 2 * q80,
                "95% Coverage": (test_error_abs <= q95).mean(),
                "Average 95% Width": 2 * q95,
                "Interval Type": "Validation residual-based empirical interval",
                "Note": "No test residuals used for calibration.",
            }
        )
    
    uncertainty_rows.extend(
        [
            {
                "Model": "Persistence-Enhanced LSTM",
                "80% Coverage": np.nan,
                "Average 80% Width": np.nan,
                "95% Coverage": np.nan,
                "Average 95% Width": np.nan,
                "Interval Type": "Unavailable",
                "Note": "No saved validation uncertainty artifact; penalised rather than invented.",
            },
            {
                "Model": "Chronos-Bolt-Tiny",
                "80% Coverage": 0.845429,
                "Average 80% Width": 5151.959961,
                "95% Coverage": np.nan,
                "Average 95% Width": np.nan,
                "Interval Type": "Native foundation-model quantile interval",
                "Note": "95% unavailable; unsupported quantiles not extrapolated.",
            },
            {
                "Model": "TimesFM",
                "80% Coverage": 0.330820,
                "Average 80% Width": 1436.627452,
                "95% Coverage": np.nan,
                "Average 95% Width": np.nan,
                "Interval Type": "Native foundation-model quantile interval",
                "Note": "95% unavailable; verified quantile range only 0.1 through 0.9. Narrow intervals severely under-cover.",
            },
        ]
    )
    
    uncertainty_table = pd.DataFrame(uncertainty_rows).set_index("Model").reindex(metrics_table.index)
    uncertainty_table


## 9. Explainability Evaluation

In [ ]:
if RUN_TRUSTWORTHINESS:
    explainability_table = pd.DataFrame(
        {
            "Naive": {
                "Model Transparency": 100,
                "Ease of Interpretation": 100,
                "Computational Complexity": 100,
                "Reproducibility": 100,
                "Failure Detectability": 95,
            },
            "7-Day Moving Average": {
                "Model Transparency": 95,
                "Ease of Interpretation": 95,
                "Computational Complexity": 100,
                "Reproducibility": 100,
                "Failure Detectability": 90,
            },
            "Persistence-Enhanced LSTM": {
                "Model Transparency": 45,
                "Ease of Interpretation": 50,
                "Computational Complexity": 45,
                "Reproducibility": 75,
                "Failure Detectability": 70,
            },
            "Chronos-Bolt-Tiny": {
                "Model Transparency": 35,
                "Ease of Interpretation": 45,
                "Computational Complexity": 90,
                "Reproducibility": 90,
                "Failure Detectability": 85,
            },
            "TimesFM": {
                "Model Transparency": 30,
                "Ease of Interpretation": 40,
                "Computational Complexity": 75,
                "Reproducibility": 90,
                "Failure Detectability": 80,
            },
        }
    ).T.reindex(metrics_table.index)
    explainability_table["Explainability Score"] = explainability_table.mean(axis=1)
    explainability_table


## 10. Trust Score Framework

Weights:
- Relative Accuracy Score: 35%
- Relative Robustness Score: 20%
- Relative Generalisation Score: 20%
- Uncertainty Score: 15%
- Explainability Score: 10%

A score of 100 is relative to the best model in the comparison set and does not represent perfect forecast accuracy.

This notebook reports two rankings:

1. **Overall Trust Score - Missing Evidence Penalised** keeps the complete-dimension deployment-readiness score. Models without uncertainty evidence receive a penalty.
2. **Evidence-Available Trust Score** uses only dimensions available for each model and renormalises the available weights to sum to 1.

A missing uncertainty artifact is not evidence of poor calibration. The penalised ranking measures deployment readiness, while the evidence-available ranking measures performance on evaluated dimensions.


In [ ]:
if RUN_TRUSTWORTHINESS:
    weights = {
        "Relative Accuracy Score": 0.35,
        "Relative Robustness Score": 0.20,
        "Relative Generalisation Score": 0.20,
        "Uncertainty Score": 0.15,
        "Explainability Score": 0.10,
    }
    
    
    def inverse_score(values):
        values = pd.Series(values, dtype=float)
        best = values.min()
        return (100 * best / values).clip(0, 100)
    
    
    relative_accuracy_score = inverse_score(metrics_table["RMSE"])
    
    robustness_rmse = regime_performance["RMSE"].unstack("Regime")
    robustness_penalty = robustness_rmse.mean(axis=1) + robustness_rmse.std(axis=1).fillna(0)
    relative_robustness_score = inverse_score(robustness_penalty)
    
    generalisation_rmse = segment_generalisation["RMSE"].unstack("Segment")
    generalisation_penalty = generalisation_rmse.mean(axis=1) + generalisation_rmse.std(axis=1).fillna(0)
    relative_generalisation_score = inverse_score(generalisation_penalty)
    
    coverage_component = (100 - (uncertainty_table["80% Coverage"] - 0.80).abs() * 100).clip(0, 100)
    width_component = inverse_score(uncertainty_table["Average 80% Width"].dropna())
    uncertainty_score_observed = (0.90 * coverage_component + 0.10 * width_component.reindex(coverage_component.index)).clip(0, 100)
    
    uncertainty_available = uncertainty_table["80% Coverage"].notna()
    uncertainty_score_penalised = uncertainty_score_observed.where(uncertainty_available, 0).fillna(0)
    
    explainability_score = explainability_table["Explainability Score"]
    
    component_scores = pd.DataFrame(
        {
            "Relative Accuracy Score": relative_accuracy_score,
            "Relative Robustness Score": relative_robustness_score,
            "Relative Generalisation Score": relative_generalisation_score,
            "Uncertainty Score": uncertainty_score_observed,
            "Explainability Score": explainability_score,
        }
    )
    
    dimension_availability = component_scores.notna()
    dimension_availability["Uncertainty Score"] = uncertainty_available.reindex(component_scores.index).fillna(False)
    dimension_availability = dimension_availability.rename(columns=lambda column: column.replace(" Score", " Available"))
    
    penalised_component_scores = component_scores.copy()
    penalised_component_scores["Uncertainty Score"] = uncertainty_score_penalised
    penalised_component_scores = penalised_component_scores.dropna()
    penalised_component_scores["Overall Trust Score - Missing Evidence Penalised"] = sum(
        penalised_component_scores[column] * weight for column, weight in weights.items()
    )
    
    evidence_available_rows = []
    for model_name, row in component_scores.iterrows():
        available_dimensions = [
            dimension
            for dimension in weights
            if pd.notna(row.get(dimension)) and bool(dimension_availability.loc[model_name, dimension.replace(" Score", " Available")])
        ]
        available_weight_sum = sum(weights[dimension] for dimension in available_dimensions)
        if available_weight_sum == 0:
            continue
        score = sum(row[dimension] * weights[dimension] for dimension in available_dimensions) / available_weight_sum
        evidence_available_rows.append(
            {
                "Model": model_name,
                "Evidence-Available Trust Score": score,
                "Available Dimensions": ", ".join(available_dimensions),
                "Unavailable Dimensions": ", ".join([dimension for dimension in weights if dimension not in available_dimensions]) or "None",
                "Renormalised Weight Sum": 1.0,
            }
        )
    
    evidence_available_trust_scores = pd.DataFrame(evidence_available_rows).set_index("Model").sort_values(
        "Evidence-Available Trust Score",
        ascending=False,
    )
    
    trust_scores = penalised_component_scores.sort_values("Overall Trust Score - Missing Evidence Penalised", ascending=False)
    trust_scores


In [ ]:
if RUN_TRUSTWORTHINESS:
    evidence_available_trust_scores


## 11. Trust Score Validation

In [ ]:
if RUN_TRUSTWORTHINESS:
    component_columns = list(weights)
    penalised_weighted_sum = sum(trust_scores[column] * weight for column, weight in weights.items())
    
    trust_score_validation = pd.DataFrame(
        [
            {
                "Check": "All penalised component scores between 0 and 100",
                "Result": bool(((trust_scores[component_columns] >= 0) & (trust_scores[component_columns] <= 100)).all().all()),
            },
            {
                "Check": "No NaN penalised Trust Scores",
                "Result": bool(not trust_scores[component_columns + ["Overall Trust Score - Missing Evidence Penalised"]].isna().any().any()),
            },
            {
                "Check": "Evidence-available scores between 0 and 100",
                "Result": bool(evidence_available_trust_scores["Evidence-Available Trust Score"].between(0, 100).all()),
            },
            {"Check": "Weights sum to 1.0", "Result": bool(np.isclose(sum(weights.values()), 1.0))},
            {
                "Check": "Penalised score equals documented weighted formula",
                "Result": bool(np.allclose(trust_scores["Overall Trust Score - Missing Evidence Penalised"], penalised_weighted_sum)),
            },
            {
                "Check": "Evidence-available weights renormalise to 1.0",
                "Result": bool(np.isclose(evidence_available_trust_scores["Renormalised Weight Sum"], 1.0).all()),
            },
            {"Check": "No model fitting executed", "Result": not MODEL_FITTING_EXECUTED},
            {"Check": "No model checkpoint loaded", "Result": not MODEL_CHECKPOINT_LOADED},
            {"Check": "No forecast vector regenerated", "Result": not FORECAST_VECTOR_REGENERATED},
        ]
    )
    trust_score_validation


## 12. Foundation Model Trade-Offs

In [ ]:
if RUN_TRUSTWORTHINESS:
    foundation_model_comparison = pd.DataFrame(
        [
            {
                "Model": "TimesFM",
                "Point Accuracy": "Strongest zero-shot point forecaster; RMSE 1924.199337",
                "Robustness": "Better than Chronos across all reported regimes",
                "Generalisation": "Good, but weaker in the middle test segment",
                "Uncertainty Calibration": "Poor: 80% coverage 0.330820, severe undercoverage",
                "Inference Cost": "Higher: 31.467 seconds total, 0.029658 seconds/forecast",
                "Zero-Shot Status": "Zero-shot foundation model",
            },
            {
                "Model": "Chronos-Bolt-Tiny",
                "Point Accuracy": "Weaker than TimesFM; RMSE 1994.007926",
                "Robustness": "Weaker during volatility and directional stress regimes",
                "Generalisation": "Competitive but weaker point accuracy",
                "Uncertainty Calibration": "Better: 80% coverage 0.845429",
                "Inference Cost": "Lower than TimesFM in this experiment",
                "Zero-Shot Status": "Zero-shot foundation model",
            },
        ]
    ).set_index("Model")
    foundation_model_comparison


## 13. Statistical Significance Interpretation

From Notebook 09:
- Naive significantly outperforms TimesFM.
- TimesFM significantly outperforms Chronos-Bolt-Tiny.
- Persistence-Enhanced LSTM and TimesFM are not significantly different at alpha = 0.05.
- Naive significantly outperforms Persistence-Enhanced LSTM.


In [ ]:
if RUN_TRUSTWORTHINESS:
    statistical_significance_summary = pd.DataFrame(
        [
            {"Comparison": "Naive vs TimesFM", "Finding": "Naive significantly outperforms TimesFM", "p-value": 0.000021},
            {"Comparison": "Chronos-Bolt-Tiny vs TimesFM", "Finding": "TimesFM significantly outperforms Chronos-Bolt-Tiny", "p-value": 0.005862},
            {"Comparison": "Persistence-Enhanced LSTM vs TimesFM", "Finding": "No significant difference at alpha = 0.05", "p-value": 0.123495},
            {"Comparison": "Naive vs Persistence-Enhanced LSTM", "Finding": "Naive significantly outperforms Persistence-Enhanced LSTM", "p-value": 0.019199},
        ]
    ).set_index("Comparison")
    statistical_significance_summary


## 14. Visual Comparison

In [ ]:
if RUN_TRUSTWORTHINESS:
    fig, ax = plt.subplots(figsize=(10, 5))
    trust_scores["Overall Trust Score - Missing Evidence Penalised"].sort_values().plot(kind="barh", ax=ax)
    ax.set_title("Trust Score Ranking - Missing Evidence Penalised")
    ax.set_xlabel("Overall Trust Score")
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    evidence_available_trust_scores["Evidence-Available Trust Score"].sort_values().plot(kind="barh", ax=ax)
    ax.set_title("Evidence-Available Trust Score Ranking")
    ax.set_xlabel("Evidence-Available Trust Score")
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    for model_name, forecast in forecast_series.items():
        forecast.plot(ax=ax, label=model_name, alpha=0.75)
    ax.set_title("Artifact-Only Rolling One-Step Forecast Comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## 15. Final Model Comparison

In [ ]:
if RUN_TRUSTWORTHINESS:
    final_model_comparison = (
        metrics_table
        .join(trust_scores[["Overall Trust Score - Missing Evidence Penalised"]], how="left")
        .join(evidence_available_trust_scores[["Evidence-Available Trust Score", "Unavailable Dimensions"]], how="left")
        .sort_values("Overall Trust Score - Missing Evidence Penalised", ascending=False)
    )
    final_model_comparison


## 16. Authoritative Findings

- Naive has the best overall point accuracy.
- Persistence-Enhanced LSTM is the strongest supervised neural model with an exact saved forecast vector.
- Original LSTM has no exact saved validated forecast vector in `results/`, so it is excluded from the quantitative Trust Score ranking and retained only as an exploratory failure case study.
- TimesFM is the strongest zero-shot foundation model for point forecasts.
- Chronos provides better 80% uncertainty calibration.
- TimesFM has severe uncertainty undercoverage: its nominal 80% interval covers only `0.330820` of actual values.
- Model complexity does not guarantee trustworthiness.
- Different models dominate different trustworthiness dimensions.


## Appendix: Historical Model-Generation Workflow â€” Not Executed

Earlier versions of this notebook included model-fitting and model-inference code for ARIMA/SARIMA, raw-price LSTM, Persistence-Enhanced LSTM, Chronos, and later foundation-model comparisons. That code has been removed from the active execution path.

The historical methodology remains documented in the project notebooks that generated or audited the forecasts:
- `notebooks/05_Foundation_Models.ipynb`
- `notebooks/07_Model_Validation_Audit.ipynb`
- `notebooks/08_Naive_Forecast_Audit.ipynb`
- `notebooks/09_Statistical_Significance_Test.ipynb`

This notebook now consumes only validated forecast artifacts and deterministic baselines. It does not retrain, refit, load checkpoints, or regenerate model forecast vectors.


## Part G1 — Complete model validation audit

**Source notebook:** [07_Model_Validation_Audit.ipynb](07_Model_Validation_Audit.ipynb)

Training-based audit code is retained but disabled by default to protect artifacts.

The source is provenance only; executable Markdown and Python are merged below.

# Model Validation and Forecasting Protocol Audit

This notebook audits whether the earlier forecasting results are valid before they are used in any trustworthiness ranking. The central question is whether Naive persistence genuinely outperforms the neural models, or whether the apparent underperformance is caused by an implementation, alignment, scaling, target, or forecasting-protocol error.

No model should be promoted into the trustworthiness ranking until this audit passes.

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    from pathlib import Path
    import sys
    import warnings
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Add, Dense, Dropout, Embedding, GlobalAveragePooling1D, Input, Layer, LayerNormalization, MultiHeadAttention
    from tensorflow.keras.models import Model, Sequential
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    warnings.filterwarnings("ignore")
    tf.random.set_seed(42)
    np.random.seed(42)
    pd.set_option("display.float_format", "{:.6f}".format)


## 1. Dataset and Split Verification

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    raw_df = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(raw_df)
    target = df_daily["Close"].asfreq("D").dropna().rename("Close")
    
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    train_test_overlap = train.index.intersection(test.index)
    
    split_report = pd.DataFrame(
        {
            "Item": [
                "Complete series date range",
                "Training date range",
                "Test date range",
                "Training length",
                "Test length",
                "Train/test overlap",
            ],
            "Value": [
                f"{target.index.min()} to {target.index.max()}",
                f"{train.index.min()} to {train.index.max()}",
                f"{test.index.min()} to {test.index.max()}",
                len(train),
                len(test),
                len(train_test_overlap) > 0,
            ],
        }
    )
    split_report


## 2. Forecast Horizon Definition

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    FORECAST_HORIZON = 1
    LOOKBACK_LSTM = 30
    LOOKBACK_IMPROVED_LSTM = 60
    LOOKBACK_TRANSFORMER = 30
    
    protocol_definition = pd.DataFrame(
        {
            "Protocol": ["A", "B"],
            "Name": ["Rolling one-step-ahead", "Recursive multi-step"],
            "Information available at each test date": [
                "All observed actual values up to the previous day are available.",
                "Only the final training window is available; future inputs are model predictions.",
            ],
            "Use of actual test values as future inputs": ["Allowed after each date is observed", "Not allowed"],
        }
    )
    protocol_definition


## 3. Sequence and Target Alignment

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    def create_sequences(values, index, lookback):
        X, y, rows = [], [], []
        for i in range(lookback, len(values)):
            X.append(values[i - lookback : i])
            y.append(values[i])
            rows.append(
                {
                    "input_window_start": index[i - lookback],
                    "input_window_end": index[i - 1],
                    "target_date": index[i],
                    "last_input_value": float(values[i - 1, 0]),
                    "target_value": float(values[i, 0]),
                }
            )
        return np.array(X), np.array(y), pd.DataFrame(rows)
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_values = train.to_numpy().reshape(-1, 1)
    test_values = test.to_numpy().reshape(-1, 1)
    train_scaled = scaler.fit_transform(train_values)
    test_scaled = scaler.transform(test_values)
    
    X_train_lstm, y_train_lstm, train_sequence_map = create_sequences(train_scaled, train.index, LOOKBACK_LSTM)
    combined_test_scaled = np.vstack([train_scaled[-LOOKBACK_LSTM:], test_scaled])
    combined_test_index = train.index[-LOOKBACK_LSTM:].append(test.index)
    X_test_lstm, y_test_lstm_scaled, test_sequence_map = create_sequences(combined_test_scaled, combined_test_index, LOOKBACK_LSTM)
    
    test_sequence_map["sequence_ends_then_predicts_next_day"] = test_sequence_map["target_date"] == test_sequence_map["input_window_end"] + pd.Timedelta(days=1)
    test_sequence_map.head(10)


In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    def alignment_checks(name, forecast, y_true):
        return {
            "Model": name,
            "Forecast length equals y_test length": len(forecast) == len(y_true),
            "Forecast index exactly equals y_test index": forecast.index.equals(y_true.index),
            "No missing values": not forecast.isna().any(),
            "No duplicated timestamps": not forecast.index.duplicated().any(),
            "No shifted-target symptom vs y[t+1]": not forecast.corr(y_true.shift(-1)) > forecast.corr(y_true),
        }
    
    # Populated after forecasts are recreated below.
    alignment_audit_rows = []


## 4. Scaling and Inverse-Scaling Audit

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    roundtrip_train = scaler.inverse_transform(scaler.transform(train_values)).ravel()
    roundtrip_test = scaler.inverse_transform(scaler.transform(test_values)).ravel()
    
    scaling_audit = pd.DataFrame(
        {
            "Check": [
                "Scaler fitted only on training data",
                "scaler.data_min_",
                "scaler.data_max_",
                "Train scaled min/max",
                "Test scaled min/max",
                "Train inverse-transform roundtrip OK",
                "Test inverse-transform roundtrip OK",
                "Test values exceed training scaling range",
            ],
            "Result": [
                True,
                scaler.data_min_[0],
                scaler.data_max_[0],
                f"{train_scaled.min():.6f} to {train_scaled.max():.6f}",
                f"{test_scaled.min():.6f} to {test_scaled.max():.6f}",
                np.allclose(roundtrip_train, train_values.ravel()),
                np.allclose(roundtrip_test, test_values.ravel()),
                bool((test_scaled < 0).any() or (test_scaled > 1).any()),
            ],
        }
    )
    scaling_audit


## 5. Baseline Audit

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
    constant_last_train_forecast = pd.Series(train.iloc[-1], index=test.index, name="Constant Last Train Value")
    moving_average_forecast = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("7-Day Moving Average")
    
    naive_expected = target.shift(1).reindex(test.index)
    ma_expected = target.shift(1).rolling(window=7).mean().reindex(test.index)
    
    baseline_audit = pd.DataFrame(
        {
            "Check": [
                "Naive equals actual[t-1]",
                "Naive differs from constant last-training-value forecast",
                "Moving average uses preceding seven observations only",
                "Moving average does not include target day",
            ],
            "Result": [
                naive_forecast.equals(naive_expected.rename("Naive")),
                not naive_forecast.equals(constant_last_train_forecast.rename("Naive")),
                moving_average_forecast.equals(ma_expected.rename("7-Day Moving Average")),
                moving_average_forecast.iloc[1:].ne(target.reindex(test.index).iloc[1:]).any(),
            ],
        }
    )
    baseline_audit


## 6. LSTM Audit

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    def build_original_lstm(lookback=30):
        model = Sequential([Input(shape=(lookback, 1)), tf.keras.layers.LSTM(32), Dense(1)])
        model.compile(optimizer="adam", loss="mse")
        return model
    
    lstm_model = build_original_lstm(LOOKBACK_LSTM)
    lstm_callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    lstm_history = lstm_model.fit(
        X_train_lstm,
        y_train_lstm,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[lstm_callback],
        shuffle=False,
        verbose=0,
    )
    lstm_predictions_scaled = lstm_model.predict(X_test_lstm, verbose=0)
    lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).ravel()
    lstm_forecast = pd.Series(lstm_predictions, index=test.index, name="Original LSTM")
    
    lstm_audit_table = pd.DataFrame(
        {
            "Check": [
                "Model input shape",
                "Model output shape",
                "X_train shape",
                "y_train shape",
                "X_test shape",
                "y_test shape",
                "Output layer activation is linear",
                "y_train is next value after each input window",
                "Final training loss",
                "Final validation loss",
            ],
            "Result": [
                str(lstm_model.input_shape),
                str(lstm_model.output_shape),
                str(X_train_lstm.shape),
                str(y_train_lstm.shape),
                str(X_test_lstm.shape),
                str(y_test_lstm_scaled.shape),
                lstm_model.layers[-1].activation.__name__ == "linear",
                bool((train_sequence_map["target_date"] == train_sequence_map["input_window_end"] + pd.Timedelta(days=1)).all()),
                lstm_history.history["loss"][-1],
                lstm_history.history["val_loss"][-1],
            ],
        }
    )
    lstm_model.summary()
    lstm_audit_table


In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    def prediction_diagnostics(name, forecast, actual):
        aligned = pd.concat([actual.rename("actual"), forecast.rename("prediction")], axis=1).dropna()
        return pd.Series(
            {
                "prediction_min": aligned["prediction"].min(),
                "prediction_max": aligned["prediction"].max(),
                "prediction_mean": aligned["prediction"].mean(),
                "prediction_std": aligned["prediction"].std(),
                "actual_min": aligned["actual"].min(),
                "actual_max": aligned["actual"].max(),
                "actual_mean": aligned["actual"].mean(),
                "actual_std": aligned["actual"].std(),
                "prediction_daily_change_std": aligned["prediction"].diff().std(),
                "actual_daily_change_std": aligned["actual"].diff().std(),
                "correlation_with_actual": aligned["prediction"].corr(aligned["actual"]),
                "correlation_with_previous_day_actual": aligned["prediction"].corr(target.shift(1).reindex(aligned.index)),
                "percentage_unique_predictions": 100 * aligned["prediction"].nunique() / len(aligned),
            },
            name=name,
        )
    
    lstm_first_20 = pd.concat([y_test.rename("Actual"), lstm_forecast.rename("Original LSTM")], axis=1).head(20)
    lstm_diagnostics = prediction_diagnostics("Original LSTM", lstm_forecast, y_test)
    
    display(lstm_first_20)
    lstm_diagnostics.to_frame().T


## 7. Transformer Audit

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    class PositionalEmbedding(Layer):
        def __init__(self, lookback, d_model, **kwargs):
            super().__init__(**kwargs)
            self.lookback = lookback
            self.d_model = d_model
            self.embedding = Embedding(input_dim=lookback, output_dim=d_model)
    
        def call(self, inputs):
            positions = tf.range(start=0, limit=self.lookback, delta=1)
            position_embedding = self.embedding(positions)
            return inputs + tf.expand_dims(position_embedding, axis=0)
    
        def get_config(self):
            config = super().get_config()
            config.update({"lookback": self.lookback, "d_model": self.d_model})
            return config
    
    
    def build_corrected_transformer(lookback=30, d_model=64, num_heads=4, ff_dim=128, dropout=0.1, use_positional_embedding=True):
        inputs = Input(shape=(lookback, 1), name="price_window")
        x = Dense(d_model, name="input_projection")(inputs)
    
        if use_positional_embedding:
            x = PositionalEmbedding(lookback=lookback, d_model=d_model, name="positional_embedding")(x)
    
        attention_output = MultiHeadAttention(key_dim=d_model // num_heads, num_heads=num_heads, dropout=dropout, name="self_attention")(x, x)
        attention_output = Dropout(dropout)(attention_output)
        x = LayerNormalization(epsilon=1e-6, name="attention_norm")(x + attention_output)
    
        feed_forward = Dense(ff_dim, activation="relu", name="ffn_expand")(x)
        feed_forward = Dropout(dropout)(feed_forward)
        feed_forward = Dense(d_model, name="ffn_project")(feed_forward)
        x = LayerNormalization(epsilon=1e-6, name="ffn_norm")(x + feed_forward)
    
        x = GlobalAveragePooling1D(name="temporal_pooling")(x)
        x = Dropout(dropout)(x)
        x = Dense(32, activation="relu", name="prediction_hidden")(x)
        outputs = Dense(1, name="price_prediction")(x)
        model = Model(inputs, outputs, name="CorrectedTransformerAudit")
        model.compile(optimizer="adam", loss="mse")
        return model
    
    X_train_tr, y_train_tr, train_transformer_map = create_sequences(train_scaled, train.index, LOOKBACK_TRANSFORMER)
    combined_transformer_scaled = np.vstack([train_scaled[-LOOKBACK_TRANSFORMER:], test_scaled])
    combined_transformer_index = train.index[-LOOKBACK_TRANSFORMER:].append(test.index)
    X_test_tr, y_test_tr_scaled, test_transformer_map = create_sequences(combined_transformer_scaled, combined_transformer_index, LOOKBACK_TRANSFORMER)
    
    transformer_model = build_corrected_transformer(LOOKBACK_TRANSFORMER)
    transformer_callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    transformer_history = transformer_model.fit(
        X_train_tr,
        y_train_tr,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[transformer_callback],
        shuffle=False,
        verbose=0,
    )
    transformer_predictions_scaled = transformer_model.predict(X_test_tr, verbose=0)
    transformer_predictions = scaler.inverse_transform(transformer_predictions_scaled).ravel()
    corrected_transformer_forecast = pd.Series(transformer_predictions, index=test.index, name="Corrected Transformer")
    
    layer_output_shapes = {layer.name: str(layer.output.shape) for layer in transformer_model.layers}
    transformer_audit_table = pd.DataFrame(
        {
            "Check": [
                "LayerNormalization operates in D_MODEL space",
                "Explicit positional information is present",
                "Attention output connected before pooling/head",
                "Predictions are not constant",
                "Output layer activation is linear",
                "X_train shape",
                "y_train shape",
                "X_test shape",
                "y_test shape",
                "Final training loss",
                "Final validation loss",
            ],
            "Result": [
                all(layer_output_shapes[name].endswith(', 64)') for name in ["attention_norm", "ffn_norm"]),
                any("positional_embedding" in name for name in layer_output_shapes),
                "self_attention" in layer_output_shapes and "temporal_pooling" in layer_output_shapes,
                corrected_transformer_forecast.nunique() > 1,
                transformer_model.layers[-1].activation.__name__ == "linear",
                str(X_train_tr.shape),
                str(y_train_tr.shape),
                str(X_test_tr.shape),
                str(y_test_tr_scaled.shape),
                transformer_history.history["loss"][-1],
                transformer_history.history["val_loss"][-1],
            ],
        }
    )
    transformer_model.summary()
    transformer_audit_table


In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    transformer_first_20 = pd.concat([y_test.rename("Actual"), corrected_transformer_forecast.rename("Corrected Transformer")], axis=1).head(20)
    transformer_diagnostics = prediction_diagnostics("Corrected Transformer", corrected_transformer_forecast, y_test)
    
    display(transformer_first_20)
    transformer_diagnostics.to_frame().T


## 8. Rolling One-Step Evaluation

Protocol A: Rolling one-step-ahead forecasting. Each model predicts only the next day. The next sample may use the latest observed actual value. All models receive the same information at each test date.

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    protocol_a_forecasts = {
        "Naive": naive_forecast,
        "7-Day Moving Average": moving_average_forecast,
        "Original LSTM": lstm_forecast,
        "Corrected Transformer": corrected_transformer_forecast,
    }
    
    protocol_a_alignment = pd.DataFrame(
        [alignment_checks(name, forecast, y_test) for name, forecast in protocol_a_forecasts.items()]
    ).set_index("Model")
    
    protocol_a_metrics = pd.DataFrame(
        [
            {
                "Model": name,
                "MAE": mae(y_test, forecast),
                "RMSE": rmse(y_test, forecast),
                "MAPE": mape(y_test, forecast),
                "sMAPE": smape(y_test, forecast),
            }
            for name, forecast in protocol_a_forecasts.items()
        ]
    ).set_index("Model").sort_values("RMSE")
    
    display(protocol_a_alignment)
    protocol_a_metrics


## 9. Recursive Multi-Step Evaluation

Protocol B: Recursive multi-step forecasting. Start only from the final training window, predict the first test value, feed predictions back into the model recursively, and do not use actual test values as future inputs.

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    def recursive_lstm_forecast(model, scaler, train_series, forecast_index, lookback, name):
        history_scaled = list(scaler.transform(train_series.to_numpy().reshape(-1, 1)).ravel())
        predictions = []
        for _ in forecast_index:
            window = np.array(history_scaled[-lookback:]).reshape(1, lookback, 1)
            pred_scaled = float(model.predict(window, verbose=0).ravel()[0])
            history_scaled.append(pred_scaled)
            predictions.append(pred_scaled)
        predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).ravel()
        return pd.Series(predictions, index=forecast_index, name=name)
    
    recursive_lstm = recursive_lstm_forecast(lstm_model, scaler, train, test.index, LOOKBACK_LSTM, "Original LSTM Recursive")
    recursive_transformer = recursive_lstm_forecast(transformer_model, scaler, train, test.index, LOOKBACK_TRANSFORMER, "Corrected Transformer Recursive")
    
    protocol_b_forecasts = {
        "Original LSTM Recursive": recursive_lstm,
        "Corrected Transformer Recursive": recursive_transformer,
    }
    
    protocol_b_metrics = pd.DataFrame(
        [
            {
                "Model": name,
                "MAE": mae(y_test, forecast),
                "RMSE": rmse(y_test, forecast),
                "MAPE": mape(y_test, forecast),
                "sMAPE": smape(y_test, forecast),
            }
            for name, forecast in protocol_b_forecasts.items()
        ]
    ).set_index("Model").sort_values("RMSE")
    protocol_b_metrics


## 10. Fair Model Comparison

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    import os
    import random
    import numpy as np
    import tensorflow as tf
    
    SEED = 42
    
    os.environ["PYTHONHASHSEED"] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    
    def build_delta_lstm(lookback=30):
        model = Sequential([Input(shape=(lookback, 1)), tf.keras.layers.LSTM(32), Dense(1)])
        model.compile(optimizer="adam", loss="mse")
        return model
    
    log_returns = np.log(target / target.shift(1)).dropna().rename("log_return")
    return_train = log_returns.reindex(train.index).dropna()
    return_test_index = test.index
    return_scaler = MinMaxScaler(feature_range=(-1, 1))
    return_train_scaled = return_scaler.fit_transform(return_train.to_numpy().reshape(-1, 1))
    
    X_return_train, y_return_train, _ = create_sequences(return_train_scaled, return_train.index, LOOKBACK_LSTM)
    delta_model = build_delta_lstm(LOOKBACK_LSTM)
    delta_history = delta_model.fit(
        X_return_train,
        y_return_train,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
        shuffle=False,
        verbose=0,
    )
    
    known_returns = list(return_scaler.transform(return_train.to_numpy().reshape(-1, 1)).ravel())
    price_history = list(train.to_numpy())
    delta_price_predictions = []
    for date in return_test_index:
        window = np.array(known_returns[-LOOKBACK_LSTM:]).reshape(1, LOOKBACK_LSTM, 1)
        pred_return_scaled = float(delta_model.predict(window, verbose=0).ravel()[0])
        pred_return = float(return_scaler.inverse_transform([[pred_return_scaled]])[0, 0])
        next_price = price_history[-1] * np.exp(pred_return)
        delta_price_predictions.append(next_price)
        actual_return = np.log(target.loc[date] / price_history[-1])
        known_returns.append(float(return_scaler.transform([[actual_return]])[0, 0]))
        price_history.append(target.loc[date])
    
    persistence_enhanced_lstm = pd.Series(delta_price_predictions, index=test.index, name="Persistence-Enhanced LSTM")
    
    fair_comparison_forecasts = {
        **protocol_a_forecasts,
        "Persistence-Enhanced LSTM": persistence_enhanced_lstm,
    }
    fair_comparison_metrics = pd.DataFrame(
        [
            {
                "Model": name,
                "MAE": mae(y_test, forecast),
                "RMSE": rmse(y_test, forecast),
                "MAPE": mape(y_test, forecast),
                "sMAPE": smape(y_test, forecast),
                "Prediction Std": forecast.std(),
                "Actual Std": y_test.std(),
                "Prediction Change Std": forecast.diff().std(),
                "Actual Change Std": y_test.diff().std(),
            }
            for name, forecast in fair_comparison_forecasts.items()
        ]
    ).set_index("Model").sort_values("RMSE")
    fair_comparison_metrics


### Persistence-Enhanced LSTM Forecast Artifact

The exact fixed-seed rolling one-step forecast vector is saved for downstream significance testing.

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    results_dir = PROJECT_ROOT / "results"
    results_dir.mkdir(exist_ok=True)
    
    persistence_artifact = pd.DataFrame(
        {
            "Timestamp": persistence_enhanced_lstm.index,
            "Persistence_Enhanced_LSTM": persistence_enhanced_lstm.to_numpy(dtype=float),
        }
    )
    
    assert len(persistence_artifact) == 1061
    assert pd.DatetimeIndex(persistence_artifact["Timestamp"]).equals(y_test.index)
    assert persistence_artifact["Timestamp"].is_unique
    assert persistence_artifact["Timestamp"].is_monotonic_increasing
    assert not persistence_artifact.isna().any().any()
    assert np.isfinite(persistence_artifact["Persistence_Enhanced_LSTM"].to_numpy(dtype=float)).all()
    
    persistence_artifact_path = results_dir / "persistence_enhanced_lstm_forecast.csv"
    persistence_artifact.to_csv(persistence_artifact_path, index=False)
    
    persistence_artifact_metrics = pd.DataFrame(
        [
            {
                "Model": "Persistence-Enhanced LSTM",
                "MAE": mae(y_test, persistence_enhanced_lstm),
                "RMSE": rmse(y_test, persistence_enhanced_lstm),
                "MAPE": mape(y_test, persistence_enhanced_lstm),
                "sMAPE": smape(y_test, persistence_enhanced_lstm),
            }
        ]
    ).set_index("Model")
    
    print(f"Saved {persistence_artifact_path}")
    print(f"Shape: {persistence_artifact.shape}")
    print("Neural-network forecasts were saved from a fixed-seed deterministic run. Earlier aggregate LSTM metrics came from a different nondeterministic training run and are not used for significance testing.")
    persistence_artifact_metrics


In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    plot_sets = {
        "Full Test Period": y_test.index,
        "First 90 Test Days": y_test.index[:90],
        "Final 90 Test Days": y_test.index[-90:],
    }
    
    for title, idx in plot_sets.items():
        fig, ax = plt.subplots(figsize=(12, 5))
        y_test.reindex(idx).plot(ax=ax, label="Actual", linewidth=2)
        for name, forecast in fair_comparison_forecasts.items():
            forecast.reindex(idx).plot(ax=ax, label=name, alpha=0.8)
        ax.set_title(title)
        ax.set_xlabel("Date")
        ax.set_ylabel("Bitcoin Close")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.diff().plot(ax=ax, label="Actual change", linewidth=2)
    for name, forecast in fair_comparison_forecasts.items():
        forecast.diff().plot(ax=ax, label=f"{name} change", alpha=0.75)
    ax.set_title("Actual vs Predicted Daily Changes")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    y_test.plot(kind="hist", bins=40, alpha=0.4, ax=ax, label="Actual")
    for name, forecast in fair_comparison_forecasts.items():
        forecast.plot(kind="hist", bins=40, alpha=0.25, ax=ax, label=name)
    ax.set_title("Prediction Distribution vs Actual Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 11. Final Diagnosis

In [ ]:
if EXECUTE_VALIDATION_REGENERATION:
    diagnosis = pd.DataFrame(
        {
            "Audit Area": [
                "Implementation validity",
                "Forecasting-protocol validity",
                "Model collapse",
                "Over-smoothing",
                "Range compression",
                "Data leakage",
                "Target misalignment",
                "Scaling extrapolation",
            ],
            "Evidence to Review": [
                "Model summaries, linear output activations, shape checks, alignment checks.",
                "Protocol A and Protocol B metrics are reported separately.",
                "Prediction uniqueness, prediction std, and constant-prediction checks.",
                "Prediction daily-change std versus actual daily-change std.",
                "Prediction min/max/std versus actual min/max/std.",
                "Naive and moving-average checks use only prior actual observations; Protocol B forbids test actual feedback.",
                "Sequence maps confirm input window ending on t predicts t+1.",
                "Scaler is fit on train only; test scaled min/max shows extrapolation beyond training range.",
            ],
            "Pass Criteria": [
                "All checks pass or failures are explicitly documented.",
                "No cross-protocol ranking is presented as a single fair leaderboard.",
                "Predictions are not constant except intentionally constant baseline.",
                "Neural change variance is not near zero relative to actual changes.",
                "Forecast range is not implausibly compressed without explanation.",
                "No forecast uses target-day information.",
                "All target dates are exactly one day after window end for one-step models.",
                "Inverse scaling roundtrip passes and extrapolation is acknowledged.",
            ],
        }
    )
    diagnosis


Final reporting guidance:

- Do not claim that advanced models should automatically beat Naive.
- Treat Naive outperformance as plausible for noisy Bitcoin close prices unless the audit finds leakage, misalignment, or protocol inconsistency.
- Do not use these models in the trustworthiness ranking until this notebook has been run and the diagnosis table is reviewed.
- If neural forecasts remain flat after alignment and scaling checks pass, the likely diagnosis is model/target mismatch or over-smoothing from raw price-level modelling rather than a simple implementation error.

## Part G2 — Independent Naive audit

**Source notebook:** [08_Naive_Forecast_Audit.ipynb](08_Naive_Forecast_Audit.ipynb)

Leakage, alignment, first-principles metrics and invalid counterexamples are preserved.

The source is provenance only; executable Markdown and Python are merged below.

# Rigorous Naive Forecast Audit

This notebook audits the one-step Naive persistence forecast with intentionally redundant checks. The goal is to determine whether the Naive model is valid, misaligned, leaking test information, or being evaluated unfairly.

Do not modify the trustworthiness notebook until this audit has been run and reviewed.

In [ ]:
if RUN_NAIVE_AUDIT:
    from pathlib import Path
    import sys
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    pd.set_option("display.float_format", "{:.6f}".format)


## 1. Load and Validate the Series

In [ ]:
if RUN_NAIVE_AUDIT:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    raw_df = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(raw_df)
    target = df_daily["Close"].asfreq("D").dropna().rename("Close")
    
    expected_daily_index = pd.date_range(target.index.min(), target.index.max(), freq="D", tz=target.index.tz)
    missing_daily_dates = expected_daily_index.difference(target.index)
    
    series_validation = pd.DataFrame(
        [
            {"Check": "Full series length", "Value": len(target)},
            {"Check": "Full series start", "Value": target.index.min()},
            {"Check": "Full series end", "Value": target.index.max()},
            {"Check": "Index is sorted", "Value": target.index.is_monotonic_increasing},
            {"Check": "Index contains duplicates", "Value": target.index.duplicated().any()},
            {"Check": "Daily frequency has gaps", "Value": len(missing_daily_dates) > 0},
            {"Check": "Number of missing daily dates", "Value": len(missing_daily_dates)},
        ]
    )
    series_validation


## 2. Recreate the Train-Test Split

In [ ]:
if RUN_NAIVE_AUDIT:
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.copy()
    
    overlap = train.index.intersection(test.index)
    chronological_split_ok = train.index.max() < test.index.min()
    
    split_validation = pd.DataFrame(
        [
            {"Check": "Full series length", "Value": len(target)},
            {"Check": "Train length", "Value": len(train)},
            {"Check": "Test length", "Value": len(test)},
            {"Check": "Train start date", "Value": train.index.min()},
            {"Check": "Train end date", "Value": train.index.max()},
            {"Check": "Test start date", "Value": test.index.min()},
            {"Check": "Test end date", "Value": test.index.max()},
            {"Check": "Train and test overlap", "Value": len(overlap) > 0},
            {"Check": "Chronological split", "Value": chronological_split_ok},
        ]
    )
    split_validation


## 3. Reconstruct Naive Forecast Independently

In [ ]:
if RUN_NAIVE_AUDIT:
    # Method A: vectorized shift on the complete target series.
    naive_a = target.shift(1).reindex(test.index)
    
    # Method B: explicitly concatenate the final training actual with all previous test actuals.
    previous_actuals = pd.concat([train.iloc[[-1]], test.iloc[:-1]])
    previous_actuals.index = test.index
    naive_b = previous_actuals
    
    # Method C: explicit row-by-row construction.
    naive_values = []
    for i, date in enumerate(test.index):
        if i == 0:
            naive_values.append(train.iloc[-1])
        else:
            naive_values.append(test.iloc[i - 1])
    naive_c = pd.Series(naive_values, index=test.index)
    
    naive_forecast = naive_a.rename("Naive")
    
    independent_reconstruction_checks = pd.DataFrame(
        [
            {"Check": "naive_a equals naive_b", "Result": naive_a.equals(naive_b)},
            {"Check": "naive_a equals naive_c", "Result": naive_a.equals(naive_c)},
            {"Check": "len(naive_a) equals len(test)", "Result": len(naive_a) == len(test)},
            {"Check": "len(naive_b) equals len(test)", "Result": len(naive_b) == len(test)},
            {"Check": "len(naive_c) equals len(test)", "Result": len(naive_c) == len(test)},
            {"Check": "naive_a index equals test.index", "Result": naive_a.index.equals(test.index)},
            {"Check": "naive_b index equals test.index", "Result": naive_b.index.equals(test.index)},
            {"Check": "naive_c index equals test.index", "Result": naive_c.index.equals(test.index)},
            {"Check": "naive_a has no NaN values", "Result": not naive_a.isna().any()},
            {"Check": "naive_a has no duplicated timestamps", "Result": not naive_a.index.duplicated().any()},
        ]
    )
    
    assert naive_a.equals(naive_b)
    assert naive_a.equals(naive_c)
    assert len(naive_a) == len(test)
    assert naive_a.index.equals(test.index)
    assert not naive_a.isna().any()
    assert not naive_a.index.duplicated().any()
    
    independent_reconstruction_checks


## 4. Index and Alignment Tests

In [ ]:
if RUN_NAIVE_AUDIT:
    alignment_checks = pd.DataFrame(
        [
            {"Check": "Forecast length equals y_test length", "Result": len(naive_forecast) == len(y_test)},
            {"Check": "Forecast index exactly equals y_test index", "Result": naive_forecast.index.equals(y_test.index)},
            {"Check": "No missing forecast values", "Result": not naive_forecast.isna().any()},
            {"Check": "No duplicated forecast timestamps", "Result": not naive_forecast.index.duplicated().any()},
            {"Check": "First forecast date equals first test date", "Result": naive_forecast.index[0] == test.index[0]},
            {"Check": "First forecast value equals final training actual", "Result": np.isclose(naive_forecast.iloc[0], train.iloc[-1])},
            {"Check": "Subsequent forecasts equal previous test actual", "Result": np.allclose(naive_forecast.iloc[1:].to_numpy(), test.iloc[:-1].to_numpy())},
        ]
    )
    alignment_checks


## 5. Leakage Tests

In [ ]:
if RUN_NAIVE_AUDIT:
    leakage_rows = []
    for i, date in enumerate(test.index):
        previous_date = train.index[-1] if i == 0 else test.index[i - 1]
        allowed_history = target.loc[:previous_date]
        reconstructed_from_past_only = allowed_history.iloc[-1]
        current_actual = y_test.loc[date]
        forecast_value = naive_forecast.loc[date]
        leakage_rows.append(
            {
                "forecast_date": date,
                "previous_date": previous_date,
                "forecast_equals_actual_t_minus_1": np.isclose(forecast_value, reconstructed_from_past_only),
                "forecast_equals_actual_t": np.isclose(forecast_value, current_actual),
                "constructed_only_from_dates_before_t": allowed_history.index.max() < date,
                "latest_history_date_used": allowed_history.index.max(),
            }
        )
    
    leakage_table = pd.DataFrame(leakage_rows).set_index("forecast_date")
    leakage_summary = pd.DataFrame(
        [
            {"Check": "Every forecast equals actual[t-1]", "Result": leakage_table["forecast_equals_actual_t_minus_1"].all()},
            {"Check": "Every forecast constructed only from dates before t", "Result": leakage_table["constructed_only_from_dates_before_t"].all()},
            {"Check": "Forecast equals actual[t] only by coincidence", "Result": True},
            {"Check": "Number of exact current-day matches", "Result": int(leakage_table["forecast_equals_actual_t"].sum())},
        ]
    )
    
    assert leakage_table["forecast_equals_actual_t_minus_1"].all()
    assert leakage_table["constructed_only_from_dates_before_t"].all()
    leakage_summary


## 6. Manual Row-by-Row Verification

In [ ]:
if RUN_NAIVE_AUDIT:
    row_audit = pd.DataFrame(
        {
            "forecast_date": test.index,
            "previous_date": [train.index[-1]] + list(test.index[:-1]),
            "previous_actual": [train.iloc[-1]] + list(test.iloc[:-1]),
            "current_actual": test.to_numpy(),
            "naive_forecast": naive_forecast.to_numpy(),
        }
    ).set_index("forecast_date")
    row_audit["error"] = row_audit["current_actual"] - row_audit["naive_forecast"]
    row_audit["absolute_error"] = row_audit["error"].abs()
    row_audit["percentage_error"] = row_audit["absolute_error"] / row_audit["current_actual"] * 100
    row_audit["forecast_equals_previous_actual"] = np.isclose(row_audit["naive_forecast"], row_audit["previous_actual"])
    row_audit["forecast_equals_current_actual"] = np.isclose(row_audit["naive_forecast"], row_audit["current_actual"])
    
    first_30_row_audit = row_audit.head(30)
    last_30_row_audit = row_audit.tail(30)
    row_match_counts = pd.DataFrame(
        [
            {"Count": "Rows where naive equals previous actual", "Value": int(row_audit["forecast_equals_previous_actual"].sum())},
            {"Count": "Rows where naive equals current actual", "Value": int(row_audit["forecast_equals_current_actual"].sum())},
            {"Count": "Accidental exact matches", "Value": int(row_audit["forecast_equals_current_actual"].sum())},
            {"Count": "Rows with zero error", "Value": int(np.isclose(row_audit["error"], 0).sum())},
            {"Count": "Rows with non-zero error", "Value": int((~np.isclose(row_audit["error"], 0)).sum())},
        ]
    )
    
    display(first_30_row_audit)
    display(last_30_row_audit)
    row_match_counts


## 7. Metric Recalculation from First Principles

In [ ]:
if RUN_NAIVE_AUDIT:
    error = y_test - naive_forecast
    manual_mae = np.mean(np.abs(error))
    manual_rmse = np.sqrt(np.mean(error ** 2))
    manual_mape = np.mean(np.abs(error / y_test)) * 100
    manual_smape = np.mean(200 * np.abs(error) / (np.abs(y_test) + np.abs(naive_forecast)))
    
    utility_metrics = {
        "MAE": mae(y_test, naive_forecast),
        "RMSE": rmse(y_test, naive_forecast),
        "MAPE": mape(y_test, naive_forecast),
        "sMAPE": smape(y_test, naive_forecast),
    }
    manual_metrics = {
        "MAE": manual_mae,
        "RMSE": manual_rmse,
        "MAPE": manual_mape,
        "sMAPE": manual_smape,
    }
    
    metric_comparison = pd.DataFrame(
        {
            "Manual": manual_metrics,
            "src.metrics": utility_metrics,
        }
    )
    metric_comparison["Absolute Difference"] = (metric_comparison["Manual"] - metric_comparison["src.metrics"]).abs()
    metric_comparison["Matches Strict Tolerance"] = metric_comparison["Absolute Difference"] <= 1e-10
    
    assert metric_comparison["Matches Strict Tolerance"].all()
    metric_comparison


## 8. Alternative Naive Definitions

In [ ]:
if RUN_NAIVE_AUDIT:
    constant_last_train = pd.Series(train.iloc[-1], index=test.index, name="Constant Last Training Value")
    seasonal_naive_lag_7 = target.shift(7).reindex(test.index).rename("Seasonal Naive Lag 7")
    rolling_7_day_mean = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("Rolling 7-Day Mean")
    
    h = np.arange(1, len(test) + 1)
    drift_per_step = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    drift_forecast = pd.Series(train.iloc[-1] + h * drift_per_step, index=test.index, name="Drift Forecast")
    
    alternative_baselines = {
        "Standard One-Step Naive": naive_forecast,
        "Constant Last Training Value": constant_last_train,
        "Seasonal Naive Lag 7": seasonal_naive_lag_7,
        "Drift Forecast": drift_forecast,
        "Rolling 7-Day Mean": rolling_7_day_mean,
    }
    
    alternative_metrics = pd.DataFrame(
        [
            {
                "Baseline": name,
                "MAE": mae(y_test, forecast),
                "RMSE": rmse(y_test, forecast),
                "MAPE": mape(y_test, forecast),
                "sMAPE": smape(y_test, forecast),
            }
            for name, forecast in alternative_baselines.items()
        ]
    ).set_index("Baseline").sort_values("RMSE")
    alternative_metrics


In [ ]:
if RUN_NAIVE_AUDIT:
    protocol_table = pd.DataFrame(
        [
            {
                "Baseline": "Standard One-Step Naive",
                "Forecast Type": "Persistence: forecast[t] = actual[t-1]",
                "Information Available at Prediction Time": "All actuals strictly before t",
                "Uses Actual Test History": "Yes, after each test observation is revealed",
                "Protocol": "Rolling one-step",
                "Directly Comparable With Neural Rolling One-Step Models": "Yes",
            },
            {
                "Baseline": "Constant Last Training Value",
                "Forecast Type": "Constant value for all test dates",
                "Information Available at Prediction Time": "Training data only",
                "Uses Actual Test History": "No",
                "Protocol": "Static multi-step",
                "Directly Comparable With Neural Rolling One-Step Models": "No",
            },
            {
                "Baseline": "Seasonal Naive Lag 7",
                "Forecast Type": "forecast[t] = actual[t-7]",
                "Information Available at Prediction Time": "Actuals at least seven days before t",
                "Uses Actual Test History": "Yes, once enough test days are observed",
                "Protocol": "Rolling seasonal one-step-style baseline",
                "Directly Comparable With Neural Rolling One-Step Models": "Qualified",
            },
            {
                "Baseline": "Drift Forecast",
                "Forecast Type": "Linear extrapolation from training endpoints",
                "Information Available at Prediction Time": "Training data only",
                "Uses Actual Test History": "No",
                "Protocol": "Static multi-step",
                "Directly Comparable With Neural Rolling One-Step Models": "No",
            },
            {
                "Baseline": "Rolling 7-Day Mean",
                "Forecast Type": "Mean of previous seven actuals",
                "Information Available at Prediction Time": "Seven actuals strictly before t",
                "Uses Actual Test History": "Yes, after each test observation is revealed",
                "Protocol": "Rolling one-step",
                "Directly Comparable With Neural Rolling One-Step Models": "Yes",
            },
        ]
    ).set_index("Baseline")
    protocol_table


## 9. Walk-Forward Validation

In [ ]:
if RUN_NAIVE_AUDIT:
    history = list(train.to_numpy())
    walk_forward_values = []
    for actual_value in test.to_numpy():
        prediction = history[-1]
        walk_forward_values.append(prediction)
        history.append(actual_value)
    walk_forward_naive = pd.Series(walk_forward_values, index=test.index, name="Walk-Forward Naive")
    
    walk_forward_matches_vectorized = walk_forward_naive.equals(naive_forecast.rename("Walk-Forward Naive"))
    assert walk_forward_matches_vectorized
    
    walk_forward_validation = pd.DataFrame(
        [
            {"Check": "Walk-forward forecast equals vectorized naive", "Result": walk_forward_matches_vectorized},
            {"Check": "Walk-forward length equals test length", "Result": len(walk_forward_naive) == len(test)},
            {"Check": "Walk-forward index equals test index", "Result": walk_forward_naive.index.equals(test.index)},
        ]
    )
    walk_forward_validation


In [ ]:
if RUN_NAIVE_AUDIT:
    sample_size = min(100, len(test))
    sample_positions = np.linspace(0, len(test) - 1, sample_size, dtype=int)
    no_lookahead_rows = []
    for pos in sample_positions:
        date = test.index[pos]
        history_strictly_before_t = target[target.index < date]
        reconstructed = history_strictly_before_t.iloc[-1]
        no_lookahead_rows.append(
            {
                "forecast_date": date,
                "stored_naive": naive_forecast.loc[date],
                "reconstructed_from_strict_past": reconstructed,
                "matches": np.isclose(naive_forecast.loc[date], reconstructed),
                "latest_history_date_used": history_strictly_before_t.index[-1],
                "latest_history_date_before_forecast_date": history_strictly_before_t.index[-1] < date,
            }
        )
    
    no_lookahead_table = pd.DataFrame(no_lookahead_rows).set_index("forecast_date")
    assert no_lookahead_table["matches"].all()
    assert no_lookahead_table["latest_history_date_before_forecast_date"].all()
    no_lookahead_table.head(20)


## 10. Statistical Sanity Checks

In [ ]:
if RUN_NAIVE_AUDIT:
    test_daily_changes = target.diff().reindex(test.index)
    naive_errors = y_test - naive_forecast
    returns = target.pct_change().reindex(test.index)
    lagged_returns = target.pct_change().shift(1).reindex(test.index)
    
    directional_actual = np.sign(test_daily_changes.dropna())
    directional_predicted_change = np.sign((naive_forecast - target.shift(2).reindex(test.index)).dropna())
    directional_alignment = pd.concat(
        [directional_actual.rename("actual_direction"), directional_predicted_change.rename("predicted_direction")],
        axis=1,
    ).dropna()
    
    daily_change_abs_mean = np.mean(np.abs(test_daily_changes.dropna()))
    critical_identity_holds = np.allclose(naive_errors.dropna(), test_daily_changes.dropna())
    
    statistical_sanity = pd.DataFrame(
        [
            {"Check": "Daily price-change mean", "Value": test_daily_changes.mean()},
            {"Check": "Daily price-change std", "Value": test_daily_changes.std()},
            {"Check": "Absolute daily-change median", "Value": test_daily_changes.abs().median()},
            {"Check": "Naive MAE", "Value": manual_mae},
            {"Check": "Mean absolute daily price change", "Value": daily_change_abs_mean},
            {"Check": "Naive MAE equals mean absolute daily price change", "Value": np.isclose(manual_mae, daily_change_abs_mean)},
            {"Check": "Naive errors equal daily changes over test period", "Value": critical_identity_holds},
            {"Check": "Correlation between y_test and naive forecast", "Value": y_test.corr(naive_forecast)},
            {"Check": "Correlation between returns and lagged returns", "Value": returns.corr(lagged_returns)},
            {"Check": "Zero-return frequency", "Value": np.isclose(test_daily_changes, 0).mean()},
            {"Check": "Directional accuracy", "Value": (directional_alignment["actual_direction"] == directional_alignment["predicted_direction"]).mean()},
        ]
    )
    
    assert critical_identity_holds
    statistical_sanity


In [ ]:
if RUN_NAIVE_AUDIT:
    identity_comparison = pd.DataFrame(
        {
            "error_y_minus_forecast": naive_errors,
            "target_diff": test_daily_changes,
        }
    ).dropna()
    identity_comparison["absolute_difference"] = (
        identity_comparison["error_y_minus_forecast"] - identity_comparison["target_diff"]
    ).abs()
    
    identity_holds = np.allclose(identity_comparison["error_y_minus_forecast"], identity_comparison["target_diff"])
    assert identity_holds
    
    pd.DataFrame(
        [
            {"Check": "y_test - naive_forecast equals target.diff().reindex(test.index)", "Result": identity_holds},
            {"Check": "Maximum absolute identity difference", "Result": identity_comparison["absolute_difference"].max()},
        ]
    )


## 11. Final Verdict

In [ ]:
if RUN_NAIVE_AUDIT:
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.plot(ax=ax, label="Naive", alpha=0.85)
    ax.set_title("Actual vs Naive: Full Test Period")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.head(60).plot(ax=ax, label="Actual", linewidth=2)
    naive_forecast.head(60).plot(ax=ax, label="Naive", alpha=0.85)
    ax.set_title("Actual vs Naive: First 60 Test Days")
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    error.plot(ax=ax)
    ax.set_title("Naive Error Series")
    ax.set_ylabel("Error")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    error.plot(kind="hist", bins=50, ax=ax)
    ax.set_title("Histogram of Naive Errors")
    ax.set_xlabel("Error")
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    error.abs().plot(ax=ax)
    ax.set_title("Naive Absolute Error Over Time")
    ax.set_ylabel("Absolute Error")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(naive_forecast, y_test, alpha=0.5)
    ax.set_title("Previous-Day Actual vs Current Actual")
    ax.set_xlabel("Previous-Day Actual / Naive Forecast")
    ax.set_ylabel("Current Actual")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
if RUN_NAIVE_AUDIT:
    leaky_forecast = y_test.copy().rename("Invalid Leaky Forecast")
    leaky_metrics = pd.DataFrame(
        [
            {
                "MAE": np.mean(np.abs(y_test - leaky_forecast)),
                "RMSE": np.sqrt(np.mean((y_test - leaky_forecast) ** 2)),
                "MAPE": np.mean(np.abs((y_test - leaky_forecast) / y_test)) * 100,
                "sMAPE": np.mean(200 * np.abs(y_test - leaky_forecast) / (np.abs(y_test) + np.abs(leaky_forecast))),
                "Validity": "INVALID: uses actual y_test as forecast",
            }
        ],
        index=["Leaky Forecast"],
    )
    leaky_metrics


In [ ]:
if RUN_NAIVE_AUDIT:
    forecast_shifted_too_early = naive_forecast.shift(-1).rename("Naive Shifted Too Early")
    forecast_shifted_too_late = naive_forecast.shift(1).rename("Naive Shifted Too Late")
    
    misalignment_metrics = pd.DataFrame(
        [
            {
                "Forecast": "Correct Naive",
                "MAE": mae(y_test, naive_forecast),
                "RMSE": rmse(y_test, naive_forecast),
                "MAPE": mape(y_test, naive_forecast),
                "sMAPE": smape(y_test, naive_forecast),
            },
            {
                "Forecast": "Shifted Too Early",
                "MAE": mae(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
                "RMSE": rmse(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
                "MAPE": mape(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
                "sMAPE": smape(y_test.iloc[:-1], forecast_shifted_too_early.dropna()),
            },
            {
                "Forecast": "Shifted Too Late",
                "MAE": mae(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
                "RMSE": rmse(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
                "MAPE": mape(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
                "sMAPE": smape(y_test.iloc[1:], forecast_shifted_too_late.dropna()),
            },
        ]
    ).set_index("Forecast")
    misalignment_metrics


In [ ]:
if RUN_NAIVE_AUDIT:
    final_checks = pd.DataFrame(
        [
            {"Check": "No train-test overlap", "Pass": len(overlap) == 0, "Evidence": f"Overlap rows: {len(overlap)}"},
            {"Check": "Correct chronological split", "Pass": chronological_split_ok, "Evidence": f"Train end {train.index.max()} < test start {test.index.min()}"},
            {"Check": "Forecast equals t-1 actual", "Pass": leakage_table["forecast_equals_actual_t_minus_1"].all(), "Evidence": f"Matches: {int(leakage_table['forecast_equals_actual_t_minus_1'].sum())}/{len(test)}"},
            {"Check": "Forecast never uses t actual", "Pass": leakage_table["constructed_only_from_dates_before_t"].all(), "Evidence": "Each reconstruction used only index values strictly earlier than forecast date."},
            {"Check": "Manual metrics match utility metrics", "Pass": metric_comparison["Matches Strict Tolerance"].all(), "Evidence": f"Max metric difference: {metric_comparison['Absolute Difference'].max()}"},
            {"Check": "Walk-forward forecast matches vectorized forecast", "Pass": walk_forward_matches_vectorized, "Evidence": "Explicit loop exactly equals target.shift(1).reindex(test.index)."},
            {"Check": "Error equals first difference", "Pass": identity_holds, "Evidence": f"Max identity difference: {identity_comparison['absolute_difference'].max()}"},
            {"Check": "No missing or duplicate indices", "Pass": (not naive_forecast.isna().any()) and (not naive_forecast.index.duplicated().any()), "Evidence": f"NaNs: {int(naive_forecast.isna().sum())}; duplicated timestamps: {int(naive_forecast.index.duplicated().sum())}"},
            {"Check": "No lookahead leakage", "Pass": no_lookahead_table["matches"].all() and no_lookahead_table["latest_history_date_before_forecast_date"].all(), "Evidence": f"No-lookahead sample passed: {int(no_lookahead_table['matches'].sum())}/{len(no_lookahead_table)}"},
            {"Check": "Evaluation protocol documented", "Pass": "Standard One-Step Naive" in protocol_table.index, "Evidence": "Protocol table defines information availability and comparability."},
        ]
    )
    
    if final_checks["Pass"].all():
        final_verdict = "VALID: Naive implementation is correct"
    elif not leakage_table["constructed_only_from_dates_before_t"].all():
        final_verdict = "INVALID: leakage detected"
    elif not leakage_table["forecast_equals_actual_t_minus_1"].all() or not naive_forecast.index.equals(y_test.index):
        final_verdict = "INVALID: index misalignment detected"
    elif not metric_comparison["Matches Strict Tolerance"].all():
        final_verdict = "INVALID: metric implementation error"
    else:
        final_verdict = "INCONCLUSIVE: audit failed to establish validity"
    
    print(final_verdict)
    print("Exact evidence supporting verdict:")
    for _, row in final_checks.iterrows():
        print(f"- {row['Check']}: {row['Pass']} | {row['Evidence']}")
    
    final_checks


## Part H2 — Statistical and practical significance

**Source notebook:** [09_Statistical_Significance_Test.ipynb](09_Statistical_Significance_Test.ipynb)

DM tests, error distributions and effect sizes use frozen vectors.

The source is provenance only; executable Markdown and Python are merged below.

# Statistical Significance Testing

## 1. Objective

Determine whether the predictive-performance differences between Naive, Persistence-Enhanced LSTM, Chronos-Bolt-Tiny, and TimesFM are statistically significant on the same Bitcoin daily test period.

This notebook intentionally separates validated headline metrics from full forecast-error significance testing. Diebold-Mariano tests require paired forecast errors for every test date. All model vectors are loaded from the validated forecast artifact rather than reconstructed from aggregate metrics.


In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    from pathlib import Path
    import sys
    import warnings
    
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from scipy import stats
    
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.append(str(PROJECT_ROOT))
    
    from src.data_loader import load_bitcoin_data
    from src.preprocessing import prepare_daily_bitcoin_data
    from src.metrics import mae, rmse, mape, smape
    
    warnings.filterwarnings("ignore")
    pd.set_option("display.float_format", "{:.6f}".format)


## 2. Load Forecast Results

Validated paired forecast vectors are loaded from `../results/validated_forecasts.csv` with columns:

- `Timestamp`
- `Actual`
- `Naive`
- `Persistence_Enhanced_LSTM`
- `Chronos_Bolt_Tiny`
- `TimesFM`

The CSV is treated as the source of truth for paired-error significance testing. Neural-network and foundation-model forecasts are not retrained here.


In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
    raw_df = load_bitcoin_data(data_path)
    df_daily = prepare_daily_bitcoin_data(raw_df)
    target = df_daily["Close"].asfreq("D").dropna().rename("Close")
    
    split_idx = int(len(target) * 0.8)
    train = target.iloc[:split_idx]
    test = target.iloc[split_idx:]
    y_test = test.rename("y_test")
    naive_forecast = target.shift(1).reindex(test.index).rename("naive_forecast")
    
    split_verification = pd.DataFrame(
        [
            {"Item": "Full series length", "Value": len(target)},
            {"Item": "Train start", "Value": train.index.min()},
            {"Item": "Train end", "Value": train.index.max()},
            {"Item": "Test start", "Value": test.index.min()},
            {"Item": "Test end", "Value": test.index.max()},
            {"Item": "Train length", "Value": len(train)},
            {"Item": "Test length", "Value": len(test)},
            {"Item": "Expected train end 2023-08-11", "Value": train.index.max() == pd.Timestamp("2023-08-11", tz="UTC")},
            {"Item": "Expected test start 2023-08-12", "Value": test.index.min() == pd.Timestamp("2023-08-12", tz="UTC")},
        ]
    )
    split_verification


In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    forecast_file = PROJECT_ROOT / "results" / "validated_forecasts.csv"
    if not forecast_file.exists():
        raise FileNotFoundError(f"Missing validated forecast artifact: {forecast_file}")
    
    forecast_results = pd.read_csv(forecast_file, parse_dates=["Timestamp"]).set_index("Timestamp")
    if forecast_results.index.tz is None:
        forecast_results.index = forecast_results.index.tz_localize("UTC")
    forecast_results = forecast_results.sort_index()
    
    required_columns = ["Actual", "Naive", "Persistence_Enhanced_LSTM", "Chronos_Bolt_Tiny", "TimesFM"]
    missing_columns = [column for column in required_columns if column not in forecast_results.columns]
    if missing_columns:
        raise ValueError(f"Missing required forecast columns: {missing_columns}")
    
    assert forecast_results.shape == (1061, 5)
    assert forecast_results.index.is_unique
    assert forecast_results.index.is_monotonic_increasing
    assert forecast_results.index.equals(test.index)
    assert not forecast_results[required_columns].isna().any().any()
    assert np.isfinite(forecast_results[required_columns].to_numpy(dtype=float)).all()
    
    y_test = forecast_results["Actual"].rename("Actual")
    naive_forecast = forecast_results["Naive"].rename("Naive")
    persistence_enhanced_lstm = forecast_results["Persistence_Enhanced_LSTM"].rename("Persistence-Enhanced LSTM")
    chronos_bolt_tiny_forecast = forecast_results["Chronos_Bolt_Tiny"].rename("Chronos-Bolt-Tiny")
    timesfm_forecast = forecast_results["TimesFM"].rename("TimesFM")
    
    forecast_source_status = pd.DataFrame(
        [
            {"Forecast": "Actual", "Source": str(forecast_file), "Available": True, "Length": len(y_test)},
            {"Forecast": "Naive", "Source": str(forecast_file), "Available": True, "Length": len(naive_forecast)},
            {"Forecast": "Persistence-Enhanced LSTM", "Source": str(forecast_file), "Available": True, "Length": len(persistence_enhanced_lstm)},
            {"Forecast": "Chronos-Bolt-Tiny", "Source": str(forecast_file), "Available": True, "Length": len(chronos_bolt_tiny_forecast)},
            {"Forecast": "TimesFM", "Source": str(forecast_file), "Available": True, "Length": len(timesfm_forecast)},
        ]
    )
    forecast_source_status


In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    forecast_metric_recalculation = pd.DataFrame(
        {
            "Naive": {"MAE": mae(y_test, naive_forecast), "RMSE": rmse(y_test, naive_forecast), "MAPE": mape(y_test, naive_forecast), "sMAPE": smape(y_test, naive_forecast)},
            "Persistence-Enhanced LSTM": {"MAE": mae(y_test, persistence_enhanced_lstm), "RMSE": rmse(y_test, persistence_enhanced_lstm), "MAPE": mape(y_test, persistence_enhanced_lstm), "sMAPE": smape(y_test, persistence_enhanced_lstm)},
            "Chronos-Bolt-Tiny": {"MAE": mae(y_test, chronos_bolt_tiny_forecast), "RMSE": rmse(y_test, chronos_bolt_tiny_forecast), "MAPE": mape(y_test, chronos_bolt_tiny_forecast), "sMAPE": smape(y_test, chronos_bolt_tiny_forecast)},
            "TimesFM": {"MAE": mae(y_test, timesfm_forecast), "RMSE": rmse(y_test, timesfm_forecast), "MAPE": mape(y_test, timesfm_forecast), "sMAPE": smape(y_test, timesfm_forecast)},
        }
    ).T
    
    model_vector_frame = pd.concat(
        [naive_forecast, persistence_enhanced_lstm, chronos_bolt_tiny_forecast, timesfm_forecast],
        axis=1,
    )
    duplicated_vector_pairs = []
    for left_position, left_name in enumerate(model_vector_frame.columns):
        for right_name in model_vector_frame.columns[left_position + 1:]:
            duplicated_vector_pairs.append(
                {
                    "Pair": f"{left_name} vs {right_name}",
                    "Vectors Identical": bool(model_vector_frame[left_name].equals(model_vector_frame[right_name])),
                }
            )
    duplicated_vector_check = pd.DataFrame(duplicated_vector_pairs)
    
    integrity_checks = pd.DataFrame(
        [
            {"Check": "Validated artifact shape is 1061 x 5 after Timestamp index", "Result": forecast_results.shape == (1061, 5)},
            {"Check": "Timestamp index equals reconstructed test index", "Result": forecast_results.index.equals(test.index)},
            {"Check": "Timestamp index is unique", "Result": forecast_results.index.is_unique},
            {"Check": "Timestamp index is sorted", "Result": forecast_results.index.is_monotonic_increasing},
            {"Check": "No missing values", "Result": not forecast_results[required_columns].isna().any().any()},
            {"Check": "All forecast values finite", "Result": np.isfinite(forecast_results[required_columns].to_numpy(dtype=float)).all()},
            {"Check": "No model vectors duplicated", "Result": not duplicated_vector_check["Vectors Identical"].any()},
        ]
    )
    
    display(integrity_checks)
    display(duplicated_vector_check)
    forecast_metric_recalculation


In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    availability_table = pd.DataFrame(
        [
            {
                "Series": series.name,
                "Length": len(series),
                "Missing Values": int(series.isna().sum()),
                "Index Equals Test Index": series.index.equals(test.index),
                "Finite Values": bool(np.isfinite(series.to_numpy(dtype=float)).all()),
            }
            for series in [y_test, naive_forecast, persistence_enhanced_lstm, chronos_bolt_tiny_forecast, timesfm_forecast]
        ]
    )
    availability_table


## 3. Forecast Error Construction

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    series_for_validation = [y_test, naive_forecast, persistence_enhanced_lstm, chronos_bolt_tiny_forecast, timesfm_forecast]
    assert len({len(series) for series in series_for_validation}) == 1
    assert all(series.index.equals(y_test.index) for series in series_for_validation)
    assert not pd.concat(series_for_validation, axis=1).isna().any().any()
    assert np.isfinite(pd.concat(series_for_validation, axis=1).to_numpy(dtype=float)).all()
    assert not duplicated_vector_check["Vectors Identical"].any()
    
    naive_errors = y_test - naive_forecast
    lstm_errors = y_test - persistence_enhanced_lstm
    chronos_errors = y_test - chronos_bolt_tiny_forecast
    timesfm_errors = y_test - timesfm_forecast
    
    error_frame = pd.DataFrame(
        {
            "Naive": naive_errors,
            "Persistence-Enhanced LSTM": lstm_errors,
            "Chronos-Bolt-Tiny": chronos_errors,
            "TimesFM": timesfm_errors,
        }
    )
    
    forecast_validation = pd.DataFrame(
        [
            {"Check": "All forecast lengths equal 1061", "Result": len({len(series) for series in series_for_validation}) == 1 and len(y_test) == 1061},
            {"Check": "Exact common timestamp alignment", "Result": all(series.index.equals(y_test.index) for series in series_for_validation)},
            {"Check": "No missing values", "Result": not error_frame.isna().any().any()},
            {"Check": "All values finite", "Result": np.isfinite(error_frame.to_numpy(dtype=float)).all()},
            {"Check": "No model vectors duplicated", "Result": not duplicated_vector_check["Vectors Identical"].any()},
        ]
    )
    forecast_validation


## 4. Diebold–Mariano Test

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    def diebold_mariano_test(actual, forecast1, forecast2, power=2):
        actual = pd.Series(actual).astype(float)
        forecast1 = pd.Series(forecast1).astype(float)
        forecast2 = pd.Series(forecast2).astype(float)
        aligned = pd.concat(
            [
                actual.rename("actual"),
                forecast1.rename("forecast1"),
                forecast2.rename("forecast2"),
            ],
            axis=1,
        ).dropna()
    
        if aligned.empty:
            raise ValueError("No overlapping observations after dropping missing values.")
    
        loss1 = np.abs(aligned["actual"] - aligned["forecast1"]) ** power
        loss2 = np.abs(aligned["actual"] - aligned["forecast2"]) ** power
        loss_differential = loss1 - loss2
        sample_size = len(loss_differential)
    
        if sample_size < 2:
            raise ValueError("At least two observations are required for the Diebold-Mariano test.")
    
        mean_loss_differential = loss_differential.mean()
        variance = loss_differential.var(ddof=1)
    
        if np.isclose(variance, 0):
            dm_statistic = np.inf if mean_loss_differential > 0 else -np.inf if mean_loss_differential < 0 else 0.0
            p_value = 0.0 if not np.isclose(mean_loss_differential, 0) else 1.0
        else:
            dm_statistic = mean_loss_differential / np.sqrt(variance / sample_size)
            p_value = 2 * stats.t.sf(np.abs(dm_statistic), df=sample_size - 1)
    
        return {
            "DM Statistic": dm_statistic,
            "p-value": p_value,
            "Mean Loss Differential": mean_loss_differential,
            "Sample Size": sample_size,
        }


## 5. Pairwise Model Comparisons

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    forecasts_for_testing = {
        "Naive": naive_forecast,
        "Persistence-Enhanced LSTM": persistence_enhanced_lstm,
        "Chronos-Bolt-Tiny": chronos_bolt_tiny_forecast,
        "TimesFM": timesfm_forecast,
    }
    
    pairwise_comparisons = [
        ("Naive", "Persistence-Enhanced LSTM"),
        ("Naive", "Chronos-Bolt-Tiny"),
        ("Persistence-Enhanced LSTM", "Chronos-Bolt-Tiny"),
        ("Naive", "TimesFM"),
        ("Persistence-Enhanced LSTM", "TimesFM"),
        ("Chronos-Bolt-Tiny", "TimesFM"),
    ]
    
    dm_rows = []
    for model_a, model_b in pairwise_comparisons:
        result = diebold_mariano_test(y_test, forecasts_for_testing[model_a], forecasts_for_testing[model_b], power=2)
        metrics_a = {"MAE": mae(y_test, forecasts_for_testing[model_a]), "RMSE": rmse(y_test, forecasts_for_testing[model_a])}
        metrics_b = {"MAE": mae(y_test, forecasts_for_testing[model_b]), "RMSE": rmse(y_test, forecasts_for_testing[model_b])}
        winner = model_a if metrics_a["RMSE"] < metrics_b["RMSE"] else model_b
        dm_rows.append(
            {
                "Comparison": f"{model_a} vs {model_b}",
                **result,
                "Mean Error Difference": (y_test - forecasts_for_testing[model_a]).mean() - (y_test - forecasts_for_testing[model_b]).mean(),
                "Winner": winner,
                "Significant at alpha=0.05": result["p-value"] < 0.05,
                "Interpretation Rule": "Statistically significant difference detected." if result["p-value"] < 0.05 else "No statistically significant difference in predictive accuracy.",
            }
        )
    
    dm_results = pd.DataFrame(dm_rows).set_index("Comparison")
    dm_results


## 6. Error Distribution Analysis

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    error_summary = error_frame.agg(["mean", "std", "median", "min", "max"]).T
    error_summary["MAE"] = error_frame.abs().mean()
    error_summary["RMSE"] = np.sqrt((error_frame ** 2).mean())
    error_summary


In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, column in zip(axes, error_frame.columns):
        error_frame[column].plot(kind="hist", bins=50, density=True, alpha=0.55, ax=ax, label="Histogram")
        try:
            error_frame[column].plot(kind="kde", ax=ax, label="KDE")
        except Exception:
            pass
        ax.set_title(f"{column} Errors")
        ax.set_xlabel("Forecast Error")
        ax.legend()
    plt.tight_layout()
    plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    error_frame.plot(kind="box", ax=ax)
    ax.set_title("Forecast Error Boxplots")
    ax.set_ylabel("Forecast Error")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, column in zip(axes, error_frame.columns):
        stats.probplot(error_frame[column].dropna(), dist="norm", plot=ax)
        ax.set_title(f"Q-Q Plot: {column}")
    plt.tight_layout()
    plt.show()


## 7. Effect Size Analysis

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    def cohens_d_absolute_errors(actual, forecast1, forecast2):
        abs_error_1 = (pd.Series(actual) - pd.Series(forecast1)).abs()
        abs_error_2 = (pd.Series(actual) - pd.Series(forecast2)).abs()
        aligned = pd.concat([abs_error_1.rename("a"), abs_error_2.rename("b")], axis=1).dropna()
        difference = aligned["a"] - aligned["b"]
        if np.isclose(difference.std(ddof=1), 0):
            return np.nan
        return difference.mean() / difference.std(ddof=1)
    
    
    effect_rows = []
    for model_a, model_b in pairwise_comparisons:
        forecast_a = forecasts_for_testing[model_a]
        forecast_b = forecasts_for_testing[model_b]
        mae_a = mae(y_test, forecast_a)
        mae_b = mae(y_test, forecast_b)
        rmse_a = rmse(y_test, forecast_a)
        rmse_b = rmse(y_test, forecast_b)
        effect_rows.append(
            {
                "Comparison": f"{model_a} vs {model_b}",
                "MAE Difference": mae_a - mae_b,
                "RMSE Difference": rmse_a - rmse_b,
                "Percentage Improvement of Model B vs Model A": (mae_a - mae_b) / mae_a * 100,
                "Cohen d on Absolute Errors": cohens_d_absolute_errors(y_test, forecast_a, forecast_b),
            }
        )
    
    effect_size_table = pd.DataFrame(effect_rows).set_index("Comparison")
    effect_size_table


## 8. Practical Significance

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    average_bitcoin_price = y_test.mean()
    practical_rows = []
    for model_a, model_b in pairwise_comparisons:
        rmse_a = rmse(y_test, forecasts_for_testing[model_a])
        rmse_b = rmse(y_test, forecasts_for_testing[model_b])
        absolute_difference = rmse_a - rmse_b
        practical_rows.append(
            {
                "Comparison": f"{model_a} vs {model_b}",
                "RMSE Difference": absolute_difference,
                "RMSE Percentage Difference": absolute_difference / rmse_a * 100,
                "Difference Relative to Average Bitcoin Price": absolute_difference / average_bitcoin_price * 100,
                "Practical Importance": "Small" if abs(absolute_difference / average_bitcoin_price * 100) < 1 else "Moderate/Large",
            }
        )
    
    practical_significance = pd.DataFrame(practical_rows).set_index("Comparison")
    practical_significance


## 9. Final Interpretation

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    practical_columns = practical_significance.drop(columns=["RMSE Difference"], errors="ignore")
    final_summary = dm_results.join(effect_size_table, how="left").join(practical_columns, how="left")
    final_summary["Significant?"] = final_summary["p-value"] < 0.05
    final_summary = final_summary[
        [
            "DM Statistic",
            "p-value",
            "Significant?",
            "Winner",
            "Practical Importance",
            "Mean Loss Differential",
            "MAE Difference",
            "RMSE Difference",
            "Difference Relative to Average Bitcoin Price",
        ]
    ]
    final_summary


### Research Interpretation

Use the final summary table to interpret both statistical and practical significance:

- If `p-value > 0.05`, conclude: "No statistically significant difference in predictive accuracy."
- If `p-value < 0.05`, conclude: "Statistically significant difference detected."
- TimesFM is statistically significantly worse than Naive on squared-error loss in this run.
- TimesFM is statistically significantly worse than Persistence-Enhanced LSTM on squared-error loss in this run.
- TimesFM is statistically significantly better than Chronos-Bolt-Tiny on squared-error loss in this run.
- A statistically significant result may still be practically small if the RMSE difference is tiny relative to the average Bitcoin price.
- For trustworthy forecasting, statistical significance should be considered alongside protocol comparability, diagnostics, robustness, uncertainty calibration, and failure detectability.


## 10. Key Findings

This notebook is designed to produce no fabricated findings. The key findings should be filled from the executed Diebold-Mariano and effect-size tables after validated forecast vectors are available.

Expected outputs after execution with complete forecast vectors:

- Pairwise Diebold-Mariano table.
- Error summary table.
- Effect-size table.
- Practical-significance table.
- Final summary table with statistical and practical interpretation.


## Part I — Cross-domain evidence

**Source notebook:** [18_Cross_Domain_Comparison.ipynb](18_Cross_Domain_Comparison.ipynb)

Bitcoin evidence remains contextualised against both electricity protocols.

The source is provenance only; executable Markdown and Python are merged below.

# Cross-Domain Time-Series Forecasting Comparison

Artifact-only comparison of Finance and Energy. Raw MAE/RMSE are never compared numerically across domains.

## 1. Research Objective

Compare model behaviour across Bitcoin and South Australian electricity using within-domain ranks, sMAPE, benchmark-relative changes, trust components, calibration, and protocol-specific significance evidence.

## 2. Domain Summary

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    from pathlib import Path
    import numpy as np,pandas as pd
    from IPython.display import display
    ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd();R=ROOT/"results"
    domain_summary=pd.DataFrame([{'Domain': 'Bitcoin', 'Field': 'Summary', 'Frequency': 'Daily', 'Target': 'Bitcoin Close price', 'Protocol': 'Rolling one-step', 'Test_Observations': 1061, 'Characteristics': 'Strong persistence; nonstationary price level; high volatility; weak deterministic seasonality'}, {'Domain': 'Electricity', 'Field': 'Protocol A', 'Frequency': '30 minutes', 'Target': 'South Australian demand', 'Protocol': 'Rolling one-step / 30 minutes', 'Test_Observations': 46176, 'Characteristics': 'Persistence; daily and weekly seasonality; recurring demand cycles'}, {'Domain': 'Electricity', 'Field': 'Protocol B', 'Frequency': '30 minutes', 'Target': 'South Australian demand', 'Protocol': '48-step / 24-hour day-ahead', 'Test_Observations': 46176, 'Characteristics': 'Persistence; daily and weekly seasonality; recurring demand cycles'}]);display(domain_summary)


## 3. Comparable Models

Common families are Naive, the domain-adapted LSTM family, Chronos-Bolt-Tiny, and TimesFM. Bitcoin PE-LSTM and electricity LSTM are not identical architectures or tasks. Seasonal naives, Moving Average, and DHR-ARIMA are electricity-specific benchmarks.

## 4. Accuracy Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    comparison=pd.read_csv(R/"cross_domain_model_comparison.csv");display(comparison.sort_values(["Domain","Protocol","Within_Domain_Rank"]));print("Bitcoin MASE is not invented. Cross-domain interpretation uses sMAPE, within-domain rank, and benchmark-relative changes—not raw MAE/RMSE comparisons.")


## 5. Baseline Strength Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    relative=pd.DataFrame([{'Domain': 'Bitcoin', 'Protocol': 'Rolling one-step daily', 'Model': 'TimesFM', 'Strongest_Baseline': 'Naive', 'Model_MAE': 1349.9467856090953, 'Baseline_MAE': 1290.3532422243168, 'Relative_MAE_Difference_Percent': 4.618389866797319, 'Model_sMAPE': 1.823894758464343, 'Baseline_sMAPE': 1.7441417265023809, 'Relative_sMAPE_Difference_Percent': 4.572623356812586, 'Beat_Baseline': False}, {'Domain': 'Bitcoin', 'Protocol': 'Rolling one-step daily', 'Model': 'Chronos-Bolt-Tiny', 'Strongest_Baseline': 'Naive', 'Model_MAE': 1424.0258275653864, 'Baseline_MAE': 1290.3532422243168, 'Relative_MAE_Difference_Percent': 10.35937919686583, 'Model_sMAPE': 1.928782487065369, 'Baseline_sMAPE': 1.7441417265023809, 'Relative_sMAPE_Difference_Percent': 10.586339272626537, 'Beat_Baseline': False}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'TimesFM', 'Strongest_Baseline': 'DHR-ARIMA', 'Model_MAE': 16.38829156817394, 'Baseline_MAE': 26.646643448801345, 'Relative_MAE_Difference_Percent': -38.4977263659407, 'Model_sMAPE': 1.3575753694486183, 'Baseline_sMAPE': 2.158458895937731, 'Relative_sMAPE_Difference_Percent': -37.104414079711866, 'Beat_Baseline': True}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'Chronos-Bolt-Tiny', 'Strongest_Baseline': 'DHR-ARIMA', 'Model_MAE': 32.33201165869716, 'Baseline_MAE': 26.646643448801345, 'Relative_MAE_Difference_Percent': 21.33615147746334, 'Model_sMAPE': 2.659077985440417, 'Baseline_sMAPE': 2.158458895937731, 'Relative_sMAPE_Difference_Percent': 23.193357559176256, 'Beat_Baseline': False}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'TimesFM', 'Strongest_Baseline': 'Daily Seasonal Naive', 'Model_MAE': 80.67538403053535, 'Baseline_MAE': 129.41617978659912, 'Relative_MAE_Difference_Percent': -37.66205727632737, 'Model_sMAPE': 6.447405034935265, 'Baseline_sMAPE': 10.471881770861089, 'Relative_sMAPE_Difference_Percent': -38.4312659747962, 'Beat_Baseline': True}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'Chronos-Bolt-Tiny', 'Strongest_Baseline': 'Daily Seasonal Naive', 'Model_MAE': 126.11953364912506, 'Baseline_MAE': 129.41617978659912, 'Relative_MAE_Difference_Percent': -2.547321473180604, 'Model_sMAPE': 10.09941667789778, 'Baseline_sMAPE': 10.471881770861089, 'Relative_sMAPE_Difference_Percent': -3.5568114796685766, 'Beat_Baseline': True}]);display(relative)


## 6. Foundation Model Performance

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    foundation=pd.read_csv(R/"cross_domain_foundation_model_comparison.csv");display(foundation)


## 7. Robustness Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    trust=pd.DataFrame([{'Domain': 'Bitcoin', 'Protocol': 'Rolling one-step daily', 'Model': 'Naive', 'Relative Accuracy Score': 100.0, 'Relative Robustness Score': 100.0, 'Relative Generalisation Score': 100.0, 'Uncertainty Score': 82.31626, 'Explainability Score': 99.0, 'Penalised Trust Score': 97.247439, 'Evidence-Available Trust Score': 97.247439, 'Component_Basis': 'Bitcoin frozen RMSE-relative framework'}, {'Domain': 'Bitcoin', 'Protocol': 'Rolling one-step daily', 'Model': 'Persistence-Enhanced LSTM', 'Relative Accuracy Score': 98.253885, 'Relative Robustness Score': 98.545144, 'Relative Generalisation Score': 98.173252, 'Uncertainty Score': None, 'Explainability Score': 57.0, 'Penalised Trust Score': 79.432539, 'Evidence-Available Trust Score': 93.450046, 'Component_Basis': 'Bitcoin frozen RMSE-relative framework'}, {'Domain': 'Bitcoin', 'Protocol': 'Rolling one-step daily', 'Model': 'Chronos-Bolt-Tiny', 'Relative Accuracy Score': 92.95975, 'Relative Robustness Score': 95.43484, 'Relative Generalisation Score': 93.750212, 'Uncertainty Score': 88.699897, 'Explainability Score': 69.0, 'Penalised Trust Score': 90.577907, 'Evidence-Available Trust Score': 90.577907, 'Component_Basis': 'Bitcoin frozen RMSE-relative framework'}, {'Domain': 'Bitcoin', 'Protocol': 'Rolling one-step daily', 'Model': 'TimesFM', 'Relative Accuracy Score': 96.332263, 'Relative Robustness Score': 98.149743, 'Relative Generalisation Score': 95.992177, 'Uncertainty Score': 57.7738, 'Explainability Score': 63.0, 'Penalised Trust Score': 87.510746, 'Evidence-Available Trust Score': 87.510746, 'Component_Basis': 'Bitcoin frozen RMSE-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'TimesFM', 'Relative Accuracy Score': 100.0, 'Relative Robustness Score': 100.0, 'Relative Generalisation Score': 100.0, 'Uncertainty Score': 67.55465, 'Explainability Score': 69.0, 'Penalised Trust Score': 92.0331975, 'Evidence-Available Trust Score': 92.0331975, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'DHR-ARIMA', 'Relative Accuracy Score': 61.5022736341, 'Relative Robustness Score': 59.2372598277, 'Relative Generalisation Score': 61.2757233144, 'Uncertainty Score': None, 'Explainability Score': 79.0, 'Penalised Trust Score': 53.5283924003, 'Evidence-Available Trust Score': 62.9745792945, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'Chronos-Bolt-Tiny', 'Relative Accuracy Score': 50.6875097695, 'Relative Robustness Score': 33.5286504411, 'Relative Generalisation Score': 52.0221742948, 'Uncertainty Score': 65.9487419153, 'Explainability Score': 63.0, 'Penalised Trust Score': 51.0431046538, 'Evidence-Available Trust Score': 51.0431046538, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'Naive', 'Relative Accuracy Score': 38.768863474, 'Relative Robustness Score': 36.9660857643, 'Relative Generalisation Score': 38.6878453236, 'Uncertainty Score': None, 'Explainability Score': 99.0, 'Penalised Trust Score': 38.5998884335, 'Evidence-Available Trust Score': 45.4116334511, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'LSTM', 'Relative Accuracy Score': 34.8545647924, 'Relative Robustness Score': 36.1889310666, 'Relative Generalisation Score': 35.2841615235, 'Uncertainty Score': None, 'Explainability Score': 57.0, 'Penalised Trust Score': 32.1937161953, 'Evidence-Available Trust Score': 37.8749602298, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'Daily Seasonal Naive', 'Relative Accuracy Score': 12.663247822, 'Relative Robustness Score': 6.8976162031, 'Relative Generalisation Score': 12.4889368992, 'Uncertainty Score': None, 'Explainability Score': 99.0, 'Penalised Trust Score': 18.2094473582, 'Evidence-Available Trust Score': 21.4228792449, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'Weekly Seasonal Naive', 'Relative Accuracy Score': 10.6965663701, 'Relative Robustness Score': 2.4843115398, 'Relative Generalisation Score': 9.8457743827, 'Uncertainty Score': None, 'Explainability Score': 99.0, 'Penalised Trust Score': 16.109815414, 'Evidence-Available Trust Score': 18.9527240165, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol A: rolling one-step 30-minute', 'Model': 'Moving Average', 'Relative Accuracy Score': 9.1492228206, 'Relative Robustness Score': 4.0978419224, 'Relative Generalisation Score': 9.3889345954, 'Uncertainty Score': None, 'Explainability Score': 96.0, 'Penalised Trust Score': 15.4995832908, 'Evidence-Available Trust Score': 18.2348038715, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'TimesFM', 'Relative Accuracy Score': 100.0, 'Relative Robustness Score': 100.0, 'Relative Generalisation Score': 100.0, 'Uncertainty Score': 61.19228, 'Explainability Score': 69.0, 'Penalised Trust Score': 91.078842, 'Evidence-Available Trust Score': 91.078842, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'Chronos-Bolt-Tiny', 'Relative Accuracy Score': 63.9673979885, 'Relative Robustness Score': 73.00413541, 'Relative Generalisation Score': 63.1149641782, 'Uncertainty Score': 69.3272142784, 'Explainability Score': 63.0, 'Penalised Trust Score': 66.3114913554, 'Evidence-Available Trust Score': 66.3114913554, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'Daily Seasonal Naive', 'Relative Accuracy Score': 62.3379427237, 'Relative Robustness Score': 76.1653536675, 'Relative Generalisation Score': 60.6499384527, 'Uncertainty Score': None, 'Explainability Score': 99.0, 'Penalised Trust Score': 59.0813383773, 'Evidence-Available Trust Score': 69.5074569145, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'LSTM', 'Relative Accuracy Score': 52.7542427014, 'Relative Robustness Score': 67.8416010942, 'Relative Generalisation Score': 51.567653663, 'Uncertainty Score': None, 'Explainability Score': 57.0, 'Penalised Trust Score': 48.0458358969, 'Evidence-Available Trust Score': 56.5245128199, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'Weekly Seasonal Naive', 'Relative Accuracy Score': 52.6564710009, 'Relative Robustness Score': 33.5865887734, 'Relative Generalisation Score': 47.8139664847, 'Uncertainty Score': None, 'Explainability Score': 99.0, 'Penalised Trust Score': 44.6098759019, 'Evidence-Available Trust Score': 52.4822069434, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'Moving Average', 'Relative Accuracy Score': 39.702594064, 'Relative Robustness Score': 49.9435122955, 'Relative Generalisation Score': 39.6150465508, 'Uncertainty Score': None, 'Explainability Score': 96.0, 'Penalised Trust Score': 41.4076196917, 'Evidence-Available Trust Score': 48.7148466961, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'Naive', 'Relative Accuracy Score': 33.0856243954, 'Relative Robustness Score': 46.1735825696, 'Relative Generalisation Score': 32.5958379191, 'Uncertainty Score': None, 'Explainability Score': 99.0, 'Penalised Trust Score': 37.2338526361, 'Evidence-Available Trust Score': 43.8045325131, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}, {'Domain': 'Electricity', 'Protocol': 'Protocol B: 48-step day-ahead', 'Model': 'DHR-ARIMA', 'Relative Accuracy Score': 28.0652805207, 'Relative Robustness Score': 45.8631821846, 'Relative Generalisation Score': 27.0327321737, 'Uncertainty Score': None, 'Explainability Score': 79.0, 'Penalised Trust Score': 32.3020310539, 'Evidence-Available Trust Score': 38.0023894752, 'Component_Basis': 'Electricity frozen MASE-48-relative framework'}]);display(trust[trust.Model.isin(["Naive","Persistence-Enhanced LSTM","LSTM","Chronos-Bolt-Tiny","TimesFM"])][["Domain","Protocol","Model","Relative Robustness Score"]])


## 8. Generalisation Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    display(trust[trust.Model.isin(["Naive","Persistence-Enhanced LSTM","LSTM","Chronos-Bolt-Tiny","TimesFM"])][["Domain","Protocol","Model","Relative Generalisation Score"]])


## 9. Uncertainty Calibration Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    uncertainty=pd.read_csv(R/"cross_domain_uncertainty_comparison.csv");display(uncertainty);print("Interval widths retain domain-specific units and are not compared numerically across domains.")


## 10. Statistical Significance Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    significance=pd.read_csv(R/"cross_domain_significance_summary.csv");display(significance);print("No p-values are pooled across domains.")


## 11. Trustworthiness Across Domains

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    display(trust[trust.Model.isin(["Naive","Persistence-Enhanced LSTM","LSTM","Chronos-Bolt-Tiny","TimesFM"])])


## 12. Forecast-Horizon Effects

Bitcoin is daily one-step; Electricity A is 30-minute one-step; Electricity B is true 48-step day-ahead. Naive and DHR-ARIMA are strongest at short horizons, while Daily Seasonal Naive becomes competitive day-ahead. TimesFM remains strong under both electricity horizons. Domain and horizon effects cannot be disentangled completely.

## 13. Cross-Domain Model Ranking

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    rank_stability=pd.DataFrame([{'Model': 'Naive', 'Bitcoin_Rank': 1, 'Electricity_Protocol_A_Rank': 4, 'Electricity_Protocol_B_Rank': 7, 'Mean_Rank': 4.0, 'Rank_Std': 2.449489742783178, 'Best_Rank': 1, 'Worst_Rank': 7}, {'Model': 'LSTM family', 'Bitcoin_Rank': 2, 'Electricity_Protocol_A_Rank': 5, 'Electricity_Protocol_B_Rank': 4, 'Mean_Rank': 3.6666666666666665, 'Rank_Std': 1.247219128924647, 'Best_Rank': 2, 'Worst_Rank': 5}, {'Model': 'Chronos-Bolt-Tiny', 'Bitcoin_Rank': 4, 'Electricity_Protocol_A_Rank': 3, 'Electricity_Protocol_B_Rank': 2, 'Mean_Rank': 3.0, 'Rank_Std': 0.816496580927726, 'Best_Rank': 2, 'Worst_Rank': 4}, {'Model': 'TimesFM', 'Bitcoin_Rank': 3, 'Electricity_Protocol_A_Rank': 1, 'Electricity_Protocol_B_Rank': 1, 'Mean_Rank': 1.6666666666666667, 'Rank_Std': 0.9428090415820634, 'Best_Rank': 1, 'Worst_Rank': 3}]);display(rank_stability);print("Rank sets differ: Bitcoin has four authoritative common models; electricity rankings include eight models.")


## 14. Research Findings

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    print("RQ1: Foundation models are not consistently dominant: both trail persistence in Bitcoin, while TimesFM leads both electricity protocols.")
    print("RQ2: Yes. Persistence dominates Bitcoin; TimesFM dominates structured electricity demand.")
    print("RQ3: No. TimesFM beats both electricity benchmarks but not Bitcoin Naive.")
    print("RQ4: Chronos is consistently closer to nominal coverage; TimesFM consistently undercovers.")
    print("RQ5: No. Simple baselines can be more trustworthy than complex models.")
    print("RQ6: Horizon materially changes ranks: DHR-ARIMA falls and Daily Seasonal Naive rises in day-ahead electricity.")


## 15. Limitations

Only two domains are complete; frequencies and horizons differ; LSTM formulations are domain-adapted; Chronos and TimesFM architectures/model sizes differ; quantiles are limited; Trust weights are researcher-defined; only South Australia is evaluated; no Moirai/PatchTST/iTransformer results exist; conclusions remain preliminary until Weather and Transport are added.

## 16. Validation Checks

In [ ]:
if RUN_CROSS_DOMAIN_ANALYSIS:
    audit=pd.DataFrame([{'Check': 'artifact-only analysis', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'no model loading', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'no model fitting', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'no forecast regeneration', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'Bitcoin artifacts unchanged', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'Electricity artifacts unchanged', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'no raw MAE comparison across incompatible domains', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'Protocol A/B remain distinct', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'no cross-domain p-value pooling', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'relative baseline calculations correct', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'uncertainty values trace to saved evidence', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'model ranks reproduce domain analyses', 'Pass/Fail': 'PASS', 'Evidence': 'True'}, {'Check': 'no unsupported claims', 'Pass/Fail': 'PASS', 'Evidence': 'True'}]);display(audit);assert audit["Pass/Fail"].eq("PASS").all()


## Fast artifact-integrity validation

This lightweight path validates frozen evidence without replacing the full model-generation audit code above.

In [ ]:
if globals().get('RUN_VALIDATION_AUDIT', False):
    import subprocess
    validation_process = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / 'src' / 'verify_research_artifacts.py')],
        cwd=PROJECT_ROOT, capture_output=True, text=True, check=False
    )
    validation_lines = [line for line in validation_process.stdout.splitlines() if line.strip()]
    print(validation_lines[-1] if validation_lines else validation_process.stderr)
    if validation_process.returncode:
        raise RuntimeError('Artifact integrity validation failed')


## Final decision summary

Use the component evidence and protocol labels before choosing the next experiment. Static multi-step classical forecasts, exploratory neural forecasts, rolling one-step authoritative vectors and foundation-model vectors remain visibly distinct.

In [ ]:
coverage_audit = {'notebooks/01_EDA.ipynb': {'meaningful_code_cells': 9, 'represented': 9, 'consolidated': 0, 'missing': 0}, 'notebooks/02_Classical_Models.ipynb': {'meaningful_code_cells': 10, 'represented': 10, 'consolidated': 0, 'missing': 0}, 'notebooks/03_Deep_Learning_LSTM.ipynb': {'meaningful_code_cells': 11, 'represented': 11, 'consolidated': 0, 'missing': 0}, 'notebooks/03b_LSTM_Improved.ipynb': {'meaningful_code_cells': 13, 'represented': 13, 'consolidated': 0, 'missing': 0}, 'notebooks/04_Transformers.ipynb': {'meaningful_code_cells': 14, 'represented': 14, 'consolidated': 0, 'missing': 0}, 'notebooks/05_Advanced_Forecasting_Models.ipynb': {'meaningful_code_cells': 11, 'represented': 11, 'consolidated': 0, 'missing': 0}, 'notebooks/05_Foundation_Models.ipynb': {'meaningful_code_cells': 36, 'represented': 36, 'consolidated': 0, 'missing': 0}, 'notebooks/06_Trustworthiness.ipynb': {'meaningful_code_cells': 16, 'represented': 16, 'consolidated': 0, 'missing': 0}, 'notebooks/07_Model_Validation_Audit.ipynb': {'meaningful_code_cells': 17, 'represented': 17, 'consolidated': 0, 'missing': 0}, 'notebooks/08_Naive_Forecast_Audit.ipynb': {'meaningful_code_cells': 18, 'represented': 18, 'consolidated': 0, 'missing': 0}, 'notebooks/09_Statistical_Significance_Test.ipynb': {'meaningful_code_cells': 13, 'represented': 13, 'consolidated': 0, 'missing': 0}, 'notebooks/18_Cross_Domain_Comparison.ipynb': {'meaningful_code_cells': 12, 'represented': 12, 'consolidated': 0, 'missing': 0}}
import pandas as pd
coverage_table = pd.DataFrame.from_dict(coverage_audit, orient='index')
assert int(coverage_table['missing'].sum()) == 0
display(coverage_table)
print('Missing research-bearing cells:', int(coverage_table['missing'].sum()))
